In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
BASE = '/content/drive/My Drive/Colab Notebooks2'
df = pd.read_parquet(f'{BASE}/paper3_arm_sample.parquet')

FRAGS = ['Call_g','Call_ge','Call_get','Call_gett','Call_getti','Call_gettime',
         'Call_cloc','Call_na','Call_rt_','Call_rt_sigact','Call_readlin','Call_socke']

cols  = [c for c in df.columns if c.startswith('Call_')]
frags = [c for c in FRAGS if c in df.columns]
total = df[cols].values.sum()

print(f"true fragment share: {df[frags].values.sum()/total*100:.4f}%")
print(df[frags].sum().sort_values(ascending=False))

# the pair that matters
for f, full in [('Call_cloc','Call_clock_gettime'),
                ('Call_socke','Call_socket'),
                ('Call_readlin','Call_readlink')]:
    if f in df.columns and full in df.columns:
        both = ((df[f] > 0) & (df[full] > 0)).sum()
        print(f"{f} / {full}: rows with both non-zero = {both}")

true fragment share: 0.0000%
Call_g            0.0
Call_ge           0.0
Call_get          0.0
Call_gett         0.0
Call_getti        0.0
Call_gettime      0.0
Call_cloc         0.0
Call_na           0.0
Call_rt_          0.0
Call_rt_sigact    0.0
Call_readlin      0.0
Call_socke        0.0
dtype: float64
Call_cloc / Call_clock_gettime: rows with both non-zero = 0
Call_socke / Call_socket: rows with both non-zero = 0
Call_readlin / Call_readlink: rows with both non-zero = 0


In [ ]:
"""
=============================================================================
ARM TRANSFER ANOMALY: VERIFICATION + CONTROLLED SENSITIVITY EXPERIMENTS
=============================================================================

Run the parts in order. Each is independent and prints its own table.

  PART 0  Fragment verification across all four architectures
          (completes the ARM-only check you already ran)

  PART 1  Baseline transfer, reproduced here so every later number is
          comparable under identical code

  PART 2  CONTROL 1 - class composition
          ARM reduced from ~96% to ~80% malware, binary diversity kept
          as high as possible. Answers: is the ARM gap driven by its
          unusual class balance?

  PART 3  CONTROL 2 - binary diversity
          x86, MIPS, MIPSEL reduced to ARM's ~1,992 unique hashes, class
          balance preserved. Answers: is the ARM gap driven by its
          smaller binary population?

          NOTE: ARM has only 1,992 hashes, so diversity can only be
          matched downward. This control therefore reduces the OTHER
          architectures, not ARM.

  PART 4  Summary table for the paper

Protocol held constant throughout: 87 common Call_* features, identical
column order, RandomForest(n_estimators=200, class_weight='balanced',
random_state=42), macro F1, hash-grouped splits, no target-architecture
data in training.
=============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'

ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}

ARCHS = list(ARCH_FILES.keys())


# =============================================================================
# SHARED HELPERS
# =============================================================================

def load_all():
    """Load the four saved samples and attach a binary label."""
    data = {}
    for arch, path in ARCH_FILES.items():
        df = pd.read_parquet(path)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        data[arch] = df
    return data


def common_features(data):
    """87 syscall columns present in all four architectures, fixed order."""
    sets = [set(c for c in df.columns if c.startswith('Call_')) for df in data.values()]
    feats = sorted(set.intersection(*sets))
    return feats


def describe(df, name):
    mal = df['label'].mean() * 100
    print(f"  {name:28s} rows={len(df):7,d}  malware={mal:5.2f}%  "
          f"hashes={df['Hash'].nunique():5,d}")


def transfer(train_df, test_df, feats):
    """Train on the full source, evaluate on the full target. Macro F1."""
    clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(train_df[feats].values, train_df['label'].values)
    pred = clf.predict(test_df[feats].values)
    return f1_score(test_df['label'].values, pred, average='macro')


def same_arch(df, feats):
    """Hash-grouped held-out split within one architecture. Macro F1."""
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    tr_i, te_i = next(gss.split(df[feats].values, df['label'].values, groups=df['Hash'].values))
    tr, te = df.iloc[tr_i], df.iloc[te_i]
    clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(tr[feats].values, tr['label'].values)
    pred = clf.predict(te[feats].values)
    return f1_score(te['label'].values, pred, average='macro')


def run_matrix(data, feats, label):
    """Same-architecture baselines plus all 12 directed transfers."""
    rows = []
    for a in ARCHS:
        rows.append({'setting': label, 'train': a, 'test': a, 'kind': 'same',
                     'macro_f1': same_arch(data[a], feats)})
    for src in ARCHS:
        for tgt in ARCHS:
            if src == tgt:
                continue
            rows.append({'setting': label, 'train': src, 'test': tgt, 'kind': 'transfer',
                         'macro_f1': transfer(data[src], data[tgt], feats)})
    return pd.DataFrame(rows)


def arm_summary(res, label):
    """Mean transfer F1 for ARM-involving vs non-ARM pairs."""
    t = res[res['kind'] == 'transfer']
    arm = t[(t.train == 'ARM') | (t.test == 'ARM')]['macro_f1']
    non = t[(t.train != 'ARM') & (t.test != 'ARM')]['macro_f1']
    return {'setting': label,
            'arm_pairs_mean': arm.mean(), 'arm_pairs_min': arm.min(),
            'non_arm_mean': non.mean(), 'non_arm_min': non.min(),
            'gap': non.mean() - arm.mean()}


# =============================================================================
# PART 0 - FRAGMENT VERIFICATION (all four architectures)
# =============================================================================

def part0_fragments(data):
    print("=" * 78)
    print("PART 0 - FRAGMENT VERIFICATION")
    print("=" * 78)

    # Known truncated names, identified by inspection of the schemas.
    # A fragment is a column whose name is not a valid Linux syscall.
    KNOWN_FRAGMENTS = {
        'ARM':    ['Call_g', 'Call_ge', 'Call_get', 'Call_gett', 'Call_getti',
                   'Call_gettime', 'Call_cloc', 'Call_na', 'Call_rt_',
                   'Call_rt_sigact', 'Call_readlin', 'Call_socke'],
        'x86':    ['Call_g', 'Call_getd', 'Call_fc', 'Call_ioct', 'Call_r',
                   'Call_re', 'Call_rt_sig', 'Call_s', 'Call_getpp'],
        'MIPS':   ['Call_rt_si'],
        'MIPSEL': [],
    }

    out = []
    for arch, df in data.items():
        cols = [c for c in df.columns if c.startswith('Call_')]
        frags = [c for c in KNOWN_FRAGMENTS.get(arch, []) if c in df.columns]
        total = df[cols].values.sum()
        fsum = df[frags].values.sum() if frags else 0.0
        nonzero = [c for c in frags if df[c].sum() > 0]
        out.append({'arch': arch, 'call_cols': len(cols), 'fragments': len(frags),
                    'fragment_counts': fsum,
                    'pct_of_all_counts': 100 * fsum / total if total else 0.0,
                    'nonzero_fragments': ', '.join(nonzero) if nonzero else 'none'})
    df_out = pd.DataFrame(out)
    print(df_out.to_string(index=False))
    print("\nIf 'fragment_counts' is 0 everywhere, the fragments are empty")
    print("schema artifacts and have no effect on any reported result.")
    print("Quote 'nonzero_fragments' in the paper if any are non-empty.\n")
    return df_out


# =============================================================================
# PART 2 - CONTROL 1: CLASS COMPOSITION
# =============================================================================

def balance_by_hash(df, target_malware_frac, seed=RANDOM_STATE):
    """
    Reduce an architecture to a target malware row fraction by dropping whole
    binaries (hashes), never individual rows. Keeps every benign hash so that
    binary diversity is preserved as far as possible.
    """
    rng = np.random.RandomState(seed)

    hash_lab = df.groupby('Hash')['label'].first()
    hash_n = df.groupby('Hash').size()

    benign_hashes = hash_lab[hash_lab == 0].index.tolist()
    malware_hashes = hash_lab[hash_lab == 1].index.tolist()

    n_benign_rows = hash_n[benign_hashes].sum()
    # rows_mal / (rows_mal + n_benign_rows) = target  ->  solve for rows_mal
    target_mal_rows = target_malware_frac * n_benign_rows / (1 - target_malware_frac)

    rng.shuffle(malware_hashes)
    kept, acc = [], 0
    for h in malware_hashes:
        if acc >= target_mal_rows:
            break
        kept.append(h)
        acc += hash_n[h]

    keep = set(benign_hashes) | set(kept)
    return df[df['Hash'].isin(keep)].copy()


def part2_class_control(data, feats, baseline_res):
    print("=" * 78)
    print("PART 2 - CONTROL 1: CLASS COMPOSITION")
    print("=" * 78)

    target = data['x86']['label'].mean()   # match x86's malware fraction
    print(f"Target malware fraction (from x86): {target*100:.2f}%\n")

    ctrl = dict(data)
    print("Before:")
    describe(data['ARM'], 'ARM (original)')
    ctrl['ARM'] = balance_by_hash(data['ARM'], target)
    print("After:")
    describe(ctrl['ARM'], 'ARM (class-controlled)')
    print()

    res = run_matrix(ctrl, feats, 'control1_class')
    print(res.to_string(index=False))
    print()
    s_base = arm_summary(baseline_res, 'baseline')
    s_ctrl = arm_summary(res, 'control1_class')
    print(pd.DataFrame([s_base, s_ctrl]).to_string(index=False))
    print("\nINTERPRETATION")
    print("  gap shrinks substantially -> ARM effect is largely class composition")
    print("  gap persists              -> class composition does not explain it\n")
    return res


# =============================================================================
# PART 3 - CONTROL 2: BINARY DIVERSITY
# =============================================================================

def cap_hashes(df, n_hashes, seed=RANDOM_STATE):
    """
    Reduce an architecture to n_hashes unique binaries, sampling malware and
    benign hashes in proportion so the class balance is preserved.
    """
    rng = np.random.RandomState(seed)
    hash_lab = df.groupby('Hash')['label'].first()

    ben = hash_lab[hash_lab == 0].index.to_numpy()
    mal = hash_lab[hash_lab == 1].index.to_numpy()

    frac = n_hashes / len(hash_lab)
    n_ben = max(1, int(round(len(ben) * frac)))
    n_mal = max(1, n_hashes - n_ben)

    keep = set(rng.choice(ben, size=min(n_ben, len(ben)), replace=False)) | \
           set(rng.choice(mal, size=min(n_mal, len(mal)), replace=False))
    return df[df['Hash'].isin(keep)].copy()


def part3_diversity_control(data, feats, baseline_res):
    print("=" * 78)
    print("PART 3 - CONTROL 2: BINARY DIVERSITY")
    print("=" * 78)

    n_arm = data['ARM']['Hash'].nunique()
    print(f"ARM unique hashes: {n_arm:,} -- capping every architecture to this.\n")

    ctrl = {}
    print("Before / after:")
    for arch, df in data.items():
        describe(df, f'{arch} (original)')
        ctrl[arch] = df if arch == 'ARM' else cap_hashes(df, n_arm)
        describe(ctrl[arch], f'{arch} (capped)')
    print()

    res = run_matrix(ctrl, feats, 'control2_diversity')
    print(res.to_string(index=False))
    print()
    print(pd.DataFrame([arm_summary(baseline_res, 'baseline'),
                        arm_summary(res, 'control2_diversity')]).to_string(index=False))
    print("\nINTERPRETATION")
    print("  non-ARM transfers fall toward ARM's level -> diversity matters")
    print("  non-ARM transfers stay high               -> diversity is not the cause\n")
    return res


# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    print("Loading samples...\n")
    data = load_all()
    for a, df in data.items():
        describe(df, a)
    print()

    feats = common_features(data)
    print(f"Common syscall features: {len(feats)}\n")

    # PART 0
    frag_table = part0_fragments(data)

    # PART 1 - baseline, under identical code
    print("=" * 78)
    print("PART 1 - BASELINE (uncontrolled)")
    print("=" * 78)
    baseline = run_matrix(data, feats, 'baseline')
    print(baseline.to_string(index=False))
    print()

    # PART 2 and PART 3
    c1 = part2_class_control(data, feats, baseline)
    c2 = part3_diversity_control(data, feats, baseline)

    # PART 4 - summary
    print("=" * 78)
    print("PART 4 - SUMMARY FOR THE PAPER")
    print("=" * 78)
    summary = pd.DataFrame([
        arm_summary(baseline, 'Original composition'),
        arm_summary(c1, 'Class-balance controlled'),
        arm_summary(c2, 'Binary-diversity controlled'),
    ])
    print(summary.to_string(index=False))

    allres = pd.concat([baseline, c1, c2], ignore_index=True)
    allres.to_csv(f'{BASE_DIR}/paper3_arm_controls_full.csv', index=False)
    summary.to_csv(f'{BASE_DIR}/paper3_arm_controls_summary.csv', index=False)
    frag_table.to_csv(f'{BASE_DIR}/paper3_fragment_check.csv', index=False)
    print(f"\nSaved to {BASE_DIR}:")
    print("  paper3_arm_controls_full.csv")
    print("  paper3_arm_controls_summary.csv")
    print("  paper3_fragment_check.csv")

Loading samples...

  x86                          rows=199,655  malware=77.44%  hashes=3,815
  ARM                          rows=199,903  malware=96.00%  hashes=1,992
  MIPS                         rows=199,749  malware=77.95%  hashes=3,997
  MIPSEL                       rows=199,693  malware=80.39%  hashes=3,868

Common syscall features: 87

PART 0 - FRAGMENT VERIFICATION
  arch  call_cols  fragments  fragment_counts  pct_of_all_counts nonzero_fragments
   x86        132          8              0.0                0.0              none
   ARM        131         12              0.0                0.0              none
  MIPS        127          1              0.0                0.0              none
MIPSEL        129          0              0.0                0.0              none

If 'fragment_counts' is 0 everywhere, the fragments are empty
schema artifacts and have no effect on any reported result.
Quote 'nonzero_fragments' in the paper if any are non-empty.

PART 1 - BASELINE (unco

In [ ]:
import pandas as pd

# Path to your original Parquet file
parquet_file_path = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_x86'

# Read the Parquet file (only necessary columns to save memory)
df = pd.read_parquet(parquet_file_path, columns=['MalwareFamily', 'Arch'])

# Check what's in the MalwareFamily column
print("="*60)
print("LABEL DISTRIBUTION IN THE DATASET")
print("="*60)

# Show unique values and their counts
label_counts = df['MalwareFamily'].value_counts()
print("\n📊 MalwareFamily distribution:")
print(label_counts)

# Show what percentage each label represents
print("\n📊 Percentage breakdown:")
print((df['MalwareFamily'].value_counts(normalize=True) * 100).round(2))

# Check if 'Benign' exists in the dataset
print("\n🔍 Is 'Benign' present in the dataset?")
has_benign = 'Benign' in df['MalwareFamily'].unique()
print(f"   {'✅ Yes' if has_benign else '❌ No'}")

# List all unique malware families
print("\n📋 All unique values in MalwareFamily:")
for family in sorted(df['MalwareFamily'].unique()):
    print(f"   - {family}")

# Check architecture distribution (optional)
print("\n🏗️ Architecture distribution:")
print(df['Arch'].value_counts())

LABEL DISTRIBUTION IN THE DATASET

📊 MalwareFamily distribution:
MalwareFamily
Mirai        19745632
Benign        5832881
DarkNexus      227611
Gafgyt          32777
Generic         15752
Name: count, dtype: int64

📊 Percentage breakdown:
MalwareFamily
Mirai        76.37
Benign       22.56
DarkNexus     0.88
Gafgyt        0.13
Generic       0.06
Name: proportion, dtype: float64

🔍 Is 'Benign' present in the dataset?
   ✅ Yes

📋 All unique values in MalwareFamily:
   - Benign
   - DarkNexus
   - Gafgyt
   - Generic
   - Mirai

🏗️ Architecture distribution:
Arch
x86    25854653
Name: count, dtype: int64


In [ ]:
# ============================================================================
# PHASE 1: SAME-ARCHITECTURE BASELINE (x86), MEMORY-SAFE STREAMING VERSION
# Fixes the "Colab stops responding" problem by never materializing the
# full parquet in memory. Streams batches, keeps a random subsample as it goes.
# ============================================================================

import pyarrow.parquet as pq
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_x86'
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Known total row count for x86, from your earlier Cell 2 output.
# This lets us compute an exact sampling probability WITHOUT a full-file pass.
KNOWN_TOTAL_ROWS = 25_854_653
TARGET_SAMPLE_SIZE = 200_000   # start small to confirm the pipeline runs; raise later
KEEP_PROB = TARGET_SAMPLE_SIZE / KNOWN_TOTAL_ROWS

print("="*70)
print("STREAMING SAMPLE FROM PARQUET (memory-safe)")
print("="*70)
print(f"Target sample size: {TARGET_SAMPLE_SIZE:,} (keep probability: {KEEP_PROB:.5f})")

pf = pq.ParquetFile(PARQUET_PATH)
schema_names = [f.name for f in pf.schema_arrow]
feature_cols = [c for c in schema_names if c.startswith('Call_')]
needed_cols = feature_cols + ['Hash', 'MalwareFamily', 'Arch']

print(f"{len(feature_cols)} syscall feature columns found")

kept_chunks = []
rows_seen = 0
rows_kept = 0

# Stream in batches - only ever holds ONE batch + the running kept sample in RAM
for batch in pf.iter_batches(batch_size=200_000, columns=needed_cols):
    bdf = batch.to_pandas()
    rows_seen += len(bdf)

    # Vectorized Bernoulli sample - decide per-row whether to keep it
    mask = rng.random(len(bdf)) < KEEP_PROB
    if mask.any():
        kept_chunks.append(bdf[mask])
        rows_kept += mask.sum()

    if rows_seen % 2_000_000 < 200_000:
        print(f"  ...scanned {rows_seen:,} rows, kept {rows_kept:,} so far")

df = pd.concat(kept_chunks, ignore_index=True)
del kept_chunks
print(f"\nFinal sampled shape: {df.shape}")

# ----------------------------------------------------------------------------
# HASH-LEAKAGE AUDIT ON THE SAMPLE
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("HASH AUDIT")
print("="*70)
n_unique_hash = df['Hash'].nunique()
print(f"Rows: {len(df):,} | Unique hashes: {n_unique_hash:,}")
if n_unique_hash < len(df):
    print("[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.")
else:
    print("[OK] Each row has a unique Hash in this sample.")

# ----------------------------------------------------------------------------
# LABELS + REPRESENTATIONS
# ----------------------------------------------------------------------------
df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
print(f"\nTrue class balance in this sample:")
print(df['label'].value_counts())
print(f"Malware %: {df['label'].mean()*100:.2f}%  (dataset true rate: ~77.4%)")

X_raw = df[feature_cols].fillna(0).values
total_syscalls = df[feature_cols].fillna(0).sum(axis=1).replace(0, 1)
X_relfreq = df[feature_cols].fillna(0).div(total_syscalls, axis=0).values

y = df['label'].values
groups = df['Hash'].values

# ----------------------------------------------------------------------------
# LEAKAGE-SAFE SPLIT + TRAIN/COMPARE
# ----------------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

print(f"\nTrain: {len(train_idx):,} | Test: {len(test_idx):,}")
print(f"Train malware %: {y[train_idx].mean()*100:.2f}")
print(f"Test malware %:  {y[test_idx].mean()*100:.2f}")

print("\n" + "="*70)
print("SAME-ARCHITECTURE (x86 -> x86) COMPARISON")
print("="*70)

results = {}
for name, X in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
    print(f"\n--- {name} ---")
    clf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    clf.fit(X[train_idx], y[train_idx])
    y_pred = clf.predict(X[test_idx])

    f1 = f1_score(y[test_idx], y_pred, average='macro')
    acc = accuracy_score(y[test_idx], y_pred)
    print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
    print(classification_report(y[test_idx], y_pred, target_names=['Benign', 'Malware']))
    results[name] = {'accuracy': acc, 'macro_f1': f1}

print("\n" + "="*70)
print("SUMMARY (Phase 1 deliverable)")
print("="*70)
for name, r in results.items():
    print(f"{name:15s} | Accuracy: {r['accuracy']:.4f} | Macro F1: {r['macro_f1']:.4f}")

# Save the sampled dataframe so Phase 2/3 can reuse it without re-streaming
df.to_parquet('/content/drive/My Drive/Colab Notebooks2/paper3_x86_sample.parquet')
print("\nSaved sample to paper3_x86_sample.parquet for reuse in later phases.")

STREAMING SAMPLE FROM PARQUET (memory-safe)
Target sample size: 200,000 (keep probability: 0.00774)
132 syscall feature columns found
  ...scanned 2,000,000 rows, kept 15,360 so far
  ...scanned 4,000,000 rows, kept 30,981 so far
  ...scanned 6,000,000 rows, kept 46,312 so far
  ...scanned 8,000,000 rows, kept 61,973 so far
  ...scanned 10,000,000 rows, kept 77,509 so far
  ...scanned 12,000,000 rows, kept 93,048 so far
  ...scanned 14,000,000 rows, kept 108,463 so far
  ...scanned 16,000,000 rows, kept 123,754 so far
  ...scanned 18,000,000 rows, kept 139,269 so far
  ...scanned 20,000,000 rows, kept 154,619 so far
  ...scanned 22,000,000 rows, kept 169,989 so far
  ...scanned 24,000,000 rows, kept 185,436 so far

Final sampled shape: (199655, 135)

HASH AUDIT
Rows: 199,655 | Unique hashes: 3,815
[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.

True class balance in this sample:
label
1    154620
0     45035
Name: count, dtype: int64
Malware %: 77.44%  (dataset t

In [ ]:
# ============================================================================
# PHASE 1: SAME-ARCHITECTURE BASELINE (ARM), MEMORY-SAFE STREAMING VERSION
# Fixes the "Colab stops responding" problem by never materializing the
# full parquet in memory. Streams batches, keeps a random subsample as it goes.
# ============================================================================

import pyarrow.parquet as pq
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_arm'
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Known total row count for arm, from your earlier Cell 2 output.
# This lets us compute an exact sampling probability WITHOUT a full-file pass.
KNOWN_TOTAL_ROWS = 21_989_296
TARGET_SAMPLE_SIZE = 200_000   # start small to confirm the pipeline runs; raise later
KEEP_PROB = TARGET_SAMPLE_SIZE / KNOWN_TOTAL_ROWS

print("="*70)
print("STREAMING SAMPLE FROM PARQUET (memory-safe)")
print("="*70)
print(f"Target sample size: {TARGET_SAMPLE_SIZE:,} (keep probability: {KEEP_PROB:.5f})")

pf = pq.ParquetFile(PARQUET_PATH)
schema_names = [f.name for f in pf.schema_arrow]
feature_cols = [c for c in schema_names if c.startswith('Call_')]
needed_cols = feature_cols + ['Hash', 'MalwareFamily', 'Arch']

print(f"{len(feature_cols)} syscall feature columns found")

kept_chunks = []
rows_seen = 0
rows_kept = 0

# Stream in batches - only ever holds ONE batch + the running kept sample in RAM
for batch in pf.iter_batches(batch_size=200_000, columns=needed_cols):
    bdf = batch.to_pandas()
    rows_seen += len(bdf)

    # Vectorized Bernoulli sample - decide per-row whether to keep it
    mask = rng.random(len(bdf)) < KEEP_PROB
    if mask.any():
        kept_chunks.append(bdf[mask])
        rows_kept += mask.sum()

    if rows_seen % 2_000_000 < 200_000:
        print(f"  ...scanned {rows_seen:,} rows, kept {rows_kept:,} so far")

df = pd.concat(kept_chunks, ignore_index=True)
del kept_chunks
print(f"\nFinal sampled shape: {df.shape}")

# ----------------------------------------------------------------------------
# HASH-LEAKAGE AUDIT ON THE SAMPLE
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("HASH AUDIT")
print("="*70)
n_unique_hash = df['Hash'].nunique()
print(f"Rows: {len(df):,} | Unique hashes: {n_unique_hash:,}")
if n_unique_hash < len(df):
    print("[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.")
else:
    print("[OK] Each row has a unique Hash in this sample.")

# ----------------------------------------------------------------------------
# LABELS + REPRESENTATIONS
# ----------------------------------------------------------------------------
df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
print(f"\nTrue class balance in this sample:")
print(df['label'].value_counts())
print(f"Malware %: {df['label'].mean()*100:.2f}%  (dataset true rate: ~77.4%)")

X_raw = df[feature_cols].fillna(0).values
total_syscalls = df[feature_cols].fillna(0).sum(axis=1).replace(0, 1)
X_relfreq = df[feature_cols].fillna(0).div(total_syscalls, axis=0).values

y = df['label'].values
groups = df['Hash'].values

# ----------------------------------------------------------------------------
# LEAKAGE-SAFE SPLIT + TRAIN/COMPARE
# ----------------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

print(f"\nTrain: {len(train_idx):,} | Test: {len(test_idx):,}")
print(f"Train malware %: {y[train_idx].mean()*100:.2f}")
print(f"Test malware %:  {y[test_idx].mean()*100:.2f}")

print("\n" + "="*70)
print("SAME-ARCHITECTURE (arm -> arm) COMPARISON")
print("="*70)

results = {}
for name, X in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
    print(f"\n--- {name} ---")
    clf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    clf.fit(X[train_idx], y[train_idx])
    y_pred = clf.predict(X[test_idx])

    f1 = f1_score(y[test_idx], y_pred, average='macro')
    acc = accuracy_score(y[test_idx], y_pred)
    print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
    print(classification_report(y[test_idx], y_pred, target_names=['Benign', 'Malware']))
    results[name] = {'accuracy': acc, 'macro_f1': f1}

print("\n" + "="*70)
print("SUMMARY (Phase 1 deliverable)")
print("="*70)
for name, r in results.items():
    print(f"{name:15s} | Accuracy: {r['accuracy']:.4f} | Macro F1: {r['macro_f1']:.4f}")

# Save the sampled dataframe so Phase 2/3 can reuse it without re-streaming
df.to_parquet('/content/drive/My Drive/Colab Notebooks2/paper3_arm_sample.parquet')
print("\nSaved sample to paper3_arm_sample.parquet for reuse in later phases.")

STREAMING SAMPLE FROM PARQUET (memory-safe)
Target sample size: 200,000 (keep probability: 0.00910)
131 syscall feature columns found
  ...scanned 2,000,000 rows, kept 18,050 so far
  ...scanned 4,000,000 rows, kept 36,463 so far
  ...scanned 6,000,000 rows, kept 54,472 so far
  ...scanned 8,000,000 rows, kept 72,848 so far
  ...scanned 10,000,000 rows, kept 91,172 so far
  ...scanned 12,000,000 rows, kept 109,458 so far
  ...scanned 14,000,000 rows, kept 127,487 so far
  ...scanned 16,000,000 rows, kept 145,532 so far
  ...scanned 18,000,000 rows, kept 163,773 so far
  ...scanned 20,000,000 rows, kept 181,857 so far

Final sampled shape: (199903, 134)

HASH AUDIT
Rows: 199,903 | Unique hashes: 1,992
[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.

True class balance in this sample:
label
1    191904
0      7999
Name: count, dtype: int64
Malware %: 96.00%  (dataset true rate: ~77.4%)

Train: 166,432 | Test: 33,471
Train malware %: 96.21
Test malware %:  94.94

SAM

In [ ]:
# ============================================================================
# PHASE 1: SAME-ARCHITECTURE BASELINE (MIPS), MEMORY-SAFE STREAMING VERSION
# Fixes the "Colab stops responding" problem by never materializing the
# full parquet in memory. Streams batches, keeps a random subsample as it goes.
# ============================================================================

import pyarrow.parquet as pq
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_mips'
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Known total row count for arm, from your earlier Cell 2 output.
# This lets us compute an exact sampling probability WITHOUT a full-file pass.
KNOWN_TOTAL_ROWS = 29_993_871
TARGET_SAMPLE_SIZE = 200_000   # start small to confirm the pipeline runs; raise later
KEEP_PROB = TARGET_SAMPLE_SIZE / KNOWN_TOTAL_ROWS

print("="*70)
print("STREAMING SAMPLE FROM PARQUET (memory-safe)")
print("="*70)
print(f"Target sample size: {TARGET_SAMPLE_SIZE:,} (keep probability: {KEEP_PROB:.5f})")

pf = pq.ParquetFile(PARQUET_PATH)
schema_names = [f.name for f in pf.schema_arrow]
feature_cols = [c for c in schema_names if c.startswith('Call_')]
needed_cols = feature_cols + ['Hash', 'MalwareFamily', 'Arch']

print(f"{len(feature_cols)} syscall feature columns found")

kept_chunks = []
rows_seen = 0
rows_kept = 0

# Stream in batches - only ever holds ONE batch + the running kept sample in RAM
for batch in pf.iter_batches(batch_size=200_000, columns=needed_cols):
    bdf = batch.to_pandas()
    rows_seen += len(bdf)

    # Vectorized Bernoulli sample - decide per-row whether to keep it
    mask = rng.random(len(bdf)) < KEEP_PROB
    if mask.any():
        kept_chunks.append(bdf[mask])
        rows_kept += mask.sum()

    if rows_seen % 2_000_000 < 200_000:
        print(f"  ...scanned {rows_seen:,} rows, kept {rows_kept:,} so far")

df = pd.concat(kept_chunks, ignore_index=True)
del kept_chunks
print(f"\nFinal sampled shape: {df.shape}")

# ----------------------------------------------------------------------------
# HASH-LEAKAGE AUDIT ON THE SAMPLE
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("HASH AUDIT")
print("="*70)
n_unique_hash = df['Hash'].nunique()
print(f"Rows: {len(df):,} | Unique hashes: {n_unique_hash:,}")
if n_unique_hash < len(df):
    print("[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.")
else:
    print("[OK] Each row has a unique Hash in this sample.")

# ----------------------------------------------------------------------------
# LABELS + REPRESENTATIONS
# ----------------------------------------------------------------------------
df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
print(f"\nTrue class balance in this sample:")
print(df['label'].value_counts())
print(f"Malware %: {df['label'].mean()*100:.2f}%  (dataset true rate: ~77.4%)")

X_raw = df[feature_cols].fillna(0).values
total_syscalls = df[feature_cols].fillna(0).sum(axis=1).replace(0, 1)
X_relfreq = df[feature_cols].fillna(0).div(total_syscalls, axis=0).values

y = df['label'].values
groups = df['Hash'].values

# ----------------------------------------------------------------------------
# LEAKAGE-SAFE SPLIT + TRAIN/COMPARE
# ----------------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

print(f"\nTrain: {len(train_idx):,} | Test: {len(test_idx):,}")
print(f"Train malware %: {y[train_idx].mean()*100:.2f}")
print(f"Test malware %:  {y[test_idx].mean()*100:.2f}")

print("\n" + "="*70)
print("SAME-ARCHITECTURE (mips -> mips) COMPARISON")
print("="*70)

results = {}
for name, X in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
    print(f"\n--- {name} ---")
    clf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    clf.fit(X[train_idx], y[train_idx])
    y_pred = clf.predict(X[test_idx])

    f1 = f1_score(y[test_idx], y_pred, average='macro')
    acc = accuracy_score(y[test_idx], y_pred)
    print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
    print(classification_report(y[test_idx], y_pred, target_names=['Benign', 'Malware']))
    results[name] = {'accuracy': acc, 'macro_f1': f1}

print("\n" + "="*70)
print("SUMMARY (Phase 1 deliverable)")
print("="*70)
for name, r in results.items():
    print(f"{name:15s} | Accuracy: {r['accuracy']:.4f} | Macro F1: {r['macro_f1']:.4f}")

# Save the sampled dataframe so Phase 2/3 can reuse it without re-streaming
df.to_parquet('/content/drive/My Drive/Colab Notebooks2/paper3_mips_sample.parquet')
print("\nSaved sample to paper3_mips_sample.parquet for reuse in later phases.")

STREAMING SAMPLE FROM PARQUET (memory-safe)
Target sample size: 200,000 (keep probability: 0.00667)
127 syscall feature columns found
  ...scanned 2,000,000 rows, kept 13,306 so far
  ...scanned 4,000,000 rows, kept 26,781 so far
  ...scanned 6,000,000 rows, kept 39,974 so far
  ...scanned 8,000,000 rows, kept 53,492 so far
  ...scanned 10,000,000 rows, kept 66,885 so far
  ...scanned 12,000,000 rows, kept 80,223 so far
  ...scanned 14,000,000 rows, kept 93,545 so far
  ...scanned 16,000,000 rows, kept 106,637 so far
  ...scanned 18,000,000 rows, kept 120,026 so far
  ...scanned 20,000,000 rows, kept 133,343 so far
  ...scanned 22,000,000 rows, kept 146,589 so far
  ...scanned 24,000,000 rows, kept 159,834 so far
  ...scanned 26,000,000 rows, kept 173,155 so far
  ...scanned 28,000,000 rows, kept 186,433 so far

Final sampled shape: (199749, 130)

HASH AUDIT
Rows: 199,749 | Unique hashes: 3,997
[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.

True class balance in

In [ ]:
# ============================================================================
# CORRECTED SAME-ARCHITECTURE BASELINE
# Uses the 87 common features (same as cross-arch matrix) AND a proper
# hash-safe GroupShuffleSplit (same rigor as original Phase 1 runs).
# This is the one apples-to-apples comparison neither prior table provided.
# ============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'

ARCH_FILES = {
    'x86': f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM': f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS': f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}

print("Loading samples...")
data = {}
for arch, path in ARCH_FILES.items():
    df = pd.read_parquet(path)
    df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
    data[arch] = df

# Same 87-feature common intersection used in the cross-architecture matrix
all_feature_sets = [set(c for c in df.columns if c.startswith('Call_')) for df in data.values()]
common_features = sorted(set.intersection(*all_feature_sets))
print(f"Common features: {len(common_features)} (should match cross-arch matrix: 87)\n")

results = {}

for arch, df in data.items():
    print(f"--- {arch}: same-architecture, 87 common features, hash-safe split ---")

    X_raw = df[common_features].fillna(0).values
    total = df[common_features].fillna(0).sum(axis=1).replace(0, 1)
    X_relfreq = df[common_features].fillna(0).div(total, axis=0).values
    y = df['label'].values
    groups = df['Hash'].values

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

    arch_results = {}
    for name, X in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
        clf = RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        )
        clf.fit(X[train_idx], y[train_idx])
        y_pred = clf.predict(X[test_idx])

        f1 = f1_score(y[test_idx], y_pred, average='macro')
        acc = accuracy_score(y[test_idx], y_pred)
        print(f"  {name:15s} | Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
        arch_results[name] = {'accuracy': acc, 'macro_f1': f1}

    results[arch] = arch_results
    print()

print("="*70)
print("CORRECTED SAME-ARCHITECTURE BASELINE (87 common features, hash-safe)")
print("="*70)
print(f"{'Architecture':12s} {'Raw F1':>10s} {'RelFreq F1':>12s}")
for arch, r in results.items():
    print(f"{arch:12s} {r['raw_counts']['macro_f1']:>10.4f} {r['relative_freq']['macro_f1']:>12.4f}")

print("\nThis table should be used as the paper's Table 3.1 (Same-Architecture")
print("Baseline), replacing both earlier versions, since it is the only one")
print("using identical features AND identical split rigor to the cross-")
print("architecture matrix (Table 3.2).")

Loading samples...
Common features: 87 (should match cross-arch matrix: 87)

--- x86: same-architecture, 87 common features, hash-safe split ---
  raw_counts      | Accuracy: 0.9862 | Macro F1: 0.9815
  relative_freq   | Accuracy: 0.9693 | Macro F1: 0.9596

--- ARM: same-architecture, 87 common features, hash-safe split ---
  raw_counts      | Accuracy: 0.9890 | Macro F1: 0.9475
  relative_freq   | Accuracy: 0.9897 | Macro F1: 0.9504

--- MIPS: same-architecture, 87 common features, hash-safe split ---
  raw_counts      | Accuracy: 0.9917 | Macro F1: 0.9880
  relative_freq   | Accuracy: 0.9613 | Macro F1: 0.9470

--- MIPSEL: same-architecture, 87 common features, hash-safe split ---
  raw_counts      | Accuracy: 0.9927 | Macro F1: 0.9890
  relative_freq   | Accuracy: 0.9797 | Macro F1: 0.9700

CORRECTED SAME-ARCHITECTURE BASELINE (87 common features, hash-safe)
Architecture     Raw F1   RelFreq F1
x86              0.9815       0.9596
ARM              0.9475       0.9504
MIPS           

In [ ]:
# ============================================================================
# PHASE 1: SAME-ARCHITECTURE BASELINE (MIPSel), MEMORY-SAFE STREAMING VERSION
# Fixes the "Colab stops responding" problem by never materializing the
# full parquet in memory. Streams batches, keeps a random subsample as it goes.
# ============================================================================

import pyarrow.parquet as pq
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_mipsel'
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Known total row count for arm, from your earlier Cell 2 output.
# This lets us compute an exact sampling probability WITHOUT a full-file pass.
KNOWN_TOTAL_ROWS = 27_622_368
TARGET_SAMPLE_SIZE = 200_000   # start small to confirm the pipeline runs; raise later
KEEP_PROB = TARGET_SAMPLE_SIZE / KNOWN_TOTAL_ROWS

print("="*70)
print("STREAMING SAMPLE FROM PARQUET (memory-safe)")
print("="*70)
print(f"Target sample size: {TARGET_SAMPLE_SIZE:,} (keep probability: {KEEP_PROB:.5f})")

pf = pq.ParquetFile(PARQUET_PATH)
schema_names = [f.name for f in pf.schema_arrow]
feature_cols = [c for c in schema_names if c.startswith('Call_')]
needed_cols = feature_cols + ['Hash', 'MalwareFamily', 'Arch']

print(f"{len(feature_cols)} syscall feature columns found")

kept_chunks = []
rows_seen = 0
rows_kept = 0

# Stream in batches - only ever holds ONE batch + the running kept sample in RAM
for batch in pf.iter_batches(batch_size=200_000, columns=needed_cols):
    bdf = batch.to_pandas()
    rows_seen += len(bdf)

    # Vectorized Bernoulli sample - decide per-row whether to keep it
    mask = rng.random(len(bdf)) < KEEP_PROB
    if mask.any():
        kept_chunks.append(bdf[mask])
        rows_kept += mask.sum()

    if rows_seen % 2_000_000 < 200_000:
        print(f"  ...scanned {rows_seen:,} rows, kept {rows_kept:,} so far")

df = pd.concat(kept_chunks, ignore_index=True)
del kept_chunks
print(f"\nFinal sampled shape: {df.shape}")

# ----------------------------------------------------------------------------
# HASH-LEAKAGE AUDIT ON THE SAMPLE
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("HASH AUDIT")
print("="*70)
n_unique_hash = df['Hash'].nunique()
print(f"Rows: {len(df):,} | Unique hashes: {n_unique_hash:,}")
if n_unique_hash < len(df):
    print("[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.")
else:
    print("[OK] Each row has a unique Hash in this sample.")

# ----------------------------------------------------------------------------
# LABELS + REPRESENTATIONS
# ----------------------------------------------------------------------------
df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
print(f"\nTrue class balance in this sample:")
print(df['label'].value_counts())
print(f"Malware %: {df['label'].mean()*100:.2f}%  (dataset true rate: ~77.4%)")

X_raw = df[feature_cols].fillna(0).values
total_syscalls = df[feature_cols].fillna(0).sum(axis=1).replace(0, 1)
X_relfreq = df[feature_cols].fillna(0).div(total_syscalls, axis=0).values

y = df['label'].values
groups = df['Hash'].values

# ----------------------------------------------------------------------------
# LEAKAGE-SAFE SPLIT + TRAIN/COMPARE
# ----------------------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_raw, y, groups=groups))

print(f"\nTrain: {len(train_idx):,} | Test: {len(test_idx):,}")
print(f"Train malware %: {y[train_idx].mean()*100:.2f}")
print(f"Test malware %:  {y[test_idx].mean()*100:.2f}")

print("\n" + "="*70)
print("SAME-ARCHITECTURE (mipsel -> mipsel) COMPARISON")
print("="*70)

results = {}
for name, X in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
    print(f"\n--- {name} ---")
    clf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    clf.fit(X[train_idx], y[train_idx])
    y_pred = clf.predict(X[test_idx])

    f1 = f1_score(y[test_idx], y_pred, average='macro')
    acc = accuracy_score(y[test_idx], y_pred)
    print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
    print(classification_report(y[test_idx], y_pred, target_names=['Benign', 'Malware']))
    results[name] = {'accuracy': acc, 'macro_f1': f1}

print("\n" + "="*70)
print("SUMMARY (Phase 1 deliverable)")
print("="*70)
for name, r in results.items():
    print(f"{name:15s} | Accuracy: {r['accuracy']:.4f} | Macro F1: {r['macro_f1']:.4f}")

# Save the sampled dataframe so Phase 2/3 can reuse it without re-streaming
df.to_parquet('/content/drive/My Drive/Colab Notebooks2/paper3_mipsel_sample.parquet')
print("\nSaved sample to paper3_mipsel_sample.parquet for reuse in later phases.")

STREAMING SAMPLE FROM PARQUET (memory-safe)
Target sample size: 200,000 (keep probability: 0.00724)
129 syscall feature columns found
  ...scanned 2,000,000 rows, kept 14,397 so far
  ...scanned 4,000,000 rows, kept 29,034 so far
  ...scanned 6,000,000 rows, kept 43,377 so far
  ...scanned 8,000,000 rows, kept 58,052 so far
  ...scanned 10,000,000 rows, kept 72,572 so far
  ...scanned 12,000,000 rows, kept 87,118 so far
  ...scanned 14,000,000 rows, kept 101,561 so far
  ...scanned 16,000,000 rows, kept 115,824 so far
  ...scanned 18,000,000 rows, kept 130,352 so far
  ...scanned 20,000,000 rows, kept 144,760 so far
  ...scanned 22,000,000 rows, kept 159,164 so far
  ...scanned 24,000,000 rows, kept 173,586 so far
  ...scanned 26,000,000 rows, kept 188,077 so far

Final sampled shape: (199693, 132)

HASH AUDIT
Rows: 199,693 | Unique hashes: 3,868
[!] Duplicate hashes present - using GroupShuffleSplit on Hash below.

True class balance in this sample:
label
1    160525
0     39168
Name:

In [ ]:
# ============================================================================
# PHASE 3: CROSS-ARCHITECTURE TRANSFER (using saved samples, no re-streaming)
# Tests ARM<->x86 for both raw counts and relative frequency.
# Extend the ARCH_FILES dict below once MIPS/MIPSEL samples are saved.
# ============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'

# Add MIPS/MIPSEL here once those Phase-1 runs finish and save their samples
ARCH_FILES = {
    'x86': f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM': f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS': f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}

print("="*70)
print("LOADING SAVED SAMPLES")
print("="*70)
data = {}
for arch, path in ARCH_FILES.items():
    df = pd.read_parquet(path)
    df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
    data[arch] = df
    print(f"{arch}: {df.shape}, malware% = {df['label'].mean()*100:.2f}")

# ----------------------------------------------------------------------------
# IMPORTANT: architectures don't have identical Call_* columns (x86 had 132,
# ARM had 131). A model trained on one architecture's feature vector can only
# be applied to another if both use the SAME columns in the SAME order.
# We take the intersection of feature columns across all loaded architectures.
# ----------------------------------------------------------------------------
all_feature_sets = []
for arch, df in data.items():
    cols = set(c for c in df.columns if c.startswith('Call_'))
    all_feature_sets.append(cols)

common_features = sorted(set.intersection(*all_feature_sets))
print(f"\nCommon feature columns across {list(data.keys())}: {len(common_features)}")
dropped = set.union(*all_feature_sets) - set(common_features)
if dropped:
    print(f"Dropped (not common to all architectures), full list ({len(dropped)}):")
    for c in sorted(dropped):
        # show which architectures actually have each dropped column,
        # useful for spotting truncated/fragmented syscall names per-arch
        present_in = [a for a, cols in zip(data.keys(), all_feature_sets) if c in cols]
        print(f"  {c:30s} present in: {present_in}")

# ----------------------------------------------------------------------------
# BUILD RAW + RELATIVE-FREQUENCY MATRICES PER ARCHITECTURE (common cols only)
# ----------------------------------------------------------------------------
X_raw = {}
X_relfreq = {}
y = {}

for arch, df in data.items():
    raw = df[common_features].fillna(0).values
    total = df[common_features].fillna(0).sum(axis=1).replace(0, 1)
    relfreq = df[common_features].fillna(0).div(total, axis=0).values

    X_raw[arch] = raw
    X_relfreq[arch] = relfreq
    y[arch] = df['label'].values

# ----------------------------------------------------------------------------
# CROSS-ARCHITECTURE TRANSFER MATRIX
# ----------------------------------------------------------------------------
archs = list(data.keys())
results = []

for representation, X_dict in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
    print("\n" + "="*70)
    print(f"REPRESENTATION: {representation}")
    print("="*70)

    for train_arch in archs:
        clf = RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        )
        clf.fit(X_dict[train_arch], y[train_arch])

        for test_arch in archs:
            y_pred = clf.predict(X_dict[test_arch])
            f1 = f1_score(y[test_arch], y_pred, average='macro')
            acc = accuracy_score(y[test_arch], y_pred)

            tag = "SAME-ARCH" if train_arch == test_arch else "CROSS-ARCH"
            print(f"  [{tag:10s}] Train={train_arch:6s} -> Test={test_arch:6s} "
                  f"| Acc={acc:.4f} | Macro F1={f1:.4f}")

            results.append({
                'representation': representation,
                'train': train_arch,
                'test': test_arch,
                'accuracy': acc,
                'macro_f1': f1,
                'same_arch': train_arch == test_arch
            })

# ----------------------------------------------------------------------------
# SUMMARY TABLE
# ----------------------------------------------------------------------------
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
print(results_df.to_string(index=False))

results_df.to_csv(f'{BASE_DIR}/paper3_cross_arch_results.csv', index=False)
print(f"\nSaved to paper3_cross_arch_results.csv")

# Quick summary: does relative_freq help more on CROSS-arch pairs specifically?
print("\n" + "="*70)
print("CROSS-ARCH ONLY: raw_counts vs relative_freq (the key comparison)")
print("="*70)
cross_only = results_df[~results_df['same_arch']]
summary = cross_only.groupby('representation')['macro_f1'].agg(['mean', 'min', 'max'])
print(summary)

LOADING SAVED SAMPLES
x86: (199655, 136), malware% = 77.44
ARM: (199903, 135), malware% = 96.00
MIPS: (199749, 131), malware% = 77.95
MIPSEL: (199693, 133), malware% = 80.39

Common feature columns across ['x86', 'ARM', 'MIPS', 'MIPSEL']: 87
Dropped (not common to all architectures), full list (94):
  Call_cacheflush                present in: ['ARM', 'MIPS', 'MIPSEL']
  Call_chown                     present in: ['MIPS', 'MIPSEL']
  Call_chown32                   present in: ['x86', 'ARM']
  Call_cloc                      present in: ['ARM']
  Call_clock_gettime64           present in: ['x86', 'MIPS', 'MIPSEL']
  Call_clock_nanosleep_time64    present in: ['x86', 'MIPS', 'MIPSEL']
  Call_dup                       present in: ['x86']
  Call_epoll_wait                present in: ['ARM']
  Call_fc                        present in: ['x86']
  Call_fchdir                    present in: ['MIPS', 'MIPSEL']
  Call_fstat64                   present in: ['x86', 'ARM', 'MIPSEL']
  Call_fstatat64

In [ ]:
# ============================================================================
# DIAGNOSTIC: Is ARM's poor cross-arch transfer partly a column-truncation bug?
# Run this using the already-saved ARM sample - no re-streaming needed.
# ============================================================================

import pandas as pd

BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
arm_df = pd.read_parquet(f'{BASE_DIR}/paper3_arm_sample.parquet')

arm_cols = [c for c in arm_df.columns if c.startswith('Call_')]
print(f"Total ARM Call_* columns: {len(arm_cols)}\n")

print("All ARM column names, sorted by length (shortest first),")
print("with character count - look for a suspicious cutoff length:\n")
for c in sorted(arm_cols, key=len):
    print(f"  len={len(c):3d}  {c}")

# Check specifically for the suspected truncation family around 'gettimeofday'
print("\n" + "="*60)
print("Columns starting with 'Call_g' (suspected truncation cluster):")
for c in sorted(arm_cols):
    if c.startswith('Call_g'):
        print(f"  {c}")

print("\n" + "="*60)
print("Compare against x86's equivalent columns (should be untruncated):")
x86_df = pd.read_parquet(f'{BASE_DIR}/paper3_x86_sample.parquet')
x86_cols = [c for c in x86_df.columns if c.startswith('Call_')]
for c in sorted(x86_cols):
    if c.startswith('Call_g'):
        print(f"  {c}")

# Look for an exact max-length pattern - if truncation is happening,
# there should be a hard ceiling on column name length for ARM specifically.
print("\n" + "="*60)
max_len_arm = max(len(c) for c in arm_cols)
max_len_x86 = max(len(c) for c in x86_cols)
print(f"Longest ARM column name: {max_len_arm} chars")
print(f"Longest x86 column name: {max_len_x86} chars")
print("If ARM's max is suspiciously short compared to x86's, and several ARM")
print("columns sit exactly at that max length, that strongly suggests")
print("truncation during ARM parquet/CSV generation - not a real difference")
print("in ARM's syscall vocabulary.")

Total ARM Call_* columns: 131

All ARM column names, sorted by length (shortest first),
with character count - look for a suspicious cutoff length:

  len=  6  Call_g
  len=  7  Call_ge
  len=  7  Call_na
  len=  8  Call_brk
  len=  8  Call_get
  len=  8  Call_rt_
  len=  9  Call_bind
  len=  9  Call_cloc
  len=  9  Call_dup2
  len=  9  Call_exit
  len=  9  Call_fork
  len=  9  Call_gett
  len=  9  Call_kill
  len=  9  Call_link
  len=  9  Call_open
  len=  9  Call_pipe
  len=  9  Call_poll
  len=  9  Call_read
  len=  9  Call_recv
  len=  9  Call_send
  len=  9  Call_sync
  len= 10  Call_chdir
  len= 10  Call_chmod
  len= 10  Call_clone
  len= 10  Call_close
  len= 10  Call_fcntl
  len= 10  Call_flock
  len= 10  Call_fstat
  len= 10  Call_fsync
  len= 10  Call_futex
  len= 10  Call_getti
  len= 10  Call_ioctl
  len= 10  Call_lseek
  len= 10  Call_mkdir
  len= 10  Call_mmap2
  len= 10  Call_mount
  len= 10  Call_pipe2
  len= 10  Call_prctl
  len= 10  Call_readv
  len= 10  Call_rmdir
  

In [ ]:
# ============================================================================
# SHAP DIAGNOSIS: compare feature importance across the 4 SAME-ARCHITECTURE
# models (not just one cross-arch pair). This answers: does ARM's own model
# rely on different features than x86/MIPS/MIPSEL's own models?
# ============================================================================

import shap
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
RANDOM_STATE = 42

ARCH_FILES = {
    'x86': f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM': f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS': f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}

print("Loading samples...")
data = {}
for arch, path in ARCH_FILES.items():
    df = pd.read_parquet(path)
    df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
    data[arch] = df

# Common features across all 4 (should be 87, as already established)
all_feature_sets = [set(c for c in df.columns if c.startswith('Call_')) for df in data.values()]
common_features = sorted(set.intersection(*all_feature_sets))
print(f"Common features: {len(common_features)}")

# ----------------------------------------------------------------------------
# Train each architecture's OWN model (same-arch), get SHAP importance for each
# ----------------------------------------------------------------------------
shap_importance = {}  # arch -> {feature: mean |SHAP|}

for arch, df in data.items():
    print(f"\n--- {arch}: training same-arch model + SHAP ---")
    X = df[common_features].fillna(0).values
    y = df['label'].values
    groups = df['Hash'].values

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    clf = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    clf.fit(X[train_idx], y[train_idx])

    # Use a modest subset of the test set for SHAP speed
    sample_n = min(1000, len(test_idx))
    X_sample = X[test_idx][:sample_n]

    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_sample)

    # Handle both possible SHAP output shapes across library versions:
    # older versions: list [neg_class_array, pos_class_array]
    # newer versions: single array shape (n_samples, n_features, n_classes)
    if isinstance(sv, list):
        pos_class_shap = sv[1]
    elif sv.ndim == 3:
        pos_class_shap = sv[:, :, 1]
    else:
        pos_class_shap = sv  # binary-only single-array output

    mean_abs_shap = np.abs(pos_class_shap).mean(axis=0)
    shap_importance[arch] = dict(zip(common_features, mean_abs_shap))

# ----------------------------------------------------------------------------
# BUILD COMPARISON TABLE: top features per architecture, side by side
# ----------------------------------------------------------------------------
importance_df = pd.DataFrame(shap_importance)
importance_df.index = [c.replace('Call_', '') for c in importance_df.index]

print("\n" + "="*70)
print("TOP 15 FEATURES PER ARCHITECTURE (own same-arch model)")
print("="*70)
for arch in ARCH_FILES.keys():
    top15 = importance_df[arch].sort_values(ascending=False).head(15)
    print(f"\n{arch}:")
    for feat, val in top15.items():
        print(f"  {feat:20s} {val:.4f}")

# ----------------------------------------------------------------------------
# KEY DIAGNOSTIC: features that are top-15 in ARM but NOT top-15 elsewhere,
# and vice versa - this is the "architecture-specific vs universal" evidence
# ----------------------------------------------------------------------------
top_sets = {arch: set(importance_df[arch].sort_values(ascending=False).head(15).index)
            for arch in ARCH_FILES.keys()}

print("\n" + "="*70)
print("FEATURES TOP-15 IN ARM BUT NOT IN x86, MIPS, OR MIPSEL:")
print("="*70)
arm_only = top_sets['ARM'] - top_sets['x86'] - top_sets['MIPS'] - top_sets['MIPSEL']
print(sorted(arm_only) if arm_only else "(none - ARM's top features overlap with others)")

print("\n" + "="*70)
print("FEATURES TOP-15 IN ALL FOUR (candidate 'universal' malware signals):")
print("="*70)
universal = top_sets['x86'] & top_sets['ARM'] & top_sets['MIPS'] & top_sets['MIPSEL']
print(sorted(universal) if universal else "(none in common)")

importance_df.to_csv(f'{BASE_DIR}/shap_importance_comparison.csv')
print(f"\nSaved full comparison table to shap_importance_comparison.csv")

Loading samples...
Common features: 87

--- x86: training same-arch model + SHAP ---

--- ARM: training same-arch model + SHAP ---

--- MIPS: training same-arch model + SHAP ---

--- MIPSEL: training same-arch model + SHAP ---

TOP 15 FEATURES PER ARCHITECTURE (own same-arch model)

x86:
  mmap2                0.0752
  readlink             0.0740
  read                 0.0615
  fcntl64              0.0609
  writev               0.0500
  getdents64           0.0394
  close                0.0261
  open                 0.0239
  ioctl                0.0210
  munmap               0.0201
  rt_sigaction         0.0170
  set_tid_address      0.0162
  mprotect             0.0153
  getpid               0.0145
  sendto               0.0144

ARM:
  clock_gettime        0.2527
  read                 0.0885
  readlink             0.0546
  open                 0.0324
  getppid              0.0229
  sendto               0.0228
  close                0.0175
  execve               0.0091
  set_tid_addre

In [ ]:
# ============================================================================
# PHASE 3: CROSS-ARCHITECTURE TRANSFER (using saved samples, no re-streaming)
# Tests ARM<->x86 for both raw counts and relative frequency.
# Extend the ARCH_FILES dict below once MIPS/MIPSEL samples are saved.
# ============================================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'

# Add MIPS/MIPSEL here once those Phase-1 runs finish and save their samples
ARCH_FILES = {
    'x86': f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM': f'{BASE_DIR}/paper3_arm_sample.parquet',
    # 'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    # 'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}

print("="*70)
print("LOADING SAVED SAMPLES")
print("="*70)
data = {}
for arch, path in ARCH_FILES.items():
    df = pd.read_parquet(path)
    df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
    data[arch] = df
    print(f"{arch}: {df.shape}, malware% = {df['label'].mean()*100:.2f}")

# ----------------------------------------------------------------------------
# IMPORTANT: architectures don't have identical Call_* columns (x86 had 132,
# ARM had 131). A model trained on one architecture's feature vector can only
# be applied to another if both use the SAME columns in the SAME order.
# We take the intersection of feature columns across all loaded architectures.
# ----------------------------------------------------------------------------
all_feature_sets = []
for arch, df in data.items():
    cols = set(c for c in df.columns if c.startswith('Call_'))
    all_feature_sets.append(cols)

common_features = sorted(set.intersection(*all_feature_sets))
print(f"\nCommon feature columns across {list(data.keys())}: {len(common_features)}")
dropped = set.union(*all_feature_sets) - set(common_features)
if dropped:
    print(f"Dropped (not common to all architectures): {sorted(dropped)[:15]}"
          f"{'...' if len(dropped) > 15 else ''}")

# ----------------------------------------------------------------------------
# BUILD RAW + RELATIVE-FREQUENCY MATRICES PER ARCHITECTURE (common cols only)
# ----------------------------------------------------------------------------
X_raw = {}
X_relfreq = {}
y = {}

for arch, df in data.items():
    raw = df[common_features].fillna(0).values
    total = df[common_features].fillna(0).sum(axis=1).replace(0, 1)
    relfreq = df[common_features].fillna(0).div(total, axis=0).values

    X_raw[arch] = raw
    X_relfreq[arch] = relfreq
    y[arch] = df['label'].values

# ----------------------------------------------------------------------------
# CROSS-ARCHITECTURE TRANSFER MATRIX
# ----------------------------------------------------------------------------
archs = list(data.keys())
results = []

for representation, X_dict in [('raw_counts', X_raw), ('relative_freq', X_relfreq)]:
    print("\n" + "="*70)
    print(f"REPRESENTATION: {representation}")
    print("="*70)

    for train_arch in archs:
        clf = RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        )
        clf.fit(X_dict[train_arch], y[train_arch])

        for test_arch in archs:
            y_pred = clf.predict(X_dict[test_arch])
            f1 = f1_score(y[test_arch], y_pred, average='macro')
            acc = accuracy_score(y[test_arch], y_pred)

            tag = "SAME-ARCH" if train_arch == test_arch else "CROSS-ARCH"
            print(f"  [{tag:10s}] Train={train_arch:6s} -> Test={test_arch:6s} "
                  f"| Acc={acc:.4f} | Macro F1={f1:.4f}")

            results.append({
                'representation': representation,
                'train': train_arch,
                'test': test_arch,
                'accuracy': acc,
                'macro_f1': f1,
                'same_arch': train_arch == test_arch
            })

# ----------------------------------------------------------------------------
# SUMMARY TABLE
# ----------------------------------------------------------------------------
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
print(results_df.to_string(index=False))

results_df.to_csv(f'{BASE_DIR}/paper3_cross_arch_results.csv', index=False)
print(f"\nSaved to paper3_cross_arch_results.csv")

# Quick summary: does relative_freq help more on CROSS-arch pairs specifically?
print("\n" + "="*70)
print("CROSS-ARCH ONLY: raw_counts vs relative_freq (the key comparison)")
print("="*70)
cross_only = results_df[~results_df['same_arch']]
summary = cross_only.groupby('representation')['macro_f1'].agg(['mean', 'min', 'max'])
print(summary)

LOADING SAVED SAMPLES
x86: (199655, 136), malware% = 77.44
ARM: (199903, 135), malware% = 96.00

Common feature columns across ['x86', 'ARM']: 101
Dropped (not common to all architectures): ['Call_cacheflush', 'Call_cloc', 'Call_clock_gettime64', 'Call_clock_nanosleep_time64', 'Call_dup', 'Call_epoll_wait', 'Call_fc', 'Call_fstatat64', 'Call_ftruncate64', 'Call_ge', 'Call_get', 'Call_getd', 'Call_getrandom', 'Call_gett', 'Call_getti']...

REPRESENTATION: raw_counts
  [SAME-ARCH ] Train=x86    -> Test=x86    | Acc=0.9889 | Macro F1=0.9843
  [CROSS-ARCH] Train=x86    -> Test=ARM    | Acc=0.9789 | Macro F1=0.8893
  [CROSS-ARCH] Train=ARM    -> Test=x86    | Acc=0.8564 | Macro F1=0.7280
  [SAME-ARCH ] Train=ARM    -> Test=ARM    | Acc=0.9976 | Macro F1=0.9846

REPRESENTATION: relative_freq
  [SAME-ARCH ] Train=x86    -> Test=x86    | Acc=0.9722 | Macro F1=0.9615
  [CROSS-ARCH] Train=x86    -> Test=ARM    | Acc=0.9767 | Macro F1=0.8807
  [CROSS-ARCH] Train=ARM    -> Test=x86    | Acc=0.8548

In [ ]:
# ============================================================================
# CHECK: does any Hash value appear in more than one architecture's sample?
# This is the only genuine residual leakage risk for the 12 cross-arch pairs
# (same-architecture duplicate hashes are a separate, already-handled issue).
# ============================================================================

import pandas as pd

BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86': f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM': f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS': f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}

hash_sets = {}
for arch, path in ARCH_FILES.items():
    df = pd.read_parquet(path, columns=['Hash'])
    hash_sets[arch] = set(df['Hash'])
    print(f"{arch}: {len(hash_sets[arch]):,} unique hashes")

print("\n" + "="*60)
print("CHECKING FOR HASH OVERLAP ACROSS ARCHITECTURES")
print("="*60)

archs = list(hash_sets.keys())
any_overlap = False
for i in range(len(archs)):
    for j in range(i + 1, len(archs)):
        a, b = archs[i], archs[j]
        overlap = hash_sets[a] & hash_sets[b]
        if overlap:
            any_overlap = True
            print(f"[!] {a} <-> {b}: {len(overlap)} shared hashes")
        else:
            print(f"[OK] {a} <-> {b}: no shared hashes")

if not any_overlap:
    print("\nNo cross-architecture hash overlap found in the sampled data.")
    print("The 12 cross-architecture transfer pairs have no shared-binary")
    print("leakage risk and do not require a held-out source split.")
else:
    print("\n[!] Overlap found - this DOES affect cross-arch leakage claims")
    print("    and would need to be addressed before finalizing the paper.")

x86: 3,815 unique hashes
ARM: 1,992 unique hashes
MIPS: 3,997 unique hashes
MIPSEL: 3,868 unique hashes

CHECKING FOR HASH OVERLAP ACROSS ARCHITECTURES
[OK] x86 <-> ARM: no shared hashes
[OK] x86 <-> MIPS: no shared hashes
[OK] x86 <-> MIPSEL: no shared hashes
[OK] ARM <-> MIPS: no shared hashes
[OK] ARM <-> MIPSEL: no shared hashes
[OK] MIPS <-> MIPSEL: no shared hashes

No cross-architecture hash overlap found in the sampled data.
The 12 cross-architecture transfer pairs have no shared-binary
leakage risk and do not require a held-out source split.


In [ ]:
# ============================================================================
# TEST INDIVIDUAL X86 SAMPLES (raw trace files) AGAINST THE PAPER'S MODEL
# Handles both "27 read" (pre-counted) and "read" (one syscall per line) formats.
# Builds the SAME 87-common-feature representation used throughout the paper,
# runs each file through the x86 same-architecture model, and explains each
# prediction with SHAP - for a "Qualitative Case Study" section.
# ============================================================================

import os
import re
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'

# --- Point these at your actual folders of raw trace files ---
MALWARE_DIR = f'{BASE_DIR}/new_x86_malware_samples'
BENIGN_DIR = f'{BASE_DIR}/new_x86_benign_samples'

# ----------------------------------------------------------------------------
# STEP 1: TRAIN THE X86 SAME-ARCHITECTURE MODEL (87 common features)
# This must match the paper's methodology exactly, so we rebuild it the same
# way rather than assuming an old saved model file is still consistent.
# ----------------------------------------------------------------------------

print("Rebuilding the x86 same-architecture model (87 common features)...")

ARCH_FILES = {
    'x86': f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM': f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS': f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
data = {}
for arch, path in ARCH_FILES.items():
    df = pd.read_parquet(path, columns=None)
    data[arch] = df

all_feature_sets = [set(c for c in df.columns if c.startswith('Call_')) for df in data.values()]
common_features = sorted(set.intersection(*all_feature_sets))
print(f"Common features: {len(common_features)}")

x86_df = data['x86'].copy()
x86_df['label'] = (x86_df['MalwareFamily'] != 'Benign').astype(int)

X = x86_df[common_features].fillna(0).values
y = x86_df['label'].values
groups = x86_df['Hash'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

clf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
clf.fit(X[train_idx], y[train_idx])
explainer = shap.TreeExplainer(clf)
print("Model ready.\n")

# ----------------------------------------------------------------------------
# STEP 2: PARSE RAW TRACE FILES INTO THE SAME 87-FEATURE REPRESENTATION
# ----------------------------------------------------------------------------

def parse_trace_file(file_path, feature_cols):
    """Handles both 'count syscall' and plain 'syscall' per-line formats."""
    syscall_counts = {}
    with open(file_path, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                count = int(parts[0])
                syscall = parts[1].strip()
                if syscall not in ['WIFEXITED', 'WEXITSTATUS']:
                    syscall_counts[syscall] = syscall_counts.get(syscall, 0) + count
            else:
                match = re.match(r'^([a-zA-Z_][a-zA-Z0-9_]*)', line)
                if match:
                    syscall = match.group(1)
                    if syscall not in ['+++', '---', 'exit', 'Process', 'WIFEXITED', 'WEXITSTATUS']:
                        syscall_counts[syscall] = syscall_counts.get(syscall, 0) + 1

    feature_vector = np.zeros(len(feature_cols))
    for i, col in enumerate(feature_cols):
        syscall_name = col.replace('Call_', '')
        if syscall_name in syscall_counts:
            feature_vector[i] = syscall_counts[syscall_name]
        else:
            lower_name = syscall_name.lower()
            for key in syscall_counts:
                if key.lower() == lower_name:
                    feature_vector[i] = syscall_counts[key]
                    break
    return feature_vector

# ----------------------------------------------------------------------------
# STEP 3: RUN EVERY NEW SAMPLE FILE THROUGH THE MODEL
# ----------------------------------------------------------------------------

def process_directory(directory, expected_label, feature_cols):
    results = []
    if not os.path.exists(directory):
        print(f"[!] Directory not found: {directory} - skipping.")
        return results

    files = [f for f in os.listdir(directory) if not f.startswith('.')]
    print(f"\nFound {len(files)} files in {directory}")

    for fname in files:
        fpath = os.path.join(directory, fname)
        x_vec = parse_trace_file(fpath, feature_cols).reshape(1, -1)

        pred = clf.predict(x_vec)[0]
        proba = clf.predict_proba(x_vec)[0]
        malware_prob = proba[1]

        pred_label = "MALWARE" if pred == 1 else "BENIGN"
        correct = "CORRECT" if pred_label == expected_label else "*** INCORRECT ***"

        print(f"\n--- {fname} ---")
        print(f"  Expected label:  {expected_label}")
        print(f"  Predicted label: {pred_label}  (malware probability: {malware_prob:.4f})")
        print(f"  Result:          {correct}")

        sv = explainer.shap_values(x_vec)
        if isinstance(sv, list):
            instance_shap = sv[1][0]
        elif sv.ndim == 3:
            instance_shap = sv[0, :, 1]
        else:
            instance_shap = sv[0]

        top5_idx = np.argsort(np.abs(instance_shap))[-5:][::-1]
        print(f"  Top 5 features driving this prediction:")
        for idx in top5_idx:
            feat_name = feature_cols[idx].replace('Call_', '')
            feat_value = x_vec[0][idx]
            shap_val = instance_shap[idx]
            direction = "-> pushes toward MALWARE" if shap_val > 0 else "-> pushes toward BENIGN"
            print(f"    {feat_name:20s} value={feat_value:8.2f}  SHAP={shap_val:+.4f}  {direction}")

        results.append({
            'file': fname,
            'expected_label': expected_label,
            'predicted_label': pred_label,
            'malware_probability': malware_prob,
            'correct': pred_label == expected_label
        })
    return results

print("="*70)
print("PROCESSING MALWARE SAMPLES")
print("="*70)
malware_results = process_directory(MALWARE_DIR, "MALWARE", common_features)

print("\n" + "="*70)
print("PROCESSING BENIGN SAMPLES")
print("="*70)
benign_results = process_directory(BENIGN_DIR, "BENIGN", common_features)

# ----------------------------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------------------------
all_results = pd.DataFrame(malware_results + benign_results)
if len(all_results) > 0:
    print("\n" + "="*70)
    print("CASE STUDY SUMMARY (paper-ready table)")
    print("="*70)
    print(all_results.to_string(index=False))
    all_results.to_csv(f'{BASE_DIR}/paper3_new_sample_case_study.csv', index=False)
    print(f"\nSaved to paper3_new_sample_case_study.csv")
    print(f"Overall accuracy on new samples: {all_results['correct'].mean()*100:.1f}% "
          f"({all_results['correct'].sum()}/{len(all_results)})")
else:
    print("\n[!] No files processed - check MALWARE_DIR and BENIGN_DIR paths above.")

Rebuilding the x86 same-architecture model (87 common features)...
Common features: 87
Model ready.

PROCESSING MALWARE SAMPLES

Found 4 files in /content/drive/My Drive/Colab Notebooks2/new_x86_malware_samples

--- testing_sample_last_gayf.txt ---
  Expected label:  MALWARE
  Predicted label: MALWARE  (malware probability: 0.8450)
  Result:          CORRECT
  Top 5 features driving this prediction:
    fcntl                value=    8.00  SHAP=+0.1858  -> pushes toward MALWARE
    getpid               value=    5.00  SHAP=+0.1021  -> pushes toward MALWARE
    nanosleep            value=    4.00  SHAP=-0.0505  -> pushes toward BENIGN
    ioctl                value=    2.00  SHAP=+0.0499  -> pushes toward MALWARE
    wait4                value=    1.00  SHAP=-0.0379  -> pushes toward BENIGN

--- testing_sample_mirai_wrong_judge.txt ---
  Expected label:  MALWARE
  Predicted label: MALWARE  (malware probability: 0.6750)
  Result:          CORRECT
  Top 5 features driving this prediction:

In [ ]:
from google.colab import drive
import os

# Forcefully remove the mount point directory if it exists and is not empty
# This is a strong measure to fix persistent mount errors.
if os.path.exists("/content/drive"):
    print("Cleaning up existing /content/drive directory...")
    !rm -rf "/content/drive" # Use shell command for recursive force delete

# Create an empty directory for the mount point
os.makedirs("/content/drive", exist_ok=True)

# Then, mount it again with force_remount
drive.mount("/content/drive", force_remount=True)
print("Drive mount command issued. Please follow authentication steps if prompted.")

Cleaning up existing /content/drive directory...
rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled
Mounted at /content/drive
Drive mount command issued. Please follow authentication steps if prompted.


In [ ]:
import pandas as pd
import os
import pyarrow.parquet as pq # Import pyarrow for more detailed parquet handling

# IMPORTANT: Replace 'path/to/your/strace.parquet' with the actual path to your file in Google Drive
# For example, if your file is in 'My Drive/Colab Notebooks/strace.parquet',
# the path would be '/content/drive/My Drive/Colab Notebooks/strace.parquet'
parquet_file_path = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_x86' # <--- ENSURE THIS PATH IS CORRECT

print(f"Attempting to access file at: {parquet_file_path}")

if not os.path.exists(parquet_file_path):
    print(f"Error: File not found at {parquet_file_path}. Please double-check the path and ensure your Google Drive is mounted correctly.")
else:
    print("File found! Attempting to read Parquet metadata...")
    try:
        # Try to read only the schema first to check file integrity
        parquet_schema = pq.read_schema(parquet_file_path)
        print("Parquet file schema loaded successfully:")
        print(parquet_schema.to_string())

        print("\nAttempting to read the first 5 rows of the Parquet file using pyarrow...")
        # Explicitly use pyarrow to read a limited number of rows by batch
        parquet_file = pq.ParquetFile(parquet_file_path)

        # Read the first batch of 5 rows
        first_rows_table = next(parquet_file.iter_batches(batch_size=5))
        df = first_rows_table.to_pandas()

        print("First 5 rows of the Parquet file:")
        print(df.head())

    except Exception as e:
        print(f"An error occurred while reading the Parquet file: {e}")
        print("This could indicate an issue with the file's internal structure or compatibility.")
        print("If the kernel is restarting, ensure pyarrow is up-to-date or re-download the file.")

In [ ]:
# Execute the Linux 'ls' command to check the directory directly
!ls '/content/drive/My Drive'

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Define your file path
parquet_file_path = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_x86'

# 1. Read ONLY the schema (columns and data types) without loading any data rows
print("--- DATASET COLUMNS ---")
schema = pq.read_schema(parquet_file_path)
for field in schema:
    print(f"Column: {field.name:<30} | Type: {field.type}")

print("\n" + "="*50 + "\n")

# 2. Read just the first 5 rows to inspect the actual data and identify the label column
print("--- PREVIEW OF THE FIRST 5 ROWS ---")
# Using pyarrow's dataset API to read just a tiny batch (very fast and memory-safe)
pf = pq.ParquetFile(parquet_file_path)
first_rows_df = next(pf.iter_batches(batch_size=5)).to_pandas()

# Display the preview
import ace_tools as pt; pt.display_html(first_rows_df) # Optional if using Colab's interactive tables
print(first_rows_df.head())

print("\n" + "="*50 + "\n")

# 3. Check unique values in likely label columns (e.g., 'class', 'label', 'category')
# Common label column names are 'class', 'label', 'category', 'is_malware', or 'type'.
# Replace 'class' with whatever target column you identified in Step 1.
target_column = 'class'  # Update this based on the column list outputted from Step 1

if target_column in schema.names:
    print(f"--- UNIQUE VALUES IN '{target_column}' COLUMN ---")
    # Read only that specific column from the file to save RAM
    labels_df = pq.read_table(parquet_file_path, columns=[target_column]).to_pandas()
    print(labels_df[target_column].value_counts())
else:
    print(f"Could not find a column named '{target_column}'. Please look at the printed schema above to find the correct column name.")

--- DATASET COLUMNS ---


FileNotFoundError: [Errno 2] Failed to open local file '/content/drive/My Drive/Colab Notebooks2/strace.parquet_x86'. Detail: [errno 2] No such file or directory

In [ ]:
import os
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# Define the original parquet file path (from previous cells)
parquet_file_path = '/content/drive/My Drive/Colab Notebooks2/strace.parquet_x86'

# Define a temporary directory to store smaller sub-files
# MODIFIED: Changed output_dir to a Google Drive path for persistence
output_dir = '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output'
os.makedirs(output_dir, exist_ok=True)

print(f"Splitting '{os.path.basename(parquet_file_path)}' into smaller files in '{output_dir}'...")

try:
    # Open the Parquet file
    parquet_file = pq.ParquetFile(parquet_file_path)

    # Determine a suitable chunk size. For an 11MB file, 10000 rows per chunk might be reasonable.
    # Adjust this based on your file's row count and memory.
    chunk_size = 10000

    file_count = 0
    # Iterate over batches and save each as a new parquet file
    for i, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
        sub_df = batch.to_pandas()
        output_file = os.path.join(output_dir, f'chunk_{i:04d}.parquet')
        sub_df.to_parquet(output_file, index=False)
        file_count += 1
        print(f"Saved {len(sub_df)} rows to '{output_file}'")

    print(f"\nFinished splitting. Created {file_count} sub-files in '{output_dir}'.")

except Exception as e:
    print(f"An error occurred during file splitting: {e}")
    print("Please ensure the original file path is correct and accessible.")

Splitting 'strace.parquet_x86' into smaller files in '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output'...
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0000.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0001.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0002.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0003.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0004.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0005.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0006.parquet'
Saved 10000 rows to '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output/chunk_0007.parquet'
Saved 10000 rows to 

In [ ]:
import os
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa

# Define the directory where sub-files were saved (now pointing to Drive)
# MODIFIED: Updated output_dir to match the Drive path used for splitting
output_dir = '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output'

all_sub_files = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith('.parquet')]
all_sub_files.sort() # Ensure consistent order if order matters

print(f"Reading and concatenating {len(all_sub_files)} sub-files from '{output_dir}' using PyArrow...")

# Check if there are any files to read
if not all_sub_files:
    print("No sub-files found to read.")
    df_full_content = pd.DataFrame() # Initialize an empty DataFrame
else:
    try:
        # Read all parquet files into a single PyArrow Table
        # This is often more memory efficient than than pd.concat for many files
        full_table = pq.read_table(all_sub_files)

        # Convert the PyArrow Table to a Pandas DataFrame
        df_full_content = full_table.to_pandas()

        print("\nAll sub-files concatenated successfully into 'df_full_content'!")
        print(f"Final DataFrame 'df_full_content' has {len(df_full_content)} rows and {len(df_full_content.columns)} columns.")

    except Exception as e:
        print(f"An error occurred during file reading and concatenation: {e}")
        print("The 'df_full_content' DataFrame might be empty or incomplete.")
        df_full_content = pd.DataFrame() # Initialize an empty DataFrame on error

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output'

In [ ]:
import os
import pandas as pd

# Define the directory where sub-files were saved
parquet_output_dir = '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output'
# Define the directory to save CSV files
csv_output_dir = '/content/drive/My Drive/Colab Notebooks2/csv_output_files'
os.makedirs(csv_output_dir, exist_ok=True)

all_sub_files = [os.path.join(parquet_output_dir, f) for f in os.listdir(parquet_output_dir) if f.endswith('.parquet')]
all_sub_files.sort() # Ensure consistent order if order matters

print(f"Starting interactive processing of {len(all_sub_files)} sub-files from '{parquet_output_dir}'...")
print(f"CSV files will be saved to '{csv_output_dir}'.")
print("You will be prompted to continue after each file is processed/saved.")

# Initialize a list to store summary information (e.g., row counts)
processed_file_summaries = []

save_all_csv = False # Flag to control saving all files without prompting

for i, file_path in enumerate(all_sub_files):
    try:
        print(f"\n--- Processing file {i+1}/{len(all_sub_files)}: '{os.path.basename(file_path)}' ---")

        # Read the current sub-file into a DataFrame
        current_df = pd.read_parquet(file_path)

        print(f"  Read {len(current_df)} rows.")
        print("  First 3 rows:")
        display(current_df.head(3))

        # Define CSV output path
        csv_file_name = os.path.basename(file_path).replace('.parquet', '.csv')
        csv_file_path = os.path.join(csv_output_dir, csv_file_name)

        if not save_all_csv:
            # For the first file, or if not saving all, ask the user
            response = input(f"Do you want to save '{csv_file_name}' to CSV? (y/n/all/q): ")
            if response.lower() == 'y' or response.lower() == 'all':
                print(f"  Saving '{csv_file_name}' to CSV. This might take a moment...")
                current_df.to_csv(csv_file_path, index=False)
                print(f"  '{csv_file_name}' saved successfully to '{csv_file_path}'.")
                if response.lower() == 'all':
                    save_all_csv = True
            elif response.lower() == 'q':
                print("Exiting iterative processing as requested.")
                break
            else: # 'n' or any other input, just proceed without saving this file
                print(f"  Skipping saving '{csv_file_name}' to CSV.")
        else:
            # If save_all_csv is True, save without prompting
            print(f"  Saving '{csv_file_name}' to CSV automatically. This might take a moment...")
            current_df.to_csv(csv_file_path, index=False)
            print(f"  '{csv_file_name}' saved successfully to '{csv_file_path}'.")

        # Example operation: Calculate a simple sum or count
        if 'Call_write' in current_df.columns:
            write_sum = current_df['Call_write'].sum()
            print(f"  Sum of 'Call_write' in this chunk: {write_sum}")

        # Store summary information if needed
        processed_file_summaries.append({
            'file': os.path.basename(file_path),
            'rows': len(current_df),
            'sum_call_write': write_sum if 'Call_write' in current_df.columns else None
        })

        # IMPORTANT: The 'current_df' variable will be overwritten in the next iteration,
        # and its memory will be freed by Python's garbage collector. This is key for memory efficiency.

        # Pause and ask the user to continue after potential save, if not in 'save_all' mode
        if not save_all_csv and i < len(all_sub_files) - 1:
            print("--- Ready for next file ---") # Explicitly state ready for input
            response = input("Press Enter to process the next file, or type 'q' and Enter to quit: ")
            if response.lower() == 'q':
                print("Exiting iterative processing as requested.")
                break

    except Exception as e:
        print(f"  Error processing '{os.path.basename(file_path)}': {e}")
        # Even on error, provide an option to continue or quit
        print("--- Ready for next file (after error) ---") # Explicitly state ready for input
        response = input("Press Enter to process the next file, or type 'q' and Enter to quit: ")
        if response.lower() == 'q':
            print("Exiting iterative processing as requested due to error.")
            break

print("\nIterative processing complete.")
if processed_file_summaries:
    print("Summary of processed files:")
    display(pd.DataFrame(processed_file_summaries))
else:
    print("No files were processed or no summary data collected.")

Starting interactive processing of 2586 sub-files from '/content/drive/My Drive/Colab Notebooks2/parquet_sub_files_output'...
CSV files will be saved to '/content/drive/My Drive/Colab Notebooks2/csv_output_files'.
You will be prompted to continue after each file is processed/saved.

--- Processing file 1/2586: 'chunk_0000.parquet' ---
  Read 10000 rows.
  First 3 rows:


,Call__llseek,Call__newselect,Call_accept,Call_access,Call_bind,Call_brk,Call_chdir,Call_chmod,Call_chown32,Call_clock_gettime,...,Call_uname,Call_unlink,Call_utimensat,Call_vfork,Call_wait4,Call_write,Call_writev,Hash,MalwareFamily,Arch
0,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86
1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86
2,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86


Do you want to save 'chunk_0000.csv' to CSV? (y/n/all/q): all
  Saving 'chunk_0000.csv' to CSV. This might take a moment...
  'chunk_0000.csv' saved successfully to '/content/drive/My Drive/Colab Notebooks2/csv_output_files/chunk_0000.csv'.
  Sum of 'Call_write' in this chunk: 40960

--- Processing file 2/2586: 'chunk_0001.parquet' ---
  Read 10000 rows.
  First 3 rows:


,Call__llseek,Call__newselect,Call_accept,Call_access,Call_bind,Call_brk,Call_chdir,Call_chmod,Call_chown32,Call_clock_gettime,...,Call_uname,Call_unlink,Call_utimensat,Call_vfork,Call_wait4,Call_write,Call_writev,Hash,MalwareFamily,Arch
0,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86
1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86
2,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86


  Saving 'chunk_0001.csv' to CSV automatically. This might take a moment...
  'chunk_0001.csv' saved successfully to '/content/drive/My Drive/Colab Notebooks2/csv_output_files/chunk_0001.csv'.
  Sum of 'Call_write' in this chunk: 0

--- Processing file 3/2586: 'chunk_0002.parquet' ---
  Read 10000 rows.
  First 3 rows:


,Call__llseek,Call__newselect,Call_accept,Call_access,Call_bind,Call_brk,Call_chdir,Call_chmod,Call_chown32,Call_clock_gettime,...,Call_uname,Call_unlink,Call_utimensat,Call_vfork,Call_wait4,Call_write,Call_writev,Hash,MalwareFamily,Arch
0,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86
1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86
2,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,004f468091d776e6b7ee4f8f276cfedf070bdd24f7e5b6...,Benign,x86


... (105 outputs left out)

,Call__llseek,Call__newselect,Call_accept,Call_access,Call_bind,Call_brk,Call_chdir,Call_chmod,Call_chown32,Call_clock_gettime,...,Call_uname,Call_unlink,Call_utimensat,Call_vfork,Call_wait4,Call_write,Call_writev,Hash,MalwareFamily,Arch
0,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,1fe05e50b9f79c14017765e22401f13153fe65d1cbd92f...,Benign,x86
1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,1fe05e50b9f79c14017765e22401f13153fe65d1cbd92f...,Benign,x86
2,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,1fe05e50b9f79c14017765e22401f13153fe65d1cbd92f...,Benign,x86


  Saving 'chunk_0055.csv' to CSV automatically. This might take a moment...
  'chunk_0055.csv' saved successfully to '/content/drive/My Drive/Colab Notebooks2/csv_output_files/chunk_0055.csv'.
  Sum of 'Call_write' in this chunk: 0

--- Processing file 57/2586: 'chunk_0056.parquet' ---
  Read 10000 rows.
  First 3 rows:


,Call__llseek,Call__newselect,Call_accept,Call_access,Call_bind,Call_brk,Call_chdir,Call_chmod,Call_chown32,Call_clock_gettime,...,Call_uname,Call_unlink,Call_utimensat,Call_vfork,Call_wait4,Call_write,Call_writev,Hash,MalwareFamily,Arch
0,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,1fe05e50b9f79c14017765e22401f13153fe65d1cbd92f...,Benign,x86
1,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,1fe05e50b9f79c14017765e22401f13153fe65d1cbd92f...,Benign,x86
2,0,0,0.0,0,0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,1fe05e50b9f79c14017765e22401f13153fe65d1cbd92f...,Benign,x86


  Saving 'chunk_0056.csv' to CSV automatically. This might take a moment...


In [ ]:
# ============================================================================
# PAPER 1: COMPLETE THESIS EXTENSION EXPERIMENT
# CIC-YNU-IoTMal Dataset - Memory Optimized with Checkpoint Saving
#
# ALL FUNCTIONS DEFINED - READY TO RUN, NOT using this code
# ============================================================================

import os
import pandas as pd
import numpy as np
import time
import joblib
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve
)
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. HELPER FUNCTIONS (DEFINED FIRST)
# ============================================================================

def get_feature_columns(df):
    """
    Get all Call_* columns from a dataframe.
    These are the syscall frequency features.
    """
    return [col for col in df.columns if col.startswith('Call_')]

def process_chunk(chunk, feature_cols, arch_filter='x86'):
    """
    Process a single chunk of data:
    - Filter for specified architecture
    - Extract features and labels
    - Return X, y as numpy arrays

    Args:
        chunk: pandas DataFrame chunk
        feature_cols: list of feature column names
        arch_filter: architecture to filter by (e.g., 'x86', 'ARM', 'MIPS')

    Returns:
        X: feature matrix (numpy array)
        y: label vector (numpy array)
    """
    # Filter for architecture if Arch column exists
    if 'Arch' in chunk.columns:
        chunk = chunk[chunk['Arch'] == arch_filter]

    if len(chunk) == 0:
        return None, None

    # Get labels - try multiple possible column names
    label_col = None
    if 'MalwareFamily' in chunk.columns:
        label_col = 'MalwareFamily'
        y = chunk[label_col].apply(lambda x: 1 if x != 'Benign' else 0).values
    elif 'label' in chunk.columns:
        label_col = 'label'
        y = chunk[label_col].values
    elif 'Label' in chunk.columns:
        label_col = 'Label'
        y = chunk[label_col].values
    else:
        # Check if there's a column with 'label' in the name
        for col in chunk.columns:
            if 'label' in col.lower():
                label_col = col
                y = chunk[col].values
                break

        if label_col is None:
            print(f"⚠️ No label column found! Available: {chunk.columns[:5].tolist()}")
            return None, None

    # Get features (only Call_* columns that exist in this chunk)
    available_feature_cols = [col for col in feature_cols if col in chunk.columns]

    if len(available_feature_cols) == 0:
        print(f"⚠️ No feature columns found in chunk!")
        return None, None

    X = chunk[available_feature_cols].fillna(0).values

    return X, y

def accumulate_features(X_accum, y_accum, X_new, y_new):
    """
    Accumulate features and labels for later processing.
    Efficiently stacks new data onto existing arrays.

    Args:
        X_accum: existing feature matrix (or None)
        y_accum: existing label vector (or None)
        X_new: new feature matrix to add
        y_new: new label vector to add

    Returns:
        Updated X_accum, y_accum
    """
    if X_new is None or len(X_new) == 0:
        return X_accum, y_accum

    if X_accum is None:
        return X_new, y_new

    # Handle case where feature dimensions might differ
    # (e.g., different chunks might have slightly different feature sets)
    if X_accum.shape[1] != X_new.shape[1]:
        # Find common features - but since we use the same feature_cols list,
        # this shouldn't happen. But let's handle it gracefully.
        min_dim = min(X_accum.shape[1], X_new.shape[1])
        X_accum = X_accum[:, :min_dim]
        X_new = X_new[:, :min_dim]
        print(f"   ⚠️ Feature dimensions aligned to {min_dim}")

    return np.vstack([X_accum, X_new]), np.concatenate([y_accum, y_new])

# ============================================================================
# 2. CHECKPOINT FUNCTIONS
# ============================================================================

def save_checkpoint(data, name):
    """Save checkpoint data to disk."""
    filepath = os.path.join(CHECKPOINT_DIR, f"{name}.pkl")
    joblib.dump(data, filepath)
    print(f"   💾 Checkpoint saved: {name}")

def load_checkpoint(name):
    """Load checkpoint data if it exists."""
    filepath = os.path.join(CHECKPOINT_DIR, f"{name}.pkl")
    if os.path.exists(filepath):
        print(f"   🔄 Loading checkpoint: {name}")
        return joblib.load(filepath)
    return None

def checkpoint_exists(name):
    """Check if a checkpoint exists."""
    return os.path.exists(os.path.join(CHECKPOINT_DIR, f"{name}.pkl"))

# ============================================================================
# 3. CONFIGURATION
# ============================================================================
RANDOM_STATE = 42
TOP_FEATURES = 30
TEST_SIZE = 0.3
ARCHITECTURE_FILTER = 'x86'
SAMPLE_FRACTION = 0.1  # Start with 10% - increase if memory allows
MAX_ROWS_PER_FILE = 50000
CHECKPOINT_DIR = '/content/drive/My Drive/Colab Notebooks2/checkpoints'

# Paths
CSV_INPUT_DIR = '/content/drive/My Drive/Colab Notebooks2/csv_output_files'
OUTPUT_DIR = '/content/drive/My Drive/Colab Notebooks2/thesis_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("="*70)
print("PAPER 1: CROSS-ARCHITECTURE BEHAVIORAL DRIFT EXPERIMENT")
print("="*70)
print(f"\n📁 Input directory: {CSV_INPUT_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📁 Checkpoint directory: {CHECKPOINT_DIR}")
print(f"📊 Sample fraction: {SAMPLE_FRACTION*100}%")
print(f"🏗️  Architecture filter: {ARCHITECTURE_FILTER}")

# ============================================================================
# 4. PROCESS FILES WITH CHECKPOINTS
# ============================================================================
print("\n" + "="*70)
print("STEP 1: LOADING DATASET (STREAMING WITH CHECKPOINTS)")
print("="*70)

# Check if we already processed files
processed_files_checkpoint = load_checkpoint('processed_files')
if processed_files_checkpoint is not None:
    processed_files = processed_files_checkpoint['processed_files']
    csv_files = processed_files_checkpoint['csv_files']
    feature_cols = processed_files_checkpoint['feature_cols']
    total_samples = processed_files_checkpoint['total_samples']
    X_accum = processed_files_checkpoint['X_accum']
    y_accum = processed_files_checkpoint['y_accum']
    print(f"\n✅ Resuming from checkpoint:")
    print(f"   Processed {processed_files}/{len(csv_files)} files")
    print(f"   Accumulated {total_samples:,} samples")
else:
    # Get all CSV files
    csv_files = [f for f in os.listdir(CSV_INPUT_DIR) if f.endswith('.csv')]
    csv_files.sort()
    print(f"\n📄 Found {len(csv_files)} CSV files")

    if len(csv_files) == 0:
        print("❌ No CSV files found! Please check the input directory.")
        print(f"   Directory contents: {os.listdir(CSV_INPUT_DIR)[:5]}")
        exit()

    # Initialize accumulators
    X_accum = None
    y_accum = None
    feature_cols = None
    total_samples = 0
    processed_files = 0

# Process files from where we left off
for i in range(processed_files, len(csv_files)):
    file = csv_files[i]
    file_path = os.path.join(CSV_INPUT_DIR, file)
    print(f"\n📂 Processing {i+1}/{len(csv_files)}: {file}")

    try:
        chunk_count = 0
        file_samples = 0

        for chunk in pd.read_csv(file_path, chunksize=MAX_ROWS_PER_FILE, low_memory=False):
            chunk_count += 1

            # Get feature columns from first chunk
            if feature_cols is None:
                feature_cols = get_feature_columns(chunk)
                print(f"   Found {len(feature_cols)} feature columns")

            # Sample the chunk if needed
            if SAMPLE_FRACTION < 1.0:
                chunk = chunk.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE)

            # Process chunk
            X_chunk, y_chunk = process_chunk(chunk, feature_cols, ARCHITECTURE_FILTER)

            if X_chunk is not None and len(X_chunk) > 0:
                X_accum, y_accum = accumulate_features(X_accum, y_accum, X_chunk, y_chunk)
                total_samples += len(X_chunk)
                file_samples += len(X_chunk)

            # Clear chunk from memory
            del chunk
            gc.collect()

            # Save checkpoint every 5 chunks
            if chunk_count % 5 == 0:
                save_checkpoint({
                    'processed_files': i,
                    'csv_files': csv_files,
                    'feature_cols': feature_cols,
                    'total_samples': total_samples,
                    'X_accum': X_accum,
                    'y_accum': y_accum
                }, 'processed_files')
                print(f"   💾 Checkpoint: {total_samples:,} samples processed")

        processed_files = i + 1
        print(f"   ✅ File complete: {file_samples:,} samples from this file")

        # Save checkpoint after each file
        save_checkpoint({
            'processed_files': processed_files,
            'csv_files': csv_files,
            'feature_cols': feature_cols,
            'total_samples': total_samples,
            'X_accum': X_accum,
            'y_accum': y_accum
        }, 'processed_files')

    except Exception as e:
        print(f"   ❌ Error: {e}")
        # Save checkpoint on error so we can resume
        save_checkpoint({
            'processed_files': i,
            'csv_files': csv_files,
            'feature_cols': feature_cols,
            'total_samples': total_samples,
            'X_accum': X_accum,
            'y_accum': y_accum
        }, 'processed_files')
        print(f"   💾 Emergency checkpoint saved at file {i}")
        continue

print(f"\n✅ Total processed: {total_samples:,} samples from {processed_files} files")

if total_samples == 0:
    print("❌ No samples collected! Check your data.")
    exit()

# Delete checkpoint after full load
if os.path.exists(os.path.join(CHECKPOINT_DIR, 'processed_files.pkl')):
    os.remove(os.path.join(CHECKPOINT_DIR, 'processed_files.pkl'))
    print("🗑️  Removed processing checkpoint")

# ============================================================================
# 5. PREPARE FINAL DATASET WITH CHECKPOINT
# ============================================================================
print("\n" + "="*70)
print("STEP 2: PREPARING FINAL DATASET")
print("="*70)

# Check if dataset preparation is already done
if checkpoint_exists('dataset_prepared'):
    dataset = load_checkpoint('dataset_prepared')
    X_accum = dataset['X']
    y_accum = dataset['y']
    feature_cols = dataset['feature_cols']
    print(f"\n✅ Loaded prepared dataset: {X_accum.shape[0]:,} samples")
else:
    # Shuffle the data
    X_accum, y_accum = shuffle(X_accum, y_accum, random_state=RANDOM_STATE)

    # Save checkpoint
    save_checkpoint({
        'X': X_accum,
        'y': y_accum,
        'feature_cols': feature_cols
    }, 'dataset_prepared')
    print(f"✅ Dataset prepared and checkpointed")

print(f"\n📊 Final dataset: {X_accum.shape[0]:,} samples × {X_accum.shape[1]} features")
print(f"   Benign:  {(y_accum == 0).sum():,}")
print(f"   Malware: {(y_accum == 1).sum():,}")
print(f"   Malware %: {((y_accum == 1).sum() / len(y_accum)) * 100:.2f}%")

# ============================================================================
# 6. TRAIN-TEST SPLIT WITH CHECKPOINT
# ============================================================================
print("\n" + "="*70)
print("STEP 3: TRAIN-TEST SPLIT")
print("="*70)

if checkpoint_exists('train_test_split'):
    split_data = load_checkpoint('train_test_split')
    X_train = split_data['X_train']
    X_test = split_data['X_test']
    y_train = split_data['y_train']
    y_test = split_data['y_test']
    print(f"\n✅ Loaded train-test split from checkpoint")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_accum, y_accum,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_accum
    )

    save_checkpoint({
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }, 'train_test_split')
    print(f"✅ Train-test split saved")

# Free up memory
del X_accum, y_accum
gc.collect()

print(f"\n📊 Split:")
print(f"   Training: {len(X_train):,} samples")
print(f"   Testing:  {len(X_test):,} samples")

# ============================================================================
# 7. FEATURE SELECTION WITH CHECKPOINT
# ============================================================================
print("\n" + "="*70)
print("STEP 4: FEATURE SELECTION")
print("="*70)

if checkpoint_exists('feature_selection'):
    fs_data = load_checkpoint('feature_selection')
    selected_features = fs_data['selected_features']
    selected_indices = fs_data['selected_indices']
    feature_importance_df = fs_data['feature_importance_df']
    print(f"\n✅ Loaded feature selection from checkpoint")
    print(f"   Selected {len(selected_features)} features")
else:
    print(f"   Selecting top {TOP_FEATURES} features...")

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    importance_matrix = np.zeros(X_train.shape[1])

    rf_selector = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    for fold_idx, (train_idx, _) in enumerate(cv.split(X_train, y_train)):
        X_fold = X_train[train_idx]
        y_fold = y_train[train_idx]
        rf_selector.fit(X_fold, y_fold)
        importance_matrix += rf_selector.feature_importances_
        print(f"   Fold {fold_idx+1}/3 complete")

    importance_matrix /= cv.get_n_splits()

    # Create feature importance dataframe
    feature_importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importance_matrix
    }).sort_values('importance', ascending=False)

    # Select top features
    selected_features = feature_importance_df.head(TOP_FEATURES)['feature'].tolist()
    selected_indices = [feature_cols.index(f) for f in selected_features]

    # Save checkpoint
    save_checkpoint({
        'selected_features': selected_features,
        'selected_indices': selected_indices,
        'feature_importance_df': feature_importance_df
    }, 'feature_selection')
    print(f"✅ Feature selection saved")

print(f"\n✅ Selected {len(selected_features)} features:")
for i, f in enumerate(selected_features[:10]):
    imp = feature_importance_df[feature_importance_df['feature'] == f]['importance'].values[0]
    print(f"   {i+1:2d}. {f.replace('Call_', ''):20s} {imp:.4f}")

# Reduce feature space
X_train_selected = X_train[:, selected_indices]
X_test_selected = X_test[:, selected_indices]
print(f"✅ Feature matrix reduced: {X_train_selected.shape[1]} features")

# ============================================================================
# 8. SCALING WITH CHECKPOINT
# ============================================================================
print("\n" + "="*70)
print("STEP 5: FEATURE SCALING")
print("="*70)

if checkpoint_exists('scaling'):
    scaling_data = load_checkpoint('scaling')
    scaler = scaling_data['scaler']
    X_train_scaled = scaling_data['X_train_scaled']
    X_test_scaled = scaling_data['X_test_scaled']
    print(f"✅ Loaded scaling from checkpoint")
else:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_selected)
    X_test_scaled = scaler.transform(X_test_selected)

    save_checkpoint({
        'scaler': scaler,
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled
    }, 'scaling')
    print(f"✅ Scaling saved")

print(f"✅ Features scaled")

# Free up memory
del X_train_selected, X_test_selected
gc.collect()

# ============================================================================
# 9. MODEL TRAINING WITH CHECKPOINT
# ============================================================================
print("\n" + "="*70)
print("STEP 6: MODEL TRAINING AND EVALUATION")
print("="*70)

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "SVM": SVC(
        kernel='linear',
        probability=True,
        random_state=RANDOM_STATE
    )
}

results = {}
overhead_results = {}
best_model = None
best_model_name = None
best_accuracy = 0

for name, model in models.items():
    # Check if this model is already trained
    model_checkpoint = load_checkpoint(f'model_{name}')
    if model_checkpoint is not None:
        print(f"\n{'='*50}")
        print(f"📊 Model: {name} (Loaded from checkpoint)")
        print(f"{'='*50}")

        model = model_checkpoint['model']
        y_pred = model_checkpoint['y_pred']
        y_proba = model_checkpoint.get('y_proba')
        training_time = model_checkpoint['training_time']
        prediction_time = model_checkpoint['prediction_time']

    else:
        print(f"\n{'='*50}")
        print(f"📊 Model: {name} (Training)")
        print(f"{'='*50}")

        train_start = time.time()
        model.fit(X_train_scaled, y_train)
        training_time = time.time() - train_start

        predict_start = time.time()
        y_pred = model.predict(X_test_scaled)
        prediction_time = time.time() - predict_start

        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test_scaled)[:, 1]
        else:
            y_proba = None

        # Save model checkpoint
        save_checkpoint({
            'model': model,
            'y_pred': y_pred,
            'y_proba': y_proba,
            'training_time': training_time,
            'prediction_time': prediction_time
        }, f'model_{name}')

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Quick CV (3-fold for speed)
    try:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=3)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
    except:
        cv_mean = 0
        cv_std = 0

    print(f"   Accuracy:            {acc:.4f}")
    print(f"   Precision (weighted): {prec:.4f}")
    print(f"   Recall (weighted):    {rec:.4f}")
    print(f"   F1-Score (weighted):  {f1:.4f}")
    print(f"   CV Mean Accuracy:    {cv_mean:.4f} (±{cv_std:.4f})")
    print(f"   Training Time:       {training_time:.4f}s")
    print(f"   Prediction Time:     {prediction_time:.4f}s")

    print(f"\n   Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malware']))

    cm = confusion_matrix(y_test, y_pred)
    print(f"\n   Confusion Matrix:")
    print(f"   [[{cm[0,0]:,} {cm[0,1]:,}]")
    print(f"    [{cm[1,0]:,} {cm[1,1]:,}]]")

    results[name] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    overhead_results[name] = {
        'training_time_sec': training_time,
        'prediction_time_sec': prediction_time
    }

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

print(f"\n{'='*50}")
print(f"🏆 BEST MODEL: {best_model_name} (Accuracy: {best_accuracy:.4f})")
print(f"{'='*50}")

# ============================================================================
# 10. SAVE FINAL RESULTS
# ============================================================================
print("\n" + "="*70)
print("STEP 7: SAVING FINAL RESULTS")
print("="*70)

# Save feature importance
feature_importance_df.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'), index=False)
print(f"✅ Saved: feature_importance.csv")

# Save selected features
pd.DataFrame({'selected_feature': selected_features}).to_csv(
    os.path.join(OUTPUT_DIR, 'selected_features.csv'), index=False
)
print(f"✅ Saved: selected_features.csv")

# Save overhead results
pd.DataFrame(overhead_results).T.to_csv(os.path.join(OUTPUT_DIR, 'overhead_results.csv'))
print(f"✅ Saved: overhead_results.csv")

# Save model and scaler
joblib.dump(best_model, os.path.join(OUTPUT_DIR, 'best_model.pkl'))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.pkl'))
print(f"✅ Saved: best_model.pkl, scaler.pkl")

# Save results summary
results_summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'CV_Accuracy': [results[m]['cv_mean'] for m in results]
})
results_summary.to_csv(os.path.join(OUTPUT_DIR, 'results_summary.csv'), index=False)
print(f"✅ Saved: results_summary.csv")

# ============================================================================
# 11. VISUALIZATIONS
# ============================================================================
print("\n" + "="*70)
print("STEP 8: GENERATING VISUALIZATIONS")
print("="*70)

# Feature Importance Plot
plt.figure(figsize=(10, 8))
top15 = feature_importance_df.head(15)
plt.barh(top15['feature'].str.replace('Call_', '')[::-1], top15['importance'][::-1])
plt.title(f'Top 15 Most Important Features (x86 Architecture)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Saved: feature_importance.png")

# ROC Curve
if results[best_model_name]['y_proba'] is not None:
    y_proba = results[best_model_name]['y_proba']
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}', linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {best_model_name}')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: roc_curve.png (AUC = {roc_auc:.3f})")

# Precision-Recall Curve
if results[best_model_name]['y_proba'] is not None:
    precision, recall, _ = precision_recall_curve(y_test, y_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, linewidth=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {best_model_name}')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'precision_recall_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: precision_recall_curve.png")

# Confusion Matrix
cm = results[best_model_name]['confusion_matrix']
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malware'],
            yticklabels=['Benign', 'Malware'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Saved: confusion_matrix.png")

# ============================================================================
# 12. FINAL SUMMARY
# ============================================================================
print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)
print("\n" + results_summary.to_string(index=False))

print(f"\n{'='*50}")
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy:  {best_accuracy:.4f}")
print(f"   CV Mean:   {results[best_model_name]['cv_mean']:.4f}")
print(f"{'='*50}")

print(f"\n📁 All results saved to: {OUTPUT_DIR}")
print("\n📋 Saved files:")
print("   ✅ feature_importance.csv")
print("   ✅ selected_features.csv")
print("   ✅ overhead_results.csv")
print("   ✅ results_summary.csv")
print("   ✅ best_model.pkl")
print("   ✅ scaler.pkl")
print("   ✅ feature_importance.png")
print("   ✅ roc_curve.png")
print("   ✅ precision_recall_curve.png")
print("   ✅ confusion_matrix.png")

# ============================================================================
# 13. CLEANUP CHECKPOINTS
# ============================================================================
print("\n" + "="*70)
print("STEP 9: CLEANING UP CHECKPOINTS")
print("="*70)

print("\n💾 Checkpoints preserved in:", CHECKPOINT_DIR)
print("   To delete later (free up space):")
print(f"   !rm -rf {CHECKPOINT_DIR}/*")

print("\n" + "="*70)
print("🎉 PAPER 1 EXPERIMENT COMPLETE!")
print("="*70)

PAPER 1: CROSS-ARCHITECTURE BEHAVIORAL DRIFT EXPERIMENT

📁 Input directory: /content/drive/My Drive/Colab Notebooks2/csv_output_files
📁 Output directory: /content/drive/My Drive/Colab Notebooks2/thesis_results
📁 Checkpoint directory: /content/drive/My Drive/Colab Notebooks2/checkpoints
📊 Sample fraction: 10.0%
🏗️  Architecture filter: x86

STEP 1: LOADING DATASET (STREAMING WITH CHECKPOINTS)

📄 Found 26 CSV files

📂 Processing 1/26: chunk_0000.csv
   Found 132 feature columns
   ✅ File complete: 1,000 samples from this file
   💾 Checkpoint saved: processed_files

📂 Processing 2/26: chunk_0001.csv
   ✅ File complete: 1,000 samples from this file
   💾 Checkpoint saved: processed_files

📂 Processing 3/26: chunk_0002.csv
   ✅ File complete: 1,000 samples from this file
   💾 Checkpoint saved: processed_files

📂 Processing 4/26: chunk_0003.csv
   ✅ File complete: 1,000 samples from this file
   💾 Checkpoint saved: processed_files

📂 Processing 5/26: chunk_0004.csv
   ✅ File complete: 1,000 s

IndexError: index 1 is out of bounds for axis 1 with size 1

In [ ]:
# ============================================================================
# PAPER 1: COMPLETE THESIS EXTENSION EXPERIMENT
# CIC-YNU-IoTMal Dataset (Malware CSV) + Thesis Benign (TXT)
#
# This code:
# 1. Reads malware CSV files (count-based, label=1)
# 2. Reads benign TXT files (Handles dual format: sequential or pre-aggregated)
# 3. Combines them into a balanced dataset
# 4. Runs Random Forest, Logistic Regression, SVM
# 5. Produces all evaluation metrics and visualizations
# ============================================================================
#Main code which uses the Benign .txt files .
import os
import pandas as pd
import numpy as np
import time
import joblib
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve
)
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. HELPER FUNCTIONS
# ============================================================================

def get_feature_columns(df):
    """Get all Call_* columns from a dataframe."""
    return [col for col in df.columns if col.startswith('Call_')]

def process_chunk(chunk, feature_cols, arch_filter='x86'):
    """Process a chunk of malware data - extract features only."""
    if 'Arch' in chunk.columns:
        chunk = chunk[chunk['Arch'] == arch_filter]

    if len(chunk) == 0:
        return None

    available_feature_cols = [col for col in feature_cols if col in chunk.columns]
    if len(available_feature_cols) == 0:
        return None

    X = chunk[available_feature_cols].fillna(0).values
    return X

def accumulate_features(X_accum, y_accum, X_new, y_new):
    """Accumulate features and labels securely."""
    if X_new is None or len(X_new) == 0:
        return X_accum, y_accum

    if X_accum is None:
        return X_new, y_new

    if X_accum.shape[1] != X_new.shape[1]:
        min_dim = min(X_accum.shape[1], X_new.shape[1])
        X_accum = X_accum[:, :min_dim]
        X_new = X_new[:, :min_dim]

    return np.vstack([X_accum, X_new]), np.concatenate([y_accum, y_new])

def parse_benign_txt_to_features(file_path, feature_cols):
    """
    Parse a benign TXT file and convert to count-based features.
    Robustly handles two format types dynamically:
    - Format A: "27 read" (Pre-aggregated frequency summary)
    - Format B: "read" (Sequential single system-call string log)
    """
    feature_dict = {col: 0 for col in feature_cols}

    with open(file_path, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if not parts:
                continue

            # Dynamic Format Engine Identification
            if parts[0].isdigit() and len(parts) >= 2:
                try:
                    # Format A Processing: Parse numerical frequencies
                    count = int(parts[0])
                    syscall = parts[1].strip()
                    call_col = f"Call_{syscall}"
                    if call_col in feature_dict:
                        feature_dict[call_col] += count
                except ValueError:
                    continue
            else:
                # Format B Processing: Step-wise raw sequential incremental counters
                syscall = parts[0].strip()
                call_col = f"Call_{syscall}"
                if call_col in feature_dict:
                    feature_dict[call_col] += 1

    return list(feature_dict.values())

# ============================================================================
# 2. CHECKPOINT FUNCTIONS
# ============================================================================

def save_checkpoint(data, name):
    filepath = os.path.join(CHECKPOINT_DIR, f"{name}.pkl")
    joblib.dump(data, filepath)
    print(f"    💾 Checkpoint saved: {name}")

def load_checkpoint(name):
    filepath = os.path.join(CHECKPOINT_DIR, f"{name}.pkl")
    if os.path.exists(filepath):
        print(f"    🔄 Loading checkpoint: {name}")
        return joblib.load(filepath)
    return None

def checkpoint_exists(name):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, f"{name}.pkl"))

# ============================================================================
# 3. CONFIGURATION
# ============================================================================

RANDOM_STATE = 42
TOP_FEATURES = 30
TEST_SIZE = 0.3
ARCHITECTURE_FILTER = 'x86'
SAMPLE_FRACTION = 0.1  # Reduce if memory errors persist
MAX_ROWS_PER_FILE = 50000

# Directory Configuration Rules
CSV_INPUT_DIR = '/content/drive/My Drive/Colab Notebooks2/csv_output_files'
BENIGN_TXT_DIR = '/content/drive/My Drive/Colab Notebooks2/benign_txt_files'
OUTPUT_DIR = '/content/drive/My Drive/Colab Notebooks2/thesis_results'
CHECKPOINT_DIR = '/content/drive/My Drive/Colab Notebooks2/checkpoints'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("="*70)
print("PAPER 1: CROSS-ARCHITECTURE BEHAVIORAL DRIFT EXPERIMENT")
print("="*70)
print(f"\n📁 Malware CSV directory: {CSV_INPUT_DIR}")
print(f"📁 Benign TXT directory: {BENIGN_TXT_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📁 Checkpoint directory: {CHECKPOINT_DIR}")
print(f"📊 Sample fraction: {SAMPLE_FRACTION*100}%")
print(f"🏗️  Architecture filter: {ARCHITECTURE_FILTER}")

# ============================================================================
# 4. VALIDATE DIRECTORIES
# ============================================================================
print("\n" + "="*70)
print("STEP 0: VALIDATING DIRECTORIES")
print("="*70)

if not os.path.exists(CSV_INPUT_DIR):
    print(f"\n❌ ERROR: Malware CSV directory does not exist!")
    print(f"    Path: {CSV_INPUT_DIR}")
    exit(1)

csv_files = [f for f in os.listdir(CSV_INPUT_DIR) if f.endswith('.csv')]
print(f"\n✅ Found {len(csv_files)} Malware CSV files")

if not os.path.exists(BENIGN_TXT_DIR):
    print(f"\n⚠️ WARNING: Benign TXT directory does not exist!")
    print(f"    Path: {BENIGN_TXT_DIR}")
    print("    Continuing with malware-only data...")
    HAS_BENIGN = False
else:
    txt_files = [f for f in os.listdir(BENIGN_TXT_DIR) if f.endswith('.txt')]
    print(f"\n✅ Found {len(txt_files)} Benign TXT files")
    HAS_BENIGN = len(txt_files) > 0
    if HAS_BENIGN:
        print(f"    First few files loaded: {txt_files[:3]}")

# ============================================================================
# 5. PROCESS FILES WITH CHECKPOINTS (MALWARE CSV + BENIGN TXT)
# ============================================================================
print("\n" + "="*70)
print("STEP 1: LOADING DATASET (STREAMING & MERGING CSV + TXT)")
print("="*70)

# Load checkpoint state if running continuously
processed_files_checkpoint = load_checkpoint('processed_files')
if processed_files_checkpoint is not None:
    processed_files = processed_files_checkpoint['processed_files']
    csv_files = processed_files_checkpoint['csv_files']
    feature_cols = processed_files_checkpoint['feature_cols']
    total_samples = processed_files_checkpoint['total_samples']
    X_accum = processed_files_checkpoint['X_accum']
    y_accum = processed_files_checkpoint['y_accum']
    print(f"\n✅ Resuming from checkpoint. Accumulated {total_samples:,} samples")
else:
    csv_files = [f for f in os.listdir(CSV_INPUT_DIR) if f.endswith('.csv')]
    csv_files.sort()
    print(f"\n📄 Found {len(csv_files)} Malware CSV files")

    X_accum = None
    y_accum = None
    feature_cols = None
    total_samples = 0
    processed_files = 0

# --- 5a: Process Malware CSV Files ---
for i in range(processed_files, len(csv_files)):
    file = csv_files[i]
    file_path = os.path.join(CSV_INPUT_DIR, file)
    file_size = os.path.getsize(file_path) / (1024 * 1024)
    print(f"\n📂 Processing Malware {i+1}/{len(csv_files)}: {file} ({file_size:.2f} MB)")

    try:
        file_samples = 0
        chunk_count = 0

        for chunk in pd.read_csv(file_path, chunksize=MAX_ROWS_PER_FILE, low_memory=False):
            chunk_count += 1

            if feature_cols is None:
                feature_cols = get_feature_columns(chunk)
                print(f"    Found {len(feature_cols)} feature columns")

            if SAMPLE_FRACTION < 1.0:
                chunk = chunk.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_STATE)

            X_chunk = process_chunk(chunk, feature_cols, ARCHITECTURE_FILTER)

            if X_chunk is not None and len(X_chunk) > 0:
                # Force-assign target label to 1 (Malware Dataset Source Verification)
                y_chunk = np.ones(len(X_chunk))
                X_accum, y_accum = accumulate_features(X_accum, y_accum, X_chunk, y_chunk)
                total_samples += len(X_chunk)
                file_samples += len(X_chunk)

            del chunk
            gc.collect()

            if chunk_count % 5 == 0:
                save_checkpoint({
                    'processed_files': i,
                    'csv_files': csv_files,
                    'feature_cols': feature_cols,
                    'total_samples': total_samples,
                    'X_accum': X_accum,
                    'y_accum': y_accum
                }, 'processed_files')
                print(f"    💾 Checkpoint: {total_samples:,} samples processed")

        processed_files = i + 1
        print(f"    ✅ Malware file complete: {file_samples:,} samples")

        save_checkpoint({
            'processed_files': processed_files,
            'csv_files': csv_files,
            'feature_cols': feature_cols,
            'total_samples': total_samples,
            'X_accum': X_accum,
            'y_accum': y_accum
        }, 'processed_files')

    except Exception as e:
        print(f"    ❌ Error processing CSV: {e}")
        continue

# --- 5b: Process and Align Benign TXT Files ---
if HAS_BENIGN and feature_cols is not None:
    txt_files = [f for f in os.listdir(BENIGN_TXT_DIR) if f.endswith('.txt')]
    print(f"\n📄 Found {len(txt_files)} Benign TXT files. Running multi-format alignment mapping...")

    benign_vectors = []
    for txt_file in txt_files:
        txt_path = os.path.join(BENIGN_TXT_DIR, txt_file)
        try:
            feature_vector = parse_benign_txt_to_features(txt_path, feature_cols)
            benign_vectors.append(feature_vector)
            print(f"    ✅ Aligned and Parsed Matrix Row: {txt_file}")
        except Exception as e:
            print(f"    ❌ Error parsing {txt_file}: {e}")

    if len(benign_vectors) > 0:
        X_benign = np.array(benign_vectors)
        y_benign = np.zeros(len(X_benign))  # Explicit assignment to 0 (Benign)

        X_accum, y_accum = accumulate_features(X_accum, y_accum, X_benign, y_benign)
        total_samples += len(X_benign)
        print(f"\n✅ Pipeline Success: Synthesized and merged {len(X_benign)} Benign vectors into data frame!")
    else:
        print(f"\n⚠️ Critical: No benign samples were parsed.")
else:
    print(f"\n⚠️ Ingestion warning: Skipping benign injection step.")

print(f"\n✅ Dataset Ingestion Complete: {total_samples:,} Total profiles logged.")

if total_samples == 0:
    print("❌ Critical dataset footprint error: 0 rows recorded.")
    exit()

# Erase raw file loop tracking checkpoints upon safe collection completion
if os.path.exists(os.path.join(CHECKPOINT_DIR, 'processed_files.pkl')):
    os.remove(os.path.join(CHECKPOINT_DIR, 'processed_files.pkl'))
    print("🗑️  Cleaned out intermediary runtime loop checkpoints.")

# ============================================================================
# 6. PREPARE FINAL DATASET
# ============================================================================
print("\n" + "="*70)
print("STEP 2: PREPARING FINAL DATASET")
print("="*70)

if checkpoint_exists('dataset_prepared'):
    dataset = load_checkpoint('dataset_prepared')
    X_accum = dataset['X']
    y_accum = dataset['y']
    feature_cols = dataset['feature_cols']
    print(f"\n✅ Loaded stabilized dataset from cache: {X_accum.shape[0]:,} records")
else:
    X_accum, y_accum = shuffle(X_accum, y_accum, random_state=RANDOM_STATE)
    save_checkpoint({
        'X': X_accum,
        'y': y_accum,
        'feature_cols': feature_cols
    }, 'dataset_prepared')
    print(f"✅ Dataset balanced, randomized, and saved.")

print(f"\n📊 Consolidated Final Matrix shape: {X_accum.shape[0]:,} instances × {X_accum.shape[1]} variables")
print(f"    Benign Samples:  {(y_accum == 0).sum():,}")
print(f"    Malware Samples: {(y_accum == 1).sum():,}")
print(f"    Malware Ratio:   {((y_accum == 1).sum() / len(y_accum)) * 100:.2f}%")

if len(np.unique(y_accum)) < 2:
    print("\n❌ DATASET IMBALANCE ERROR: Experiment contains only a single target class footprint.")
    print("    Statistical classification cannot resolve model weights without variance.")
    exit(1)

# ============================================================================
# 7. TRAIN-TEST SPLIT WITH STRATIFICATION
# ============================================================================
print("\n" + "="*70)
print("STEP 3: TRAIN-TEST SPLIT")
print("="*70)

if checkpoint_exists('train_test_split'):
    split_data = load_checkpoint('train_test_split')
    X_train = split_data['X_train']
    X_test = split_data['X_test']
    y_train = split_data['y_train']
    y_test = split_data['y_test']
    print(f"\n✅ Train/Test validation matrices loaded from checkpoint.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_accum, y_accum,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_accum
    )
    save_checkpoint({
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }, 'train_test_split')
    print(f"✅ Stratified split complete.")

del X_accum, y_accum
gc.collect()

print(f"\n📊 Split Summary:")
print(f"    Train Subsample Space: {len(X_train):,} rows")
print(f"    Test Evaluation Space:  {len(X_test):,} rows")
print(f"    Train Benign Base:      {(y_train == 0).sum():,}")
print(f"    Train Malware Base:     {(y_train == 1).sum():,}")

# ============================================================================
# 8. FEATURE SELECTION (with class weighting)
# ============================================================================
print("\n" + "="*70)
print("STEP 4: FEATURE SELECTION")
print("="*70)

if checkpoint_exists('feature_selection'):
    fs_data = load_checkpoint('feature_selection')
    selected_features = fs_data['selected_features']
    selected_indices = fs_data['selected_indices']
    feature_importance_df = fs_data['feature_importance_df']
    print(f"\n✅ Stable features loaded from cache.")
else:
    print(f"    Extracting top {TOP_FEATURES} high-variance features...")

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    importance_matrix = np.zeros(X_train.shape[1])

    # Dynamic weight calculation to prevent model bias due to uneven text sample sizes
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

    rf_selector = RandomForestClassifier(
        n_estimators=100,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    for fold_idx, (train_idx, _) in enumerate(cv.split(X_train, y_train)):
        X_fold = X_train[train_idx]
        y_fold = y_train[train_idx]
        rf_selector.fit(X_fold, y_fold)
        importance_matrix += rf_selector.feature_importances_
        print(f"    Fold {fold_idx+1}/3 Evaluated")

    importance_matrix /= cv.get_n_splits()

    feature_importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importance_matrix
    }).sort_values('importance', ascending=False)

    selected_features = feature_importance_df.head(TOP_FEATURES)['feature'].tolist()
    selected_indices = [feature_cols.index(f) for f in selected_features]

    save_checkpoint({
        'selected_features': selected_features,
        'selected_indices': selected_indices,
        'feature_importance_df': feature_importance_df
    }, 'feature_selection')
    print(f"✅ Gini ranking calculation optimized and saved.")

print(f"\n✅ Selected Top 10 Core Explanatory Syscalls:")
for i, f in enumerate(selected_features[:10]):
    imp = feature_importance_df[feature_importance_df['feature'] == f]['importance'].values[0]
    print(f"    {i+1:2d}. {f.replace('Call_', ''):20s} Entropy Loss Reduction: {imp:.4f}")

X_train_selected = X_train[:, selected_indices]
X_test_selected = X_test[:, selected_indices]

# ============================================================================
# 9. SCALING
# ============================================================================
print("\n" + "="*70)
print("STEP 5: FEATURE SCALING")
print("="*70)

if checkpoint_exists('scaling'):
    scaling_data = load_checkpoint('scaling')
    scaler = scaling_data['scaler']
    X_train_scaled = scaling_data['X_train_scaled']
    X_test_scaled = scaling_data['X_test_scaled']
    print(f"✅ Loaded StandardScaler from checkpoint.")
else:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_selected)
    X_test_scaled = scaler.transform(X_test_selected)

    save_checkpoint({
        'scaler': scaler,
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled
    }, 'scaling')
    print(f"✅ Mean Variance Normalization complete.")

del X_train_selected, X_test_selected
gc.collect()

# ============================================================================
# 10. MODEL TRAINING WITH CLASS WEIGHTS
# ============================================================================
print("\n" + "="*70)
print("STEP 6: MODEL TRAINING AND EVALUATION")
print("="*70)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"\n📊 Penalty Balancing Matrix: Benign Target Scale={class_weights[0]:.2f}, Malware Target Scale={class_weights[1]:.2f}")

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "SVM": SVC(
        kernel='linear',
        class_weight=class_weight_dict,
        probability=True,
        random_state=RANDOM_STATE
    )
}

results = {}
overhead_results = {}
best_model = None
best_model_name = None
best_accuracy = 0

for name, model in models.items():
    model_checkpoint = load_checkpoint(f'model_{name}')
    if model_checkpoint is not None:
        print(f"\n{'='*50}")
        print(f"📊 Model evaluation loaded: {name}")
        print(f"{'='*50}")

        model = model_checkpoint['model']
        y_pred = model_checkpoint['y_pred']
        y_proba = model_checkpoint.get('y_proba')
        training_time = model_checkpoint['training_time']
        prediction_time = model_checkpoint['prediction_time']
    else:
        print(f"\n{'='*50}")
        print(f"📊 Modeling Architecture execution: {name}")
        print(f"{'='*50}")

        train_start = time.time()
        model.fit(X_train_scaled, y_train)
        training_time = time.time() - train_start

        predict_start = time.time()
        y_pred = model.predict(X_test_scaled)
        prediction_time = time.time() - predict_start

        if hasattr(model, 'predict_proba'):
            try:
                y_proba = model.predict_proba(X_test_scaled)[:, 1]
            except IndexError:
                y_proba = None
        else:
            y_proba = None

        save_checkpoint({
            'model': model,
            'y_pred': y_pred,
            'y_proba': y_proba,
            'training_time': training_time,
            'prediction_time': prediction_time
        }, f'model_{name}')

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    try:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=3)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
    except:
        cv_mean = 0
        cv_std = 0

    print(f"    Accuracy Score:         {acc:.4f}")
    print(f"    Precision (weighted):   {prec:.4f}")
    print(f"    Recall (weighted):      {rec:.4f}")
    print(f"    F1-Score (weighted):    {f1:.4f}")
    print(f"    CV Cross-Fold Accuracy: {cv_mean:.4f} (±{cv_std:.4f})")
    print(f"    Compute Pipeline Delay: {training_time:.4f}s")
    print(f"    Inference Step Latency: {prediction_time:.4f}s")

    print(f"\n    Classification Profile:")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malware'], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    print(f"\n    Confusion Matrix Layout:")
    print(f"    [[True Benign: {cm[0,0]:,} / False Malware: {cm[0,1]:,}]")
    print(f"     [False Benign: {cm[1,0]:,} / True Malware: {cm[1,1]:,}]]")

    results[name] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'cv_mean': cv_mean,
        'cv_std': cv_std,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    overhead_results[name] = {
        'training_time_sec': training_time,
        'prediction_time_sec': prediction_time
    }

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

print(f"\n{'='*50}")
print(f"🏆 EVALUATION WINNER: {best_model_name} ({best_accuracy*100:.2f}% Accuracy)")
print(f"{'='*50}")

# ============================================================================
# 11. SAVE RESULTS
# ============================================================================
print("\n" + "="*70)
print("STEP 7: SAVING FINAL RESULTS")
print("="*70)

feature_importance_df.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'), index=False)
pd.DataFrame({'selected_feature': selected_features}).to_csv(os.path.join(OUTPUT_DIR, 'selected_features.csv'), index=False)
pd.DataFrame(overhead_results).T.to_csv(os.path.join(OUTPUT_DIR, 'overhead_results.csv'))

joblib.dump(best_model, os.path.join(OUTPUT_DIR, 'best_model.pkl'))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.pkl'))

results_summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'CV_Accuracy': [results[m]['cv_mean'] for m in results]
})
results_summary.to_csv(os.path.join(OUTPUT_DIR, 'results_summary.csv'), index=False)
print(f"✅ Metrics, models, and arrays exported successfully to {OUTPUT_DIR}")

# ============================================================================
# 12. VISUALIZATIONS
# ============================================================================
print("\n" + "="*70)
print("STEP 8: GENERATING VISUALIZATIONS")
print("="*70)

# Feature Importance Plot
plt.figure(figsize=(10, 8))
top15 = feature_importance_df.head(15)
plt.barh(top15['feature'].str.replace('Call_', '')[::-1], top15['importance'][::-1])
plt.title(f'Top 15 Most Important Features ({ARCHITECTURE_FILTER} Architecture)')
plt.xlabel('Gini Importance Index')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=300, bbox_inches='tight')
plt.close()

# ROC Curve Generation
if results[best_model_name]['y_proba'] is not None:
    y_proba = results[best_model_name]['y_proba']
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}', linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {best_model_name} ({ARCHITECTURE_FILTER})')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()

# Precision-Recall Curve Generation
if results[best_model_name]['y_proba'] is not None:
    precision, recall, _ = precision_recall_curve(y_test, y_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, linewidth=2, color='purple')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve - {best_model_name} ({ARCHITECTURE_FILTER})')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'precision_recall_curve.png'), dpi=300, bbox_inches='tight')
    plt.close()

# Confusion Matrix Graphical Plot
cm = results[best_model_name]['confusion_matrix']
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malware'],
            yticklabels=['Benign', 'Malware'])
plt.title(f'Confusion Matrix - {best_model_name} ({ARCHITECTURE_FILTER})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.close()
print("✅ Figures and graphs exported cleanly.")

# ============================================================================
# 13. FINAL SUMMARY REPORT
# ============================================================================
print("\n" + "="*70)
print("📊 FINAL BENCHMARK SUMMARY")
print("="*70)
print("\n" + results_summary.to_string(index=False))
print("\n" + "="*70)
print("🎉 THESIS REPLICATION MATRIX GENERATION COMPLETE!")
print("="*70)

PAPER 1: CROSS-ARCHITECTURE BEHAVIORAL DRIFT EXPERIMENT

📁 Malware CSV directory: /content/drive/My Drive/Colab Notebooks2/csv_output_files
📁 Benign TXT directory: /content/drive/My Drive/Colab Notebooks2/benign_txt_files
📁 Output directory: /content/drive/My Drive/Colab Notebooks2/thesis_results
📁 Checkpoint directory: /content/drive/My Drive/Colab Notebooks2/checkpoints
📊 Sample fraction: 10.0%
🏗️  Architecture filter: x86

STEP 0: VALIDATING DIRECTORIES

✅ Found 26 Malware CSV files

✅ Found 14 Benign TXT files
    First few files loaded: ['benign4.txt', 'benign6.txt', 'benign13.txt']

STEP 1: LOADING DATASET (STREAMING & MERGING CSV + TXT)

📄 Found 26 Malware CSV files

📂 Processing Malware 1/26: chunk_0000.csv (4.51 MB)
    Found 132 feature columns
    ✅ Malware file complete: 1,000 samples
    💾 Checkpoint saved: processed_files

📂 Processing Malware 2/26: chunk_0001.csv (4.51 MB)
    ✅ Malware file complete: 1,000 samples
    💾 Checkpoint saved: processed_files

📂 Processing Ma

In [ ]:
# ============================================================================
# TEST YOUR SAVED MODEL ON A SINGLE STRACE LOG
# ============================================================================
##old and not correct code with respect to ..txt file sample , not using it
import os
import pandas as pd
import numpy as np
import joblib
import re
from collections import Counter

# ============================================================================
# 1. LOAD YOUR SAVED MODEL AND SCALER
# ============================================================================

OUTPUT_DIR = '/content/drive/My Drive/Colab Notebooks2/thesis_results'

print("="*70)
print("LOADING SAVED MODEL AND SCALER")
print("="*70)

# Load the best model (RandomForest from your experiment)
model = joblib.load(os.path.join(OUTPUT_DIR, 'best_model.pkl'))
print(f"✅ Model loaded: {type(model).__name__}")

# Load the scaler
scaler = joblib.load(os.path.join(OUTPUT_DIR, 'scaler.pkl'))
print(f"✅ Scaler loaded")

# Load selected features (top 30 from your experiment)
selected_features_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'selected_features.csv'))
selected_features = selected_features_df['selected_feature'].tolist()
print(f"✅ Selected features: {len(selected_features)} features")

# Load feature importance (for reference)
feature_importance = pd.read_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'))
print(f"✅ Feature importance loaded")

print(f"\n📊 Model ready for testing!")

# ============================================================================
# 2. FUNCTION: PARSE THE TESTING SAMPLE
# ============================================================================

def parse_testing_sample(file_path):
    """
    Parse the testing sample (thesis format: "count syscall").
    """
    print(f"\n📂 Parsing: {file_path}")

    syscall_counts = {}

    with open(file_path, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Format: "10 rt_sigprocmask"
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                count = int(parts[0])
                # Handle multi-word syscalls (e.g., "rt_sigprocmask")
                syscall_name = ' '.join(parts[1:])
                syscall_counts[syscall_name] = syscall_counts.get(syscall_name, 0) + count

    return syscall_counts

def convert_to_feature_vector(syscall_counts, selected_features):
    """
    Convert syscall counts to a feature vector matching the training format.
    """
    # Create feature vector (all zeros)
    feature_vector = np.zeros(len(selected_features))

    for i, col in enumerate(selected_features):
        # Extract syscall name from 'Call_syscallname'
        syscall_name = col.replace('Call_', '')

        # Try exact match first
        if syscall_name in syscall_counts:
            feature_vector[i] = syscall_counts[syscall_name]
        else:
            # Try lowercase/uppercase variations
            lower_name = syscall_name.lower()
            if lower_name in syscall_counts:
                feature_vector[i] = syscall_counts[lower_name]
            else:
                # Try to find by matching the last part
                for key in syscall_counts:
                    if key.lower() == lower_name:
                        feature_vector[i] = syscall_counts[key]
                        break

    return feature_vector

# ============================================================================
# 3. ANALYZE YOUR TESTING SAMPLE
# ============================================================================

print("\n" + "="*70)
print("🔬 ANALYZING TESTING SAMPLE")
print("="*70)

# Path to your testing sample
TEST_FILE = '/content/drive/My Drive/Colab Notebooks2/testing_sample.txt'  # Update this path

# Parse the file
syscall_counts = parse_testing_sample(TEST_FILE)

print(f"\n📊 Extracted {len(syscall_counts)} unique syscalls")
print("\n   Syscall counts:")
for name, count in sorted(syscall_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"   {name:20s}: {count:3d}")

# Convert to feature vector
feature_vector = convert_to_feature_vector(syscall_counts, selected_features)

# Show which features were found
print("\n   Features found in the top 30:")
found_features = []
for i, col in enumerate(selected_features):
    if feature_vector[i] > 0:
        found_features.append((col.replace('Call_', ''), feature_vector[i]))
        print(f"   {col.replace('Call_', ''):20s}: {feature_vector[i]:.0f}")

if not found_features:
    print("   ⚠️ No features from the top 30 found in this sample!")

# Scale the features
feature_vector_scaled = scaler.transform(feature_vector.reshape(1, -1))

# Predict
prediction = model.predict(feature_vector_scaled)[0]

# Get probability
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(feature_vector_scaled)[0]
    malware_prob = proba[1] if len(proba) > 1 else 0
else:
    malware_prob = None

# ============================================================================
# 4. RESULTS
# ============================================================================

print("\n" + "="*70)
print("📋 PREDICTION RESULT")
print("="*70)

result_str = "🔴 MALWARE" if prediction == 1 else "🟢 BENIGN"
print(f"\n   {result_str}")

if malware_prob is not None:
    print(f"   Confidence: {malware_prob*100:.2f}% malware probability")
    print(f"   Confidence: {(1 - malware_prob)*100:.2f}% benign probability")

# Show detailed feature contribution
print("\n" + "="*70)
print("📊 FEATURE CONTRIBUTION ANALYSIS")
print("="*70)

# Get feature importance for the top 15 features
top_features = feature_importance.head(15)

print("\n   Feature Profile (Top 15 important features):")
print(f"   {'Syscall':<20s} {'Importance':<12s} {'Value':<10s} {'Status'}")
print("   " + "-"*55)

for _, row in top_features.iterrows():
    feature = row['feature']
    importance = row['importance']

    # Find the index of this feature
    try:
        idx = selected_features.index(feature)
        value = feature_vector[idx]

        # Determine if this is a malware-like value
        # (High values in these features may indicate malware)
        status = "⚠️ MALWARE" if value > 0 else "✓ NORMAL"

        print(f"   {feature.replace('Call_', ''):<20s} {importance:.4f}     {value:<10.0f} {status}")
    except ValueError:
        continue

# ============================================================================
# 5. INTERPRETATION
# ============================================================================

print("\n" + "="*70)
print("🔍 INTERPRETATION")
print("="*70)

# Check for malware indicators
malware_indicators = {
    'socket': 'Network socket creation - common in malware C2',
    'connect': 'Outgoing connection - potential C2 communication',
    'inet_addr': 'IP address resolution - often used for C2',
    'htons': 'Network byte order - used in network protocols',
    'fork': 'Process creation - common in malware spreading',
    'execve': 'Program execution - malware loading payloads',
    'setsid': 'Create new session - daemonization/stealth',
    'chdir': 'Change directory - often used to hide',
    'prctl': 'Process control - used for stealth'
}

print("\n   Malware indicators found in this sample:")
indicators_found = []
for syscall, count in syscall_counts.items():
    if syscall in malware_indicators:
        indicators_found.append(syscall)
        print(f"   • {syscall}: {count} calls - {malware_indicators[syscall]}")

if not indicators_found:
    print("   No obvious malware indicators found.")

# ============================================================================
# 6. VERDICT
# ============================================================================

print("\n" + "="*70)
print("⚖️ FINAL VERDICT")
print("="*70)

if prediction == 1:
    print("\n   🔴 The sample is classified as MALWARE")
    print("\n   Key reasons:")
    if 'socket' in syscall_counts:
        print("   • Network socket creation detected (C2 communication)")
    if 'connect' in syscall_counts:
        print("   • Outbound network connection detected")
    if 'fork' in syscall_counts or 'execve' in syscall_counts:
        print("   • Process manipulation detected")
    if 'setsid' in syscall_counts:
        print("   • Daemonization detected (attempts to hide)")
    if malware_prob is not None:
        print(f"   • Confidence: {malware_prob*100:.2f}%")
else:
    print("\n   🟢 The sample is classified as BENIGN")
    print("\n   Key reasons:")
    print("   • No suspicious network activity detected")
    print("   • Normal system call patterns")
    if malware_prob is not None:
        print(f"   • Confidence: {(1-malware_prob)*100:.2f}%")

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE")
print("="*70)

In [ ]:
# ============================================================================
# FIXED: TEST YOUR SAVED MODEL ON A SINGLE STRACE LOG
# This version handles BOTH count and sequence formats updated
# ============================================================================
##i am using this code for testing and b
import os
import pandas as pd
import numpy as np
import joblib
import re
from collections import Counter

# ============================================================================
# 1. LOAD YOUR SAVED MODEL AND SCALER
# ============================================================================

OUTPUT_DIR = '/content/drive/My Drive/Colab Notebooks2/thesis_results'

print("="*70)
print("LOADING SAVED MODEL AND SCALER")
print("="*70)

# Load the best model
model = joblib.load(os.path.join(OUTPUT_DIR, 'best_model.pkl'))
print(f"✅ Model loaded: {type(model).__name__}")

# Load the scaler
scaler = joblib.load(os.path.join(OUTPUT_DIR, 'scaler.pkl'))
print(f"✅ Scaler loaded")

# Load selected features
selected_features_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'selected_features.csv'))
selected_features = selected_features_df['selected_feature'].tolist()
print(f"✅ Selected features: {len(selected_features)} features")

# Load feature importance
feature_importance = pd.read_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'))
print(f"✅ Feature importance loaded")

print(f"\n📊 Model ready for testing!")

# ============================================================================
# 2. IMPROVED PARSER (HANDLES BOTH FORMATS)
# ============================================================================

def parse_testing_sample(file_path):
    """
    Parse testing sample in ANY format:
    - Format A: "27 read" (count format)
    - Format B: "read" (sequence format)
    - Format C: Raw strace output
    """
    print(f"\n📂 Parsing: {file_path}")

    syscall_counts = {}

    with open(file_path, 'r', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # --- Format A: "27 read" ---
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                count = int(parts[0])
                syscall_name = ' '.join(parts[1:])
                # Skip non-syscall entries
                if syscall_name not in ['WIFEXITED', 'WEXITSTATUS']:
                    syscall_counts[syscall_name] = syscall_counts.get(syscall_name, 0) + count
                continue

            # --- Format B: "read" (sequence format) ---
            match = re.match(r'^([a-zA-Z_][a-zA-Z0-9_]*)', line)
            if match:
                syscall_name = match.group(1)
                # Skip non-syscall lines
                if syscall_name not in ['+++', '---', 'exit', 'Process', 'WIFEXITED', 'WEXITSTATUS']:
                    syscall_counts[syscall_name] = syscall_counts.get(syscall_name, 0) + 1

    return syscall_counts

def convert_to_feature_vector(syscall_counts, selected_features):
    """
    Convert syscall counts to a feature vector matching the training format.
    """
    feature_vector = np.zeros(len(selected_features))

    for i, col in enumerate(selected_features):
        syscall_name = col.replace('Call_', '')

        # Try exact match
        if syscall_name in syscall_counts:
            feature_vector[i] = syscall_counts[syscall_name]
        else:
            # Try lowercase
            lower_name = syscall_name.lower()
            if lower_name in syscall_counts:
                feature_vector[i] = syscall_counts[lower_name]
            else:
                # Try to find by matching
                for key in syscall_counts:
                    if key.lower() == lower_name:
                        feature_vector[i] = syscall_counts[key]
                        break

    return feature_vector

# ============================================================================
# 3. ANALYZE THE TESTING SAMPLE
# ============================================================================

print("\n" + "="*70)
print("🔬 ANALYZING TESTING SAMPLE")
print("="*70)

# Path to your testing sample
TEST_FILE = '/content/drive/My Drive/Colab Notebooks2/testing_sample.txt'

# Parse the file
syscall_counts = parse_testing_sample(TEST_FILE)

print(f"\n📊 Extracted {len(syscall_counts)} unique syscalls")
print("\n   Syscall counts:")
for name, count in sorted(syscall_counts.items(), key=lambda x: x[1], reverse=True)[:15]:
    print(f"   {name:20s}: {count:3d}")

# Convert to feature vector
feature_vector = convert_to_feature_vector(syscall_counts, selected_features)

# Show which features were found
print("\n   Features found in the top 30:")
for i, col in enumerate(selected_features):
    if feature_vector[i] > 0:
        print(f"   {col.replace('Call_', ''):20s}: {feature_vector[i]:.0f}")

# Scale the features
feature_vector_scaled = scaler.transform(feature_vector.reshape(1, -1))

# Predict
prediction = model.predict(feature_vector_scaled)[0]

# Get probability
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(feature_vector_scaled)[0]
    malware_prob = proba[1] if len(proba) > 1 else 0
else:
    malware_prob = None

# ============================================================================
# 4. RESULTS
# ============================================================================

print("\n" + "="*70)
print("📋 PREDICTION RESULT")
print("="*70)

result_str = "🔴 MALWARE" if prediction == 1 else "🟢 BENIGN"
print(f"\n   {result_str}")

if malware_prob is not None:
    print(f"   Confidence: {malware_prob*100:.2f}% malware probability")
    print(f"   Confidence: {(1 - malware_prob)*100:.2f}% benign probability")

# ============================================================================
# 5. COMPARE WITH EXPECTED LABEL
# ============================================================================

print("\n" + "="*70)
print("📊 VALIDATION ANALYSIS")
print("="*70)

# Manually check if this file is in the benign folder
file_name = os.path.basename(TEST_FILE)
expected_label = "BENIGN" if file_name.startswith('benign') else "UNKNOWN"

print(f"\n   File name: {file_name}")
print(f"   Expected label: {expected_label}")
print(f"   Model prediction: {result_str}")

if expected_label != "UNKNOWN":
    if expected_label == result_str:
        print("\n   ✅ Correct prediction!")
    else:
        print("\n   ❌ INCORRECT prediction! The model misclassified this sample.")

# ============================================================================
# 6. FEATURE CONTRIBUTION ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("📊 FEATURE CONTRIBUTION ANALYSIS")
print("="*70)

top_features = feature_importance.head(15)

print("\n   Feature Profile (Top 15 important features):")
print(f"   {'Syscall':<20s} {'Importance':<12s} {'Value':<10s} {'Status'}")
print("   " + "-"*55)

for _, row in top_features.iterrows():
    feature = row['feature']
    importance = row['importance']

    try:
        idx = selected_features.index(feature)
        value = feature_vector[idx]
        status = "⚠️ MALWARE" if value > 0 else "✓ NORMAL"
        print(f"   {feature.replace('Call_', ''):<20s} {importance:.4f}     {value:<10.0f} {status}")
    except ValueError:
        continue

# ============================================================================
# 7. INTERPRETATION
# ============================================================================

print("\n" + "="*70)
print("🔍 INTERPRETATION")
print("="*70)

# Check for malware indicators
malware_indicators = {
    'socket': 'Network socket creation',
    'connect': 'Outgoing connection',
    'inet_addr': 'IP address resolution',
    'htons': 'Network byte order',
    'fork': 'Process creation',
    'execve': 'Program execution',
    'setsid': 'Daemonization/stealth',
    'chdir': 'Change directory',
    'prctl': 'Process control'
}

print("\n   Malware indicators found in this sample:")
indicators_found = []
for syscall, count in syscall_counts.items():
    if syscall in malware_indicators:
        indicators_found.append(syscall)
        print(f"   • {syscall}: {count} calls - {malware_indicators[syscall]}")

if not indicators_found:
    print("   No obvious malware indicators found.")

# ============================================================================
# 8. FINAL VERDICT
# ============================================================================

print("\n" + "="*70)
print("⚖️ FINAL VERDICT")
print("="*70)

if prediction == 1:
    print("\n   🔴 The sample is classified as MALWARE")
    if expected_label != "UNKNOWN" and expected_label == "BENIGN":
        print("\n   ⚠️ WARNING: This is a FALSE POSITIVE!")
        print("   The model incorrectly classified a benign sample as malware.")
        print("\n   Possible reasons:")
        print("   1. The model was trained on different benign patterns")
        print("   2. The feature extraction is incomplete")
        print("   3. Class imbalance caused bias")
    else:
        print("\n   Key reasons:")
        if 'socket' in syscall_counts:
            print("   • Network socket creation detected")
        if 'connect' in syscall_counts:
            print("   • Outbound network connection detected")
        if malware_prob is not None:
            print(f"   • Confidence: {malware_prob*100:.2f}%")
else:
    print("\n   🟢 The sample is classified as BENIGN")
    if expected_label != "UNKNOWN" and expected_label == "MALWARE":
        print("\n   ⚠️ WARNING: This is a FALSE NEGATIVE!")
        print("   The model incorrectly classified malware as benign.")
    else:
        if malware_prob is not None:
            print(f"   • Confidence: {(1-malware_prob)*100:.2f}%")

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE")
print("="*70)

LOADING SAVED MODEL AND SCALER
✅ Model loaded: RandomForestClassifier
✅ Scaler loaded
✅ Selected features: 30 features
✅ Feature importance loaded

📊 Model ready for testing!

🔬 ANALYZING TESTING SAMPLE

📂 Parsing: /content/drive/My Drive/Colab Notebooks2/testing_sample.txt

📊 Extracted 32 unique syscalls

   Syscall counts:
   mmap                : 172
   mprotect            :  56
   close               :  47
   openat              :  43
   read                :  42
   newfstatat          :  40
   futex               :  38
   rt_sigprocmask      :   8
   brk                 :   7
   getpid              :   6
   rt_sigaction        :   5
   write               :   5
   pread64             :   4
   munmap              :   3
   socket              :   3

   Features found in the top 30:
   getrandom           : 2
   access              : 1
   fcntl               : 2
   prlimit64           : 1
   pread64             : 4
   mmap                : 172
   brk                 : 7
   mprotect  

In [ ]:
import pandas as pd
import os

# Define the directory where your CSV files are saved
csv_output_dir = '/content/drive/My Drive/Colab Notebooks2/csv_output_files'

# Get a list of all CSV files in the directory
all_csv_files = [f for f in os.listdir(csv_output_dir) if f.endswith('.csv')]

if all_csv_files:
    # Pick the first CSV file for inspection
    sample_csv_file = os.path.join(csv_output_dir, all_csv_files[0])
    print(f"Inspecting sample CSV file: {sample_csv_file}")

    try:
        # Load the sample CSV file into a Pandas DataFrame
        df_sample = pd.read_csv(sample_csv_file)

        # Display the first 5 rows to see the column structure
        print("\nFirst 5 rows of the sample CSV:")
        display(df_sample.head())

        # Check for the 'MalwareFamily' column
        if 'MalwareFamily' in df_sample.columns:
            print("\n'MalwareFamily' column found!")
            print("This column indicates the classification of the sample.")
            print("Unique values in 'MalwareFamily' (first 10):", df_sample['MalwareFamily'].unique()[:10])
            # If 'Benign' is a value, it means the sample is benign, otherwise it's malicious (specific malware family name)
        else:
            print("\n'MalwareFamily' column NOT found. Looking for other potential label columns...")
            # Try to infer other label columns
            potential_label_cols = [col for col in df_sample.columns if 'label' in col.lower()]
            if potential_label_cols:
                print(f"Found potential label columns: {potential_label_cols}")
                for col in potential_label_cols:
                    print(f"Unique values in '{col}' (first 10):", df_sample[col].unique()[:10])
            else:
                print("No common 'label' or 'MalwareFamily' columns found. You might need to inspect the data manually.")

    except Exception as e:
        print(f"An error occurred while reading the sample CSV: {e}")
else:
    print(f"No CSV files found in the directory: {csv_output_dir}")
    print("Please ensure the CSV conversion step was completed successfully.")

In [ ]:
import os

# The path that caused the error
csv_input_dir = '/content/drive/My Drive/Colab Notebooks2/csv_output_files'

print(f"Checking directory: {csv_input_dir}")

if os.path.exists(csv_input_dir):
    print(f"✅ Directory exists: {csv_input_dir}")

    # List contents to see if any CSV files are present
    csv_files_found = [f for f in os.listdir(csv_input_dir) if f.endswith('.csv')]
    if csv_files_found:
        print(f"✅ Found {len(csv_files_found)} CSV files in the directory. First 5: {csv_files_found[:5]}")
    else:
        print(f"⚠️ Directory '{csv_input_dir}' exists, but no CSV files were found inside it.")
        print("This suggests the previous step to generate CSVs might not have completed successfully.")
else:
    print(f"❌ Directory does NOT exist: {csv_input_dir}")
    print("You need to ensure the previous step (converting Parquet chunks to CSV) is run successfully to create this directory and its contents.")

print("\n--- Troubleshooting Tips ---")
print("1.  **Re-run the CSV generation cell (cell `36d3e05b`):** Make sure it completes without errors.")
print("2.  **Check your Google Drive:** Manually verify that the path 'My Drive/Colab Notebooks2/csv_output_files' exists and contains CSV files.")
print("3.  **Confirm correct mounting:** Though indicated as mounted, occasionally there can be issues. Try to remount if necessary.")

Checking directory: /content/drive/My Drive/Colab Notebooks2/csv_output_files
✅ Directory exists: /content/drive/My Drive/Colab Notebooks2/csv_output_files
✅ Found 26 CSV files in the directory. First 5: ['chunk_0001.csv', 'chunk_0002.csv', 'chunk_0000.csv', 'chunk_0003.csv', 'chunk_0004.csv']

--- Troubleshooting Tips ---
1.  **Re-run the CSV generation cell (cell `36d3e05b`):** Make sure it completes without errors.
2.  **Check your Google Drive:** Manually verify that the path 'My Drive/Colab Notebooks2/csv_output_files' exists and contains CSV files.
3.  **Confirm correct mounting:** Though indicated as mounted, occasionally there can be issues. Try to remount if necessary.


In [ ]:
"""
=============================================================================
CLASSIFIER-FAMILY ROBUSTNESS CHECK
=============================================================================

Question: is the ARM transfer gap a property of the DATA, or of how Random
Forest thresholds absolute syscall counts?

Three classifiers with deliberately different inductive biases:

  Random Forest        axis-aligned splits on absolute counts (your baseline)
  Logistic Regression  linear decision boundary, completely different bias.
                       If ARM still fails here, the effect is in the data.
  XGBoost              tree-based but a different fitting procedure
                       (boosting rather than bagging)

Everything else is held constant: the same 87 common features, the same
hash-grouped same-architecture splits, the same full-source/full-target
transfer protocol, raw counts only, seed 42, macro F1.

NOTE ON SCALING: Logistic Regression needs standardised inputs. The scaler
is fitted on the SOURCE training data only and applied unchanged to the
target, which is the correct protocol for a held-out-target evaluation.
Random Forest and XGBoost are scale-invariant and receive raw counts.

Expected runtime: 30-60 minutes on Colab.
=============================================================================
"""

import sys
import subprocess
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except ImportError:
    print("installing xgboost...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'

ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)


# ---------------------------------------------------------------------------
# MODELS
# ---------------------------------------------------------------------------

def make_model(name, y_train):
    """Build a fresh classifier. Imbalance handled the same way throughout."""
    if name == 'RandomForest':
        return RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1)

    if name == 'LogisticRegression':
        # scaler + LR as one pipeline, so the scaler is fitted on train only
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight='balanced', max_iter=2000,
                               random_state=RANDOM_STATE, n_jobs=-1))

    if name == 'XGBoost':
        # scale_pos_weight is XGBoost's equivalent of class_weight='balanced'
        n_neg = int((y_train == 0).sum())
        n_pos = int((y_train == 1).sum())
        spw = n_neg / max(n_pos, 1)
        return xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            scale_pos_weight=spw, random_state=RANDOM_STATE,
            n_jobs=-1, tree_method='hist', eval_metric='logloss',
            verbosity=0)

    raise ValueError(name)


MODELS = ['RandomForest', 'LogisticRegression', 'XGBoost']


# ---------------------------------------------------------------------------
# DATA
# ---------------------------------------------------------------------------

def load_all():
    data = {}
    for arch, path in ARCH_FILES.items():
        df = pd.read_parquet(path)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        data[arch] = df
    return data


def common_features(data):
    sets = [set(c for c in df.columns if c.startswith('Call_')) for df in data.values()]
    return sorted(set.intersection(*sets))


# ---------------------------------------------------------------------------
# EVALUATION
# ---------------------------------------------------------------------------

def same_arch(df, feats, model_name):
    """Hash-grouped held-out split within one architecture."""
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    tr_i, te_i = next(gss.split(df[feats].values, df['label'].values,
                                groups=df['Hash'].values))
    tr, te = df.iloc[tr_i], df.iloc[te_i]
    clf = make_model(model_name, tr['label'].values)
    clf.fit(tr[feats].values, tr['label'].values)
    pred = clf.predict(te[feats].values)
    return f1_score(te['label'].values, pred, average='macro')


def transfer(src_df, tgt_df, feats, model_name):
    """Train on full source, evaluate on full target. No target data in training."""
    clf = make_model(model_name, src_df['label'].values)
    clf.fit(src_df[feats].values, src_df['label'].values)
    pred = clf.predict(tgt_df[feats].values)
    return f1_score(tgt_df['label'].values, pred, average='macro')


def run_all(data, feats):
    rows = []
    for m in MODELS:
        print(f"\n{'='*70}\n{m}\n{'='*70}")
        for a in ARCHS:
            s = same_arch(data[a], feats, m)
            rows.append({'model': m, 'train': a, 'test': a,
                         'kind': 'same', 'macro_f1': s})
            print(f"  same      {a:7s} -> {a:7s}  {s:.4f}")
        for src in ARCHS:
            for tgt in ARCHS:
                if src == tgt:
                    continue
                s = transfer(data[src], data[tgt], feats, m)
                involves_arm = (src == 'ARM' or tgt == 'ARM')
                rows.append({'model': m, 'train': src, 'test': tgt,
                             'kind': 'transfer', 'macro_f1': s})
                tag = '  <- ARM' if involves_arm else ''
                print(f"  transfer  {src:7s} -> {tgt:7s}  {s:.4f}{tag}")
    return pd.DataFrame(rows)


def summarise(res):
    out = []
    for m in MODELS:
        t = res[(res.model == m) & (res.kind == 'transfer')]
        arm = t[(t.train == 'ARM') | (t.test == 'ARM')]['macro_f1']
        non = t[(t.train != 'ARM') & (t.test != 'ARM')]['macro_f1']
        into = t[(t.test == 'ARM')]['macro_f1']
        out_of = t[(t.train == 'ARM')]['macro_f1']
        same = res[(res.model == m) & (res.kind == 'same')]['macro_f1']
        out.append({
            'Classifier': m,
            'Same-arch mean': round(same.mean(), 4),
            'ARM pairs mean': round(arm.mean(), 4),
            'Non-ARM mean':   round(non.mean(), 4),
            'Gap':            round(non.mean() - arm.mean(), 4),
            'Into ARM mean':  round(into.mean(), 4),
            'Out of ARM mean': round(out_of.mean(), 4),
        })
    return pd.DataFrame(out)


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    print("Loading samples...")
    data = load_all()
    for a, df in data.items():
        print(f"  {a:7s} rows={len(df):7,d}  malware={df['label'].mean()*100:5.2f}%  "
              f"hashes={df['Hash'].nunique():5,d}")

    feats = common_features(data)
    print(f"\nCommon syscall features: {len(feats)}")

    res = run_all(data, feats)

    print(f"\n{'='*70}\nSUMMARY FOR THE PAPER\n{'='*70}")
    summary = summarise(res)
    print(summary.to_string(index=False))

    print("\nHOW TO READ THIS")
    print("  Gap stays large for all three  -> the ARM effect is in the DATA.")
    print("     This is the strong outcome: it is not an artefact of how")
    print("     Random Forest splits on absolute counts.")
    print("  Gap closes for LogisticRegression only -> the effect depends on")
    print("     nonlinear thresholding of counts, which points at trace scale.")
    print("  Gap closes for all -> the effect is specific to your RF setup and")
    print("     the paper's framing needs revising.")
    print("\n  'Into ARM' vs 'Out of ARM' shows whether the asymmetry you found")
    print("  with Random Forest reproduces under other classifiers.")

    res.to_csv(f'{BASE_DIR}/paper3_classifier_full.csv', index=False)
    summary.to_csv(f'{BASE_DIR}/paper3_classifier_summary.csv', index=False)
    print(f"\nSaved:\n  paper3_classifier_full.csv\n  paper3_classifier_summary.csv")

Loading samples...
  x86     rows=199,655  malware=77.44%  hashes=3,815
  ARM     rows=199,903  malware=96.00%  hashes=1,992
  MIPS    rows=199,749  malware=77.95%  hashes=3,997
  MIPSEL  rows=199,693  malware=80.39%  hashes=3,868

Common syscall features: 87

RandomForest
  same      x86     -> x86      0.9815
  same      ARM     -> ARM      0.9475
  same      MIPS    -> MIPS     0.9880
  same      MIPSEL  -> MIPSEL   0.9890
  transfer  x86     -> ARM      0.8443  <- ARM
  transfer  x86     -> MIPS     0.9888
  transfer  x86     -> MIPSEL   0.9887
  transfer  ARM     -> x86      0.7324  <- ARM
  transfer  ARM     -> MIPS     0.7482  <- ARM
  transfer  ARM     -> MIPSEL   0.7779  <- ARM
  transfer  MIPS    -> x86      0.9792
  transfer  MIPS    -> ARM      0.8354  <- ARM
  transfer  MIPS    -> MIPSEL   0.9872
  transfer  MIPSEL  -> x86      0.9806
  transfer  MIPSEL  -> ARM      0.8067  <- ARM
  transfer  MIPSEL  -> MIPS     0.9892

LogisticRegression
  same      x86     -> x86      0.

In [ ]:
# Ensure pyarrow is installed for Parquet file handling
!pip install pyarrow

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score

BASE = '/content/drive/My Drive/Colab Notebooks2'
FILES = {'x86':'paper3_x86_sample.parquet','ARM':'paper3_arm_sample.parquet',
         'MIPS':'paper3_mips_sample.parquet','MIPSEL':'paper3_mipsel_sample.parquet'}

data = {}
for a,f in FILES.items():
    d = pd.read_parquet(f'{BASE}/{f}')
    d['label'] = (d['MalwareFamily'] != 'Benign').astype(int)
    data[a] = d
feats = sorted(set.intersection(*[{c for c in d.columns if c.startswith('Call_')}
                                  for d in data.values()]))

for a, d in data.items():
    X = d[feats].values.astype(float)
    X = X / np.maximum(X.sum(axis=1, keepdims=True), 1)      # relative frequency
    y = d['label'].values
    tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42)
                  .split(X, y, groups=d['Hash'].values))
    clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                 random_state=42, n_jobs=-1).fit(X[tr], y[tr])
    print(f"{a:7s} relative-frequency same-arch macro F1 = "
          f"{f1_score(y[te], clf.predict(X[te]), average='macro'):.4f}")

x86     relative-frequency same-arch macro F1 = 0.9596
ARM     relative-frequency same-arch macro F1 = 0.9504
MIPS    relative-frequency same-arch macro F1 = 0.9470
MIPSEL  relative-frequency same-arch macro F1 = 0.9700


In [ ]:
"""
Settles two open questions before final submission.

Q1  Is the 96% ARM malware share a ROW-level or a HASH-level property?
    If malware binaries produce far more trace rows than benign ones, then
    the class-composition control adjusted a row-weighting effect rather
    than a binary-level imbalance, and Sections 3.2 and 5.5 should say so.

Q2  How many total columns does each Parquet file have?
    Section 2.4 claims the dimensionalities reported in the multimodal
    study (461 / 392 / 392 / 409) exceed the number of syscall-count
    columns available per architecture. If the TOTAL column counts match
    those numbers, that study is counting every column in the STRACE file,
    not only the Call_* ones, and the sentence must be rephrased.
"""

import pandas as pd

BASE = '/content/drive/My Drive/Colab Notebooks2'
FILES = {
    'x86':    f'{BASE}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE}/paper3_mipsel_sample.parquet',
}

print(f"{'arch':8s} {'hashes':>7s} {'mal-hash%':>10s} {'mal-row%':>9s} "
      f"{'rows/mal-hash':>14s} {'rows/ben-hash':>14s} {'all cols':>9s} {'Call_*':>7s}")
print("-" * 86)

for arch, path in FILES.items():
    df = pd.read_parquet(path)
    df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)

    g = df.groupby('Hash')['label'].agg(['min', 'max'])
    if not (g['min'] == g['max']).all():
        print(f"  WARNING [{arch}]: some hashes carry both labels")

    mal_h = int(g['max'].sum())
    ben_h = len(g) - mal_h
    mal_r = int((df.label == 1).sum())
    ben_r = int((df.label == 0).sum())
    call_cols = sum(1 for c in df.columns if c.startswith('Call_'))

    print(f"{arch:8s} {len(g):7,d} {mal_h/len(g)*100:9.1f}% "
          f"{mal_r/len(df)*100:8.1f}% "
          f"{mal_r/max(mal_h,1):14.1f} {ben_r/max(ben_h,1):14.1f} "
          f"{df.shape[1]:9d} {call_cols:7d}")

print()
print("READING THE OUTPUT")
print("  mal-hash% close to mal-row%  -> 96% is a genuine binary-level imbalance;")
print("                                  no change needed to the manuscript.")
print("  mal-hash% well below mal-row% -> the 96% is a row-weighting effect driven")
print("                                  by trace volume. Say so in Sections 3.2")
print("                                  and 5.5, and describe what Control 1")
print("                                  actually adjusted.")
print()
print("  'all cols' near 461/392/392/409 -> the multimodal study counts every")
print("                                  column in the STRACE file. Rephrase the")
print("                                  final sentence of Section 2.4 accordingly.")

arch      hashes  mal-hash%  mal-row%  rows/mal-hash  rows/ben-hash  all cols  Call_*
--------------------------------------------------------------------------------------
x86        3,815      37.4%     77.4%          108.5           18.8       136     132
ARM        1,992      79.1%     96.0%          121.8           19.2       135     131
MIPS       3,997      38.0%     78.0%          102.4           17.8       131     127
MIPSEL     3,868      38.2%     80.4%          108.8           16.4       133     129

READING THE OUTPUT
  mal-hash% close to mal-row%  -> 96% is a genuine binary-level imbalance;
                                  no change needed to the manuscript.
  mal-hash% well below mal-row% -> the 96% is a row-weighting effect driven
                                  by trace volume. Say so in Sections 3.2
                                  and 5.5, and describe what Control 1
                                  actually adjusted.

  'all cols' near 461/392/392/409 -> the mu

In [ ]:
"""
=============================================================================
IS THE ARM GAP A SEPARABILITY LOSS OR A THRESHOLD/PRIOR EFFECT?
=============================================================================

Macro F1 is threshold-dependent and prior-sensitive. On an ARM target the
benign class is ~4% of rows, so F1_benign is computed on very few positives
and moves sharply when the decision threshold is mismatched. A model trained
on x86 (77% malware) carries a threshold suited to that prior; applied to
ARM (96% malware) that threshold is wrong even if the classes remain
perfectly separable.

This script separates the two possibilities.

  PART A  Threshold-independent metrics for all 16 settings:
            ROC-AUC, PR-AUC (benign as positive), balanced accuracy,
            per-class recall, alongside the macro F1 already reported.

  PART B  Two prior-shift corrections, applied to the same predictions:
            (1) Saerens-Latinne-Decaestecker EM, which re-estimates the
                target prior from UNLABELLED target data. Deployable.
            (2) Oracle threshold, chosen with knowledge of the true target
                prior. Not deployable; reported as an upper bound.

HOW TO READ THE RESULT
  AUC stays high (>0.95) while macro F1 falls
      -> separability is preserved; the gap is an operating-point effect.
         The paper's claim should be restated accordingly, and the
         correction in Part B becomes part of the contribution.
  AUC falls with macro F1
      -> separability genuinely degrades. Prior shift is ruled out and the
         present claim is confirmed under a threshold-independent metric.
  Mixed by direction
      -> report per direction; into-ARM and out-of-ARM may differ.

Runtime: 20-40 minutes.
=============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                             balanced_accuracy_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)


# ---------------------------------------------------------------- data

def load_all():
    d = {}
    for a, p in ARCH_FILES.items():
        df = pd.read_parquet(p)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        d[a] = df
    return d


def common_features(data):
    return sorted(set.intersection(
        *[{c for c in df.columns if c.startswith('Call_')} for df in data.values()]))


def rf():
    return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=RANDOM_STATE, n_jobs=-1)


# ------------------------------------------------- prior-shift correction

def saerens_em(p_source, prior_source, max_iter=200, tol=1e-7):
    """
    Saerens-Latinne-Decaestecker EM. Re-estimates the target prior from the
    source model's probabilities on UNLABELLED target data, then rescales
    those probabilities to the new prior. Uses no target labels.
    """
    prior = float(np.clip(p_source.mean(), 1e-6, 1 - 1e-6))
    ps = float(np.clip(prior_source, 1e-6, 1 - 1e-6))
    p = np.clip(p_source, 1e-9, 1 - 1e-9)
    for _ in range(max_iter):
        num = (prior / ps) * p
        den = num + ((1 - prior) / (1 - ps)) * (1 - p)
        p_adj = num / den
        new = float(np.clip(p_adj.mean(), 1e-6, 1 - 1e-6))
        if abs(new - prior) < tol:
            prior = new
            break
        prior = new
    num = (prior / ps) * p
    den = num + ((1 - prior) / (1 - ps)) * (1 - p)
    return prior, num / den


def oracle_threshold(y_true, p):
    """Threshold matching the TRUE target prior. Upper bound only."""
    k = int(round((1 - y_true.mean()) * len(y_true)))      # expected benign count
    if k <= 0:
        return 0.5
    return float(np.sort(p)[max(k - 1, 0)])


# ------------------------------------------------------------- evaluate

def evaluate(y, p, prior_source):
    """All metrics for one source->target setting."""
    pred = (p >= 0.5).astype(int)
    out = {
        'macro_f1':        f1_score(y, pred, average='macro'),
        'roc_auc':         roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        'pr_auc_benign':   average_precision_score(1 - y, 1 - p) if len(np.unique(y)) > 1 else np.nan,
        'balanced_acc':    balanced_accuracy_score(y, pred),
        'recall_malware':  recall_score(y, pred, pos_label=1),
        'recall_benign':   recall_score(y, pred, pos_label=0),
        'target_prior':    y.mean(),
    }
    # correction 1: Saerens EM (no target labels)
    est_prior, p_em = saerens_em(p, prior_source)
    out['em_est_prior'] = est_prior
    out['macro_f1_em'] = f1_score(y, (p_em >= 0.5).astype(int), average='macro')
    # correction 2: oracle threshold (upper bound)
    thr = oracle_threshold(y, p)
    out['macro_f1_oracle'] = f1_score(y, (p >= thr).astype(int), average='macro')
    return out


# ----------------------------------------------------------------- main

if __name__ == '__main__':
    print("Loading...")
    data = load_all()
    feats = common_features(data)
    print(f"Common features: {len(feats)}\n")

    rows = []

    # same-architecture, hash-grouped
    for a in ARCHS:
        df = data[a]
        tr_i, te_i = next(GroupShuffleSplit(1, test_size=0.2, random_state=RANDOM_STATE)
                          .split(df[feats].values, df['label'].values, groups=df['Hash'].values))
        tr, te = df.iloc[tr_i], df.iloc[te_i]
        clf = rf().fit(tr[feats].values, tr['label'].values)
        p = clf.predict_proba(te[feats].values)[:, 1]
        r = evaluate(te['label'].values, p, tr['label'].mean())
        r.update({'train': a, 'test': a, 'kind': 'same'})
        rows.append(r)
        print(f"same      {a:7s} macroF1={r['macro_f1']:.4f}  AUC={r['roc_auc']:.4f}")

    # directed transfers
    for src in ARCHS:
        clf = rf().fit(data[src][feats].values, data[src]['label'].values)
        prior_src = data[src]['label'].mean()
        for tgt in ARCHS:
            if src == tgt:
                continue
            y = data[tgt]['label'].values
            p = clf.predict_proba(data[tgt][feats].values)[:, 1]
            r = evaluate(y, p, prior_src)
            r.update({'train': src, 'test': tgt, 'kind': 'transfer'})
            rows.append(r)
            tag = ' <- ARM' if 'ARM' in (src, tgt) else ''
            print(f"transfer  {src:7s}->{tgt:7s} macroF1={r['macro_f1']:.4f} "
                  f"AUC={r['roc_auc']:.4f} balAcc={r['balanced_acc']:.4f} "
                  f"EM={r['macro_f1_em']:.4f} oracle={r['macro_f1_oracle']:.4f}{tag}")

    res = pd.DataFrame(rows)[[
        'kind', 'train', 'test', 'target_prior', 'macro_f1', 'roc_auc',
        'pr_auc_benign', 'balanced_acc', 'recall_malware', 'recall_benign',
        'em_est_prior', 'macro_f1_em', 'macro_f1_oracle']]

    print("\n" + "=" * 78)
    print("FULL TABLE")
    print("=" * 78)
    print(res.round(4).to_string(index=False))

    t = res[res.kind == 'transfer']
    arm = t[(t.train == 'ARM') | (t.test == 'ARM')]
    non = t[(t.train != 'ARM') & (t.test != 'ARM')]
    into = t[t.test == 'ARM']
    out_of = t[t.train == 'ARM']

    print("\n" + "=" * 78)
    print("SUMMARY: does the gap survive threshold-independent metrics?")
    print("=" * 78)
    summary = pd.DataFrame([
        {'group': 'ARM-involving (6)', **{m: arm[m].mean() for m in
            ['macro_f1', 'roc_auc', 'pr_auc_benign', 'balanced_acc',
             'macro_f1_em', 'macro_f1_oracle']}},
        {'group': 'Non-ARM (6)',       **{m: non[m].mean() for m in
            ['macro_f1', 'roc_auc', 'pr_auc_benign', 'balanced_acc',
             'macro_f1_em', 'macro_f1_oracle']}},
        {'group': 'Into ARM (3)',      **{m: into[m].mean() for m in
            ['macro_f1', 'roc_auc', 'pr_auc_benign', 'balanced_acc',
             'macro_f1_em', 'macro_f1_oracle']}},
        {'group': 'Out of ARM (3)',    **{m: out_of[m].mean() for m in
            ['macro_f1', 'roc_auc', 'pr_auc_benign', 'balanced_acc',
             'macro_f1_em', 'macro_f1_oracle']}},
    ])
    print(summary.round(4).to_string(index=False))

    print("\nGAPS (non-ARM minus ARM-involving):")
    for m in ['macro_f1', 'roc_auc', 'pr_auc_benign', 'balanced_acc',
              'macro_f1_em', 'macro_f1_oracle']:
        print(f"  {m:18s} {non[m].mean() - arm[m].mean():+.4f}")

    print("\nINTERPRETATION")
    print("  roc_auc gap near zero while macro_f1 gap is large")
    print("     -> separability preserved; the ARM gap is an operating-point effect.")
    print("  macro_f1_em recovers most of the gap")
    print("     -> a deployable correction exists, using no target labels.")
    print("  roc_auc gap large as well")
    print("     -> separability genuinely degrades; the present claim is confirmed.")

    res.to_csv(f'{BASE_DIR}/paper3_prior_shift_full.csv', index=False)
    summary.to_csv(f'{BASE_DIR}/paper3_prior_shift_summary.csv', index=False)
    print(f"\nSaved: paper3_prior_shift_full.csv, paper3_prior_shift_summary.csv")

Loading...
Common features: 87

same      x86     macroF1=0.9815  AUC=0.9945
same      ARM     macroF1=0.9475  AUC=0.9962
same      MIPS    macroF1=0.9880  AUC=0.9969
same      MIPSEL  macroF1=0.9890  AUC=0.9970
transfer  x86    ->ARM     macroF1=0.8450 AUC=0.9865 balAcc=0.9772 EM=0.8923 oracle=0.6216 <- ARM
transfer  x86    ->MIPS    macroF1=0.9888 AUC=0.9961 balAcc=0.9899 EM=0.9882 oracle=0.9878
transfer  x86    ->MIPSEL  macroF1=0.9887 AUC=0.9968 balAcc=0.9903 EM=0.9888 oracle=0.9878
transfer  ARM    ->x86     macroF1=0.7350 AUC=0.6407 balAcc=0.6918 EM=0.7409 oracle=0.7162 <- ARM
transfer  ARM    ->MIPS    macroF1=0.7513 AUC=0.6822 balAcc=0.7054 EM=0.7602 oracle=0.7412 <- ARM
transfer  ARM    ->MIPSEL  macroF1=0.7818 AUC=0.7121 balAcc=0.7299 EM=0.7651 oracle=0.7668 <- ARM
transfer  MIPS   ->x86     macroF1=0.9792 AUC=0.9937 balAcc=0.9834 EM=0.9793 oracle=0.9780
transfer  MIPS   ->ARM     macroF1=0.8357 AUC=0.9911 balAcc=0.9760 EM=0.8811 oracle=0.6200 <- ARM
transfer  MIPS   ->MIPSEL

In [ ]:
"""
=============================================================================
DO SEMANTICALLY EQUIVALENT SYSCALLS LAND IN DIFFERENT COLUMNS?
=============================================================================

ARM EABI, MIPS o32, and x86-32 differ in syscall naming for the same
underlying operation: mmap/mmap2, stat/stat64, fcntl/fcntl64, lseek/_llseek,
clock_gettime/clock_gettime64, getuid/getuid32, and others.

An exact-name intersection keeps BOTH variants when both names appear in all
four schemas. But if each architecture populates only one variant, the
feature space is silently misaligned: a model trained where fcntl64 carries
the signal finds that column near-zero on an architecture that populates
fcntl instead.

Table 7 of the manuscript already hints at this. ARM ranks fcntl among its
distinctive features while x86, MIPS, and MIPSEL all rank fcntl64.

  PART A  Diagnostic. For each alias group, report the column totals per
          architecture, and quantify how much of each architecture's syscall
          volume sits in columns that are effectively empty elsewhere.

  PART B  Re-run. Merge each alias group into a single feature, rebuild the
          common space, and repeat the same-architecture baselines and the
          twelve directed transfers. Compare against the current results.

HOW TO READ IT
  Part A shows one variant populated per architecture
      -> the current 87-feature space is misaligned. Part B is the corrected
         measurement, and the manuscript's transfer figures must be revised.
  Part A shows both variants populated everywhere
      -> no misalignment. Report the check as a negative result; it closes
         the objection.
  Part B closes the ARM gap
      -> the gap was substantially an alignment artefact.
  Part B leaves the gap
      -> alignment is ruled out and the finding is strengthened.

NOTE: this dataset is now used by other published work, so whichever way the
result goes it is worth reporting precisely.

Runtime: 30-50 minutes.
=============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, recall_score
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)

# Groups of names denoting the same underlying operation across 32-bit ABIs.
# The first entry is used as the merged feature name.
ALIAS_GROUPS = [
    ['mmap', 'mmap2', 'mmap_pgoff'],
    ['stat', 'stat64', 'newstat'],
    ['fstat', 'fstat64', 'newfstat', 'newfstatat', 'fstatat64'],
    ['lstat', 'lstat64', 'newlstat'],
    ['fcntl', 'fcntl64'],
    ['lseek', '_llseek', 'llseek'],
    ['clock_gettime', 'clock_gettime64'],
    ['clock_getres', 'clock_getres_time64'],
    ['getdents', 'getdents64'],
    ['sendfile', 'sendfile64'],
    ['truncate', 'truncate64'],
    ['ftruncate', 'ftruncate64'],
    ['pread', 'pread64'],
    ['pwrite', 'pwrite64'],
    ['getuid', 'getuid32'], ['geteuid', 'geteuid32'],
    ['getgid', 'getgid32'], ['getegid', 'getegid32'],
    ['setuid', 'setuid32'], ['setgid', 'setgid32'],
    ['getgroups', 'getgroups32'], ['setgroups', 'setgroups32'],
    ['select', '_newselect', 'newselect'],
    ['statfs', 'statfs64'], ['fstatfs', 'fstatfs64'],
    ['fadvise64', 'fadvise64_64'],
    ['futex', 'futex_time64'],
    ['ppoll', 'ppoll_time64'],
    ['pselect6', 'pselect6_time64'],
    ['utimensat', 'utimensat_time64'],
    ['socketcall'],          # multiplexer on x86-32; flagged, not merged
    ['ipc'],                 # multiplexer; flagged, not merged
]


def load_all():
    d = {}
    for a, p in ARCH_FILES.items():
        df = pd.read_parquet(p)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        d[a] = df
    return d


def call_cols(df):
    return [c for c in df.columns if c.startswith('Call_')]


# ============================================================ PART A

def part_a(data):
    print("=" * 78)
    print("PART A - ALIAS DIAGNOSTIC")
    print("=" * 78)
    print("Column totals per architecture. '-' = column absent from schema,")
    print("0 = column present but never populated.\n")

    flagged = []
    for group in ALIAS_GROUPS:
        cols = ['Call_' + g for g in group]
        present_any = any(c in data[a].columns for a in ARCHS for c in cols)
        if not present_any:
            continue
        rows = []
        for g, c in zip(group, cols):
            r = {'variant': g}
            for a in ARCHS:
                r[a] = float(data[a][c].sum()) if c in data[a].columns else None
            rows.append(r)
        t = pd.DataFrame(rows).set_index('variant')

        # which variant does each architecture actually use?
        used = {}
        for a in ARCHS:
            col = t[a].dropna()
            used[a] = col.idxmax() if len(col) and col.max() > 0 else None
        distinct = len({v for v in used.values() if v})
        mark = '   <<< SPLIT ACROSS ARCHITECTURES' if distinct > 1 else ''
        if distinct > 1:
            flagged.append((group[0], used.copy()))

        print(f"[{group[0]}]{mark}")
        disp = t.copy()
        for a in ARCHS:
            disp[a] = disp[a].map(lambda v: '-' if pd.isna(v) else f"{v:,.0f}")
        print(disp.to_string())
        print(f"  dominant variant: " +
              ", ".join(f"{a}={used[a] or 'none'}" for a in ARCHS) + "\n")

    print("-" * 78)
    if flagged:
        print(f"{len(flagged)} alias group(s) are populated differently across architectures:")
        for name, used in flagged:
            print(f"  {name:18s} " + ", ".join(f"{a}={used[a] or 'none'}" for a in ARCHS))
        print("\nThese are genuine misalignments in the exact-name intersection.")
    else:
        print("No alias group is split across architectures.")
        print("The exact-name intersection is not misaligned by naming variants.")
    print()

    # how much volume sits in columns that are empty elsewhere?
    print("DEAD-FEATURE VOLUME")
    print("Share of each architecture's syscall counts held in common-space")
    print("columns that are identically zero in at least one other architecture.\n")
    common = sorted(set.intersection(*[set(call_cols(data[a])) for a in ARCHS]))
    zero_in = {a: {c for c in common if data[a][c].sum() == 0} for a in ARCHS}
    for a in ARCHS:
        others = set().union(*[zero_in[o] for o in ARCHS if o != a])
        risky = [c for c in others if data[a][c].sum() > 0]
        tot = data[a][common].values.sum()
        vol = data[a][risky].values.sum() if risky else 0.0
        print(f"  {a:7s} {vol/tot*100:6.2f}%  of volume in {len(risky):3d} columns "
              f"empty elsewhere")
    print()
    return flagged


# ============================================================ PART B

def build_alias_map(data):
    """name -> merged name, for groups with more than one member."""
    m = {}
    for group in ALIAS_GROUPS:
        if len(group) < 2:
            continue
        for g in group:
            m['Call_' + g] = 'Call_' + group[0]
    return m


def merged_frame(df, alias_map, feature_order):
    """Collapse alias columns by summation, return a matrix in fixed order."""
    cols = call_cols(df)
    out = pd.DataFrame(index=df.index)
    for c in cols:
        tgt = alias_map.get(c, c)
        out[tgt] = out[tgt] + df[c].values if tgt in out.columns else df[c].values
    for f in feature_order:
        if f not in out.columns:
            out[f] = 0.0
    return out[feature_order].values.astype(float)


def rf():
    return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=RANDOM_STATE, n_jobs=-1)


def part_b(data):
    print("=" * 78)
    print("PART B - RE-RUN ON THE ALIAS-MERGED FEATURE SPACE")
    print("=" * 78)

    alias_map = build_alias_map(data)
    merged_sets = []
    for a in ARCHS:
        merged_sets.append({alias_map.get(c, c) for c in call_cols(data[a])})
    feats = sorted(set.intersection(*merged_sets))
    print(f"Common features after alias merging: {len(feats)} "
          f"(exact-name intersection gave 87)\n")

    X = {a: merged_frame(data[a], alias_map, feats) for a in ARCHS}
    y = {a: data[a]['label'].values for a in ARCHS}
    h = {a: data[a]['Hash'].values for a in ARCHS}

    rows = []
    for a in ARCHS:
        tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=RANDOM_STATE)
                      .split(X[a], y[a], groups=h[a]))
        clf = rf().fit(X[a][tr], y[a][tr])
        p = clf.predict_proba(X[a][te])[:, 1]
        pred = (p >= 0.5).astype(int)
        rows.append({'kind': 'same', 'train': a, 'test': a,
                     'macro_f1': f1_score(y[a][te], pred, average='macro'),
                     'roc_auc': roc_auc_score(y[a][te], p),
                     'recall_benign': recall_score(y[a][te], pred, pos_label=0)})
        print(f"same      {a:7s} macroF1={rows[-1]['macro_f1']:.4f} "
              f"AUC={rows[-1]['roc_auc']:.4f}")

    for src in ARCHS:
        clf = rf().fit(X[src], y[src])
        for tgt in ARCHS:
            if src == tgt:
                continue
            p = clf.predict_proba(X[tgt])[:, 1]
            pred = (p >= 0.5).astype(int)
            rows.append({'kind': 'transfer', 'train': src, 'test': tgt,
                         'macro_f1': f1_score(y[tgt], pred, average='macro'),
                         'roc_auc': roc_auc_score(y[tgt], p),
                         'recall_benign': recall_score(y[tgt], pred, pos_label=0)})
            tag = ' <- ARM' if 'ARM' in (src, tgt) else ''
            print(f"transfer  {src:7s}->{tgt:7s} macroF1={rows[-1]['macro_f1']:.4f} "
                  f"AUC={rows[-1]['roc_auc']:.4f} "
                  f"benignRecall={rows[-1]['recall_benign']:.4f}{tag}")

    res = pd.DataFrame(rows)
    t = res[res.kind == 'transfer']
    arm = t[(t.train == 'ARM') | (t.test == 'ARM')]
    non = t[(t.train != 'ARM') & (t.test != 'ARM')]
    into = t[t.test == 'ARM']
    out_of = t[t.train == 'ARM']

    print("\n" + "=" * 78)
    print("COMPARISON WITH THE EXACT-NAME INTERSECTION")
    print("=" * 78)
    print(f"{'group':20s} {'macroF1':>9s} {'AUC':>9s} {'benignRec':>10s}   (87-feature values)")
    ref = {'ARM-involving (6)': (0.7908, 0.8334, None),
           'Non-ARM (6)':       (0.9856, 0.9958, None),
           'Into ARM (3)':      (0.8300, 0.9885, 0.9882),
           'Out of ARM (3)':    (0.7560, 0.6783, 0.4214)}
    for name, grp in [('ARM-involving (6)', arm), ('Non-ARM (6)', non),
                      ('Into ARM (3)', into), ('Out of ARM (3)', out_of)]:
        r = ref[name]
        print(f"{name:20s} {grp.macro_f1.mean():9.4f} {grp.roc_auc.mean():9.4f} "
              f"{grp.recall_benign.mean():10.4f}   "
              f"({r[0]:.4f} / {r[1]:.4f}"
              + (f" / {r[2]:.4f})" if r[2] else ")"))

    print(f"\nARM gap (macro F1): {non.macro_f1.mean()-arm.macro_f1.mean():+.4f}  (was +0.1926)")
    print(f"ARM gap (ROC-AUC):  {non.roc_auc.mean()-arm.roc_auc.mean():+.4f}  (was +0.1624)")
    print(f"Out-of-ARM AUC:     {out_of.roc_auc.mean():.4f}          (was 0.6783)")

    print("\nINTERPRETATION")
    print("  Out-of-ARM AUC rises toward 0.99 -> the collapse was an alignment")
    print("     artefact. The manuscript's transfer results must be revised.")
    print("  Out-of-ARM AUC stays near 0.68   -> alignment is ruled out and the")
    print("     finding is strengthened; report this as a negative control.")

    res.to_csv(f'{BASE_DIR}/paper3_alias_merged.csv', index=False)
    print(f"\nSaved: paper3_alias_merged.csv")
    return res


if __name__ == '__main__':
    print("Loading...\n")
    data = load_all()
    flagged = part_a(data)
    part_b(data)

Loading...

PART A - ALIAS DIAGNOSTIC
Column totals per architecture. '-' = column absent from schema,
0 = column present but never populated.

[mmap]
               x86    ARM    MIPS  MIPSEL
variant                                  
mmap           429      -     250     226
mmap2       50,759  4,232  45,099  37,123
mmap_pgoff       -      -       -       -
  dominant variant: x86=mmap2, ARM=mmap2, MIPS=mmap2, MIPSEL=mmap2

[stat]   <<< SPLIT ACROSS ARCHITECTURES
            x86     ARM    MIPS  MIPSEL
variant                                
stat     38,375       -  40,646  33,051
stat64        -  25,817       -       -
newstat       -       -       -       -
  dominant variant: x86=stat, ARM=stat64, MIPS=stat, MIPSEL=stat

[fstat]   <<< SPLIT ACROSS ARCHITECTURES
               x86     ARM    MIPS  MIPSEL
variant                                   
fstat       13,200      74  13,205  12,758
fstat64        198  20,076       -       0
newfstat         -       -       -       -
newfstata

In [ ]:
"""
FINAL RERUN ON THE FIXED FEATURE SET
====================================
Plain purpose: every number in the paper's Results section was measured on
the old feature set, where ARM's syscall names did not line up with the other
three. This script measures everything again on the fixed set, in one run, so
all paper numbers come from the same place.

What it runs (all Random Forest unless stated, seed 42, same hash-grouped
splits and same full-source -> full-target transfer as the paper):

  1  exact-name space (the old 87 features)       -> should match the paper
  2  strict merged space (93 features)             -> the main fixed result
  3  strict + relative frequency
  4  strict + Logistic Regression
  5  strict + XGBoost
  6  strict without the fcntl merge                (fcntl breaks the stated rule)
  7  old space with ONLY clock_gettime removed     (is it one column?)
  8  controls on the strict space:
       a. class-mix control   (ARM cut to ~77.5% malware rows)
       b. diversity control   (other three cut to 1,992 binaries)
       c. ARM cut to ~620 binaries, original class mix
  9  leave-one-merged-column-out on the strict space

Expected time: roughly 1 to 1.5 hours on free Colab. A timer prints after
each step. Results are saved after every step, so nothing is lost if the
session stops.
"""

import sys, subprocess, time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, recall_score

SEED = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)

# Pairs of names for the same operation (differ only in width / ABI naming).
# fstatat64, newfstatat, pselect6, ppoll, socketcall, ipc, select are NOT merged:
# they are different operations or multiplexers.
GROUPS = {
    'stat':          ['stat', 'stat64'],
    'fstat':         ['fstat', 'fstat64'],
    'lstat':         ['lstat', 'lstat64'],
    'fcntl':         ['fcntl', 'fcntl64'],
    'clock_gettime': ['clock_gettime', 'clock_gettime64'],
    'getdents':      ['getdents', 'getdents64'],
    'getuid':        ['getuid', 'getuid32'],
    'geteuid':       ['geteuid', 'geteuid32'],
    'getgid':        ['getgid', 'getgid32'],
    'setuid':        ['setuid', 'setuid32'],
    'setgid':        ['setgid', 'setgid32'],
    'chown':         ['chown', 'chown32'],
}

T0 = time.time()
def stamp(msg):
    print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)


# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
def load():
    data = {}
    for a, p in ARCH_FILES.items():
        df = pd.read_parquet(p)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        data[a] = df
    return data

data = load()
schema = {a: set(c for c in df.columns if c.startswith('Call_')) for a, df in data.items()}
EXACT = sorted(set.intersection(*schema.values()))
stamp(f"loaded; exact-name common features = {len(EXACT)}  (paper says 87)")


# ---------------------------------------------------------------------------
# WHAT DOES '-' MEAN?  print, for every name, whether the column exists
# ---------------------------------------------------------------------------
print("\nSCHEMA PRESENCE  (absent = column not in the file; number = column total)")
for g, members in GROUPS.items():
    print(f"  [{g}]")
    for m in members:
        c = 'Call_' + m
        cells = []
        for a in ARCHS:
            if c in data[a].columns:
                cells.append(f"{a}={int(data[a][c].sum()):,}")
            else:
                cells.append(f"{a}=absent")
        print(f"      {m:18s} " + "  ".join(cells))


# ---------------------------------------------------------------------------
# FEATURE SPACES
# A "spec" is a list of (feature name, [columns whose counts are summed]).
# ---------------------------------------------------------------------------
def exact_spec():
    return [(c, [c]) for c in EXACT]

def merged_spec(groups):
    qualified = {}
    for g, members in groups.items():
        cols = ['Call_' + m for m in members]
        # a group is used only if every architecture has at least one member
        if all(any(c in schema[a] for c in cols) for a in ARCHS):
            qualified[g] = cols
    used = set(c for cols in qualified.values() for c in cols)
    spec = [(c, [c]) for c in EXACT if c not in used]
    spec += [('M_' + g, cols) for g, cols in qualified.items()]
    return spec, sorted(qualified)

STRICT, strict_groups = merged_spec(GROUPS)
STRICT_NO_FCNTL, _ = merged_spec({k: v for k, v in GROUPS.items() if k != 'fcntl'})
EXACT_SPEC = exact_spec()
EXACT_NO_CLOCK = [(n, c) for n, c in EXACT_SPEC if n != 'Call_clock_gettime']

stamp(f"strict space = {len(STRICT)} features (earlier run: 93); merged groups: {strict_groups}")
stamp(f"strict without fcntl merge = {len(STRICT_NO_FCNTL)}; exact minus clock_gettime = {len(EXACT_NO_CLOCK)}")


def matrix(df, spec, relfreq=False):
    cols_out = []
    for _, cols in spec:
        present = [c for c in cols if c in df.columns]
        if present:
            cols_out.append(df[present].fillna(0).sum(axis=1).values)
        else:
            cols_out.append(np.zeros(len(df)))
    M = np.column_stack(cols_out).astype(np.float32)
    if relfreq:
        t = M.sum(axis=1, keepdims=True)
        t[t == 0] = 1
        M = M / t
    return M

def prep(dfs, spec, relfreq=False):
    X = {a: matrix(d, spec, relfreq) for a, d in dfs.items()}
    y = {a: d['label'].values for a, d in dfs.items()}
    h = {a: d['Hash'].values for a, d in dfs.items()}
    return X, y, h


# ---------------------------------------------------------------------------
# MODELS AND METRICS
# ---------------------------------------------------------------------------
def make_model(name, ytr):
    if name == 'RF':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                      random_state=SEED, n_jobs=-1)
    if name == 'LR':
        # scaler is fitted on the SOURCE training data only
        return make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight='balanced', max_iter=2000,
                                                random_state=SEED, n_jobs=-1))
    if name == 'XGB':
        spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
        return xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 scale_pos_weight=spw, random_state=SEED, n_jobs=-1,
                                 tree_method='hist', eval_metric='logloss', verbosity=0)
    raise ValueError(name)

def metrics(clf, X, y):
    p = clf.predict_proba(X)[:, 1]
    pred = (p >= 0.5).astype(int)
    return dict(f1=f1_score(y, pred, average='macro'),
                auc=roc_auc_score(y, p),
                benrec=recall_score(y, pred, pos_label=0))

def run_config(X, y, h, model, do_same=True, cols=None):
    """cols: optional column index list, to drop features without rebuilding."""
    sel = (lambda M: M) if cols is None else (lambda M: M[:, cols])
    rows = []
    if do_same:
        for a in ARCHS:
            gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
            tr, te = next(gss.split(X[a], y[a], groups=h[a]))
            clf = make_model(model, y[a][tr])
            clf.fit(sel(X[a])[tr], y[a][tr])
            rows.append(dict(src=a, tgt=a, kind='same', **metrics(clf, sel(X[a])[te], y[a][te])))
    for s in ARCHS:
        clf = make_model(model, y[s])
        clf.fit(sel(X[s]), y[s])
        for t in ARCHS:
            if t != s:
                rows.append(dict(src=s, tgt=t, kind='transfer', **metrics(clf, sel(X[t]), y[t])))
    return pd.DataFrame(rows)

def summarise(res, label):
    t = res[res.kind == 'transfer']
    arm = t[(t.src == 'ARM') | (t.tgt == 'ARM')]
    non = t[(t.src != 'ARM') & (t.tgt != 'ARM')]
    into = t[t.tgt == 'ARM']
    out = t[t.src == 'ARM']
    same = res[res.kind == 'same'].f1.mean() if (res.kind == 'same').any() else np.nan
    return dict(config=label, same=same,
                arm_f1=arm.f1.mean(), non_f1=non.f1.mean(),
                gap_f1=non.f1.mean() - arm.f1.mean(),
                gap_auc=non.auc.mean() - arm.auc.mean(),
                into_f1=into.f1.mean(), out_f1=out.f1.mean(),
                out_auc=out.auc.mean(), out_benrec=out.benrec.mean(),
                non_benrec=non.benrec.mean())

ALL_ROWS, SUMMARY = [], []
def record(res, label):
    r = res.copy(); r['config'] = label
    ALL_ROWS.append(r)
    SUMMARY.append(summarise(res, label))
    pd.concat(ALL_ROWS).to_csv(f'{BASE_DIR}/paper3_final_all_pairs.csv', index=False)
    pd.DataFrame(SUMMARY).to_csv(f'{BASE_DIR}/paper3_final_summary.csv', index=False)
    s = SUMMARY[-1]
    stamp(f"{label:34s} gap_f1={s['gap_f1']:+.4f} gap_auc={s['gap_auc']:+.4f} "
          f"into={s['into_f1']:.4f} out={s['out_f1']:.4f} out_benrec={s['out_benrec']:.4f}")


# ---------------------------------------------------------------------------
# 1-7  MAIN CONFIGURATIONS
# ---------------------------------------------------------------------------
print("\n" + "=" * 78 + "\nMAIN CONFIGURATIONS\n" + "=" * 78)

Xe, ye, he = prep(data, EXACT_SPEC)
res = run_config(Xe, ye, he, 'RF')
record(res, '1 exact-name 87, RF')
g = summarise(res, '')['gap_f1']
if abs(g - 0.1926) > 0.005:
    print(f"  WARNING: exact-space gap {g:.4f} differs from the earlier 0.1926.")

# 7 uses the exact matrices with one column removed
names_e = [n for n, _ in EXACT_SPEC]
keep_noclock = [i for i, n in enumerate(names_e) if n != 'Call_clock_gettime']
res = run_config(Xe, ye, he, 'RF', cols=keep_noclock)
record(res, '7 exact minus clock_gettime only, RF')
del Xe

Xs, ys, hs = prep(data, STRICT)
res = run_config(Xs, ys, hs, 'RF')
record(res, '2 strict 93, RF')
strict_pairs = res.copy()
g = summarise(res, '')['gap_f1']
if abs(g - 0.0734) > 0.005:
    print(f"  WARNING: strict gap {g:.4f} differs from the earlier 0.0734.")

res = run_config(Xs, ys, hs, 'LR');  record(res, '4 strict, Logistic Regression')
res = run_config(Xs, ys, hs, 'XGB'); record(res, '5 strict, XGBoost')

Xr, yr, hr = prep(data, STRICT, relfreq=True)
res = run_config(Xr, yr, hr, 'RF'); record(res, '3 strict, relative frequency, RF')
del Xr

Xn, yn, hn = prep(data, STRICT_NO_FCNTL)
res = run_config(Xn, yn, hn, 'RF'); record(res, '6 strict without fcntl merge, RF')
del Xn


# ---------------------------------------------------------------------------
# 8  CONTROLS (on the strict space, RF)
# ---------------------------------------------------------------------------
print("\n" + "=" * 78 + "\nCONTROLS ON THE STRICT SPACE\n" + "=" * 78)

def share_report(label, d):
    g = d.groupby('Hash')['label'].max()
    print(f"    {label:10s} rows={len(d):7,d} hashes={len(g):5,d} "
          f"malware rows={d.label.mean()*100:5.1f}%  malware binaries={g.mean()*100:5.1f}%")

def arm_class_control(df, target=0.775):
    g = df.groupby('Hash').agg(label=('label', 'max'), n=('label', 'size'))
    ben_rows = g.loc[g.label == 0, 'n'].sum()
    mal = g[g.label == 1].sample(frac=1, random_state=SEED)
    allowed = target / (1 - target) * ben_rows
    keep = mal[mal.n.cumsum() <= allowed].index
    keep_h = set(g[g.label == 0].index) | set(keep)
    return df[df.Hash.isin(keep_h)]

def shrink_binaries(df, n):
    """Keep n binaries, sampling malware and benign binaries in the original binary-level proportion."""
    g = df.groupby('Hash')['label'].max()
    rng = np.random.RandomState(SEED)
    keep = []
    for lab in (0, 1):
        hs = g[g == lab].index.values
        k = int(round(n * len(hs) / len(g)))
        keep.extend(rng.choice(hs, size=min(k, len(hs)), replace=False))
    return df[df.Hash.isin(set(keep))]

# 8a  class-mix control: ARM cut to 77.5% malware rows by removing whole malware binaries
d = dict(data); d['ARM'] = arm_class_control(data['ARM'])
print("  8a class-mix control (earlier run gave 35,544 rows / 620 hashes):")
share_report('ARM', d['ARM'])
X, y, h = prep(d, STRICT)
record(run_config(X, y, h, 'RF', do_same=False), '8a strict + ARM class-mix control')

# 8b  diversity control: the other three cut to 1,992 binaries
d = dict(data)
print("  8b diversity control:")
for a in ['x86', 'MIPS', 'MIPSEL']:
    d[a] = shrink_binaries(data[a], 1992)
    share_report(a, d[a])
X, y, h = prep(d, STRICT)
record(run_config(X, y, h, 'RF', do_same=False), '8b strict + others cut to 1,992 binaries')

# 8c  ARM cut to ~620 binaries, original class mix (tests that the 620 side effect is harmless)
d = dict(data); d['ARM'] = shrink_binaries(data['ARM'], 620)
print("  8c ARM cut to 620 binaries, original binary-level mix:")
share_report('ARM', d['ARM'])
X, y, h = prep(d, STRICT)
record(run_config(X, y, h, 'RF', do_same=False), '8c strict + ARM cut to 620 binaries')


# ---------------------------------------------------------------------------
# 9  LEAVE ONE MERGED COLUMN OUT (strict space)
# ---------------------------------------------------------------------------
print("\n" + "=" * 78 + "\nLEAVE ONE MERGED COLUMN OUT (strict space)\n" + "=" * 78)
names_s = [n for n, _ in STRICT]
for g_name in ['clock_gettime', 'getdents', 'fstat', 'stat', 'fcntl']:
    drop = 'M_' + g_name
    if drop not in names_s:
        continue
    keep = [i for i, n in enumerate(names_s) if n != drop]
    res = run_config(Xs, ys, hs, 'RF', do_same=False, cols=keep)
    record(res, f'9 strict minus {g_name}')


# ---------------------------------------------------------------------------
# FINAL TABLES
# ---------------------------------------------------------------------------
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
print("\n" + "=" * 78 + "\nSUMMARY TABLE (paste this)\n" + "=" * 78)
print(pd.DataFrame(SUMMARY).round(4).to_string(index=False))

print("\n" + "=" * 78 + "\nSTRICT-SPACE TRANSFER MATRIX, RF (macro F1)\n" + "=" * 78)
t = strict_pairs[strict_pairs.kind == 'transfer']
print(t.pivot(index='src', columns='tgt', values='f1').reindex(index=ARCHS, columns=ARCHS).round(4).to_string())
print("\nSTRICT-SPACE SAME-ARCHITECTURE, RF (macro F1)")
print(strict_pairs[strict_pairs.kind == 'same'][['src', 'f1']].round(4).to_string(index=False))

print("\nSaved: paper3_final_all_pairs.csv, paper3_final_summary.csv")

[   0.1 min] loaded; exact-name common features = 87  (paper says 87)

SCHEMA PRESENCE  (absent = column not in the file; number = column total)
  [stat]
      stat               x86=38,375  ARM=absent  MIPS=40,646  MIPSEL=33,051
      stat64             x86=absent  ARM=25,817  MIPS=absent  MIPSEL=absent
  [fstat]
      fstat              x86=13,200  ARM=74  MIPS=13,205  MIPSEL=12,758
      fstat64            x86=198  ARM=20,076  MIPS=absent  MIPSEL=0
  [lstat]
      lstat              x86=815  ARM=absent  MIPS=absent  MIPSEL=absent
      lstat64            x86=absent  ARM=179  MIPS=absent  MIPSEL=absent
  [fcntl]
      fcntl              x86=22,621  ARM=18,619  MIPS=24,673  MIPSEL=24,704
      fcntl64            x86=21,842  ARM=20,842  MIPS=23,622  MIPSEL=22,800
  [clock_gettime]
      clock_gettime      x86=2  ARM=121,634  MIPS=0  MIPSEL=2
      clock_gettime64    x86=363,181  ARM=absent  MIPS=338,248  MIPSEL=265,567
  [getdents]
      getdents           x86=721  ARM=104,418  MIPS=ab

In [ ]:
"""
=============================================================================
CORRECTED ABI-AWARE ALIAS HARMONISATION
=============================================================================

The earlier alias map grouped fstat with fstatat. That was wrong: fstat
operates on a file descriptor, fstatat performs a pathname lookup relative
to a directory file descriptor. They are related but not interchangeable.

INCLUSION RULE used here. Two names are merged only if they denote the SAME
kernel operation under different 32-bit ABI conventions, differing only in:
  - offset or inode width            (stat/stat64, fcntl/fcntl64)
  - uid/gid width                    (getuid/getuid32)
  - time_t width                     (clock_gettime/clock_gettime64)
  - directory-entry struct width     (getdents/getdents64)
and the observed data show each architecture populating ONE member.

EXCLUDED, and why:
  fstatat / newfstatat  - pathname-relative lookup, not an fstat variant
  pselect6 / select     - different signatures and semantics
  ppoll / poll          - different signatures
  socketcall, ipc       - multiplexers; cannot be merged by summation
  mmap / mmap2          - kept separate in the strict map; mmap2 takes a
                          page-offset argument. Included only in the
                          permissive map below, for sensitivity.

Two maps are evaluated so the choice is visible rather than assumed:
  STRICT      only unambiguous width variants
  PERMISSIVE  strict plus mmap/mmap2 and lseek/_llseek

PART 1  Print both maps with observed per-architecture totals.
PART 2  Re-run baselines and all twelve transfers under BOTH maps, against
        freshly recomputed 87-feature reference values (no hard-coded
        numbers).
PART 3  Recompute SHAP on the harmonised space, since the ARM column of the
        manuscript's current SHAP table is computed on misaligned features.

Runtime: 50-80 minutes.
=============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, recall_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)

# --- STRICT: width variants of the same kernel operation only -------------
STRICT_GROUPS = [
    ['stat',          'stat64'],
    ['fstat',         'fstat64'],          # fstatat deliberately excluded
    ['lstat',         'lstat64'],
    ['fcntl',         'fcntl64'],
    ['clock_gettime', 'clock_gettime64'],
    ['clock_getres',  'clock_getres_time64'],
    ['getdents',      'getdents64'],
    ['sendfile',      'sendfile64'],
    ['truncate',      'truncate64'],
    ['ftruncate',     'ftruncate64'],
    ['pread',         'pread64'],
    ['pwrite',        'pwrite64'],
    ['statfs',        'statfs64'],
    ['fstatfs',       'fstatfs64'],
    ['getuid',        'getuid32'],
    ['geteuid',       'geteuid32'],
    ['getgid',        'getgid32'],
    ['getegid',       'getegid32'],
    ['setuid',        'setuid32'],
    ['setgid',        'setgid32'],
    ['getgroups',     'getgroups32'],
    ['setgroups',     'setgroups32'],
    ['getresuid',     'getresuid32'],
    ['getresgid',     'getresgid32'],
    ['setresuid',     'setresuid32'],
    ['setresgid',     'setresgid32'],
    ['setreuid',      'setreuid32'],
    ['setregid',      'setregid32'],
    ['chown',         'chown32'],
    ['fchown',        'fchown32'],
    ['lchown',        'lchown32'],
    ['fadvise64',     'fadvise64_64'],
    ['futex',         'futex_time64'],
    ['utimensat',     'utimensat_time64'],
]

# --- PERMISSIVE: strict plus two arguable pairs, for sensitivity ----------
PERMISSIVE_EXTRA = [
    ['mmap',  'mmap2'],       # mmap2 takes a page offset; same mapping op
    ['lseek', '_llseek'],     # _llseek is the 64-bit-offset form
]

# names NEVER merged, listed so the exclusion is explicit in the output
EXCLUDED = ['fstatat64', 'newfstatat', 'pselect6', 'pselect6_time64',
            'ppoll', 'ppoll_time64', 'socketcall', 'ipc',
            'select', '_newselect']


def rf():
    return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=SEED, n_jobs=-1)


def load_all():
    d = {}
    for a, p in ARCH_FILES.items():
        df = pd.read_parquet(p)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        d[a] = df
    return d


def call_cols(df):
    return [c for c in df.columns if c.startswith('Call_')]


def build_map(groups):
    m = {}
    for g in groups:
        for n in g:
            m['Call_' + n] = 'Call_' + g[0]
    return m


def merged_matrix(df, amap, order):
    out = {}
    for c in call_cols(df):
        t = amap.get(c, c)
        out[t] = out[t] + df[c].values if t in out else df[c].values.astype(float)
    return np.column_stack([out.get(f, np.zeros(len(df))) for f in order])


def common_merged(data, amap):
    sets = [{amap.get(c, c) for c in call_cols(data[a])} for a in ARCHS]
    return sorted(set.intersection(*sets))


# ============================================================ PART 1

def part1(data):
    print("=" * 78)
    print("PART 1 - ALIAS MAP, WITH OBSERVED TOTALS")
    print("=" * 78)
    print("A group qualifies only if each architecture populates ONE member.\n")
    for title, groups in [("STRICT", STRICT_GROUPS),
                          ("PERMISSIVE EXTRA", PERMISSIVE_EXTRA)]:
        print(f"--- {title} ---")
        for g in groups:
            cols = ['Call_' + n for n in g]
            if not any(c in data[a].columns for a in ARCHS for c in cols):
                continue
            tot = {}
            for n, c in zip(g, cols):
                tot[n] = {a: (float(data[a][c].sum()) if c in data[a].columns else None)
                          for a in ARCHS}
            dom = {}
            for a in ARCHS:
                vals = {n: tot[n][a] for n in g if tot[n][a] is not None}
                dom[a] = max(vals, key=vals.get) if vals and max(vals.values()) > 0 else None
            split = len({v for v in dom.values() if v}) > 1
            if not split:
                continue
            print(f"  [{g[0]}]  SPLIT")
            for n in g:
                s = "  ".join(f"{a}=" + ("-" if tot[n][a] is None else f"{tot[n][a]:,.0f}")
                              for a in ARCHS)
                print(f"      {n:22s} {s}")
        print()
    print("EXCLUDED from merging (different operations or multiplexers):")
    print("  " + ", ".join(EXCLUDED) + "\n")


# ============================================================ PART 2

def run_space(data, feats, amap, label):
    X = {a: (merged_matrix(data[a], amap, feats) if amap
             else data[a][feats].values.astype(float)) for a in ARCHS}
    y = {a: data[a].label.values for a in ARCHS}
    h = {a: data[a].Hash.values for a in ARCHS}
    rows = []
    for a in ARCHS:
        tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
                      .split(X[a], y[a], groups=h[a]))
        clf = rf().fit(X[a][tr], y[a][tr])
        p = clf.predict_proba(X[a][te])[:, 1]
        rows.append({'space': label, 'kind': 'same', 'train': a, 'test': a,
                     'macro_f1': f1_score(y[a][te], (p >= .5).astype(int), average='macro'),
                     'roc_auc': roc_auc_score(y[a][te], p),
                     'recall_benign': recall_score(y[a][te], (p >= .5).astype(int), pos_label=0)})
    for src in ARCHS:
        clf = rf().fit(X[src], y[src])
        for tgt in ARCHS:
            if src == tgt:
                continue
            p = clf.predict_proba(X[tgt])[:, 1]
            pr = (p >= .5).astype(int)
            rows.append({'space': label, 'kind': 'transfer', 'train': src, 'test': tgt,
                         'macro_f1': f1_score(y[tgt], pr, average='macro'),
                         'roc_auc': roc_auc_score(y[tgt], p),
                         'recall_benign': recall_score(y[tgt], pr, pos_label=0)})
    return pd.DataFrame(rows)


def summarise(r, label, nfeat):
    t = r[r.kind == 'transfer']
    arm = t[(t.train == 'ARM') | (t.test == 'ARM')]
    non = t[(t.train != 'ARM') & (t.test != 'ARM')]
    into = t[t.test == 'ARM']; out_of = t[t.train == 'ARM']
    return {
        'space': label, 'features': nfeat,
        'same_mean': r[r.kind == 'same'].macro_f1.mean(),
        'arm_f1': arm.macro_f1.mean(), 'non_f1': non.macro_f1.mean(),
        'gap_f1': non.macro_f1.mean() - arm.macro_f1.mean(),
        'arm_auc': arm.roc_auc.mean(), 'non_auc': non.roc_auc.mean(),
        'gap_auc': non.roc_auc.mean() - arm.roc_auc.mean(),
        'into_f1': into.macro_f1.mean(), 'out_f1': out_of.macro_f1.mean(),
        'out_auc': out_of.roc_auc.mean(),
        'out_benrec': out_of.recall_benign.mean(),
        'non_benrec': non.recall_benign.mean(),
    }


def part2(data):
    print("=" * 78)
    print("PART 2 - THREE FEATURE SPACES, FRESHLY COMPUTED")
    print("=" * 78)
    exact = sorted(set.intersection(*[set(call_cols(data[a])) for a in ARCHS]))
    sm = build_map(STRICT_GROUPS)
    pm = build_map(STRICT_GROUPS + PERMISSIVE_EXTRA)
    fs = common_merged(data, sm)
    fp = common_merged(data, pm)
    print(f"  exact-name : {len(exact)} features")
    print(f"  strict     : {len(fs)} features")
    print(f"  permissive : {len(fp)} features\n")

    out = []
    for label, feats, amap in [('exact', exact, None),
                               ('strict', fs, sm),
                               ('permissive', fp, pm)]:
        print(f"running {label}...")
        r = run_space(data, feats, amap, label)
        out.append(r)
        s = summarise(r, label, len(feats))
        print(f"  gap_f1={s['gap_f1']:+.4f}  gap_auc={s['gap_auc']:+.4f}  "
              f"out_auc={s['out_auc']:.4f}  out_benrec={s['out_benrec']:.4f}")
    allr = pd.concat(out, ignore_index=True)

    print("\n" + "=" * 78)
    print("SUMMARY")
    print("=" * 78)
    summ = pd.DataFrame([summarise(allr[allr.space == l], l, n) for l, n in
                         [('exact', len(exact)), ('strict', len(fs)), ('permissive', len(fp))]])
    print(summ.round(4).to_string(index=False))
    print("\n  If strict and permissive agree, the result does not depend on the")
    print("  two arguable pairs and the strict map can be used in the paper.")
    print("  If they differ materially, report the strict map and note the")
    print("  sensitivity.\n")
    allr.to_csv(f'{BASE_DIR}/paper3_alias_corrected_full.csv', index=False)
    summ.to_csv(f'{BASE_DIR}/paper3_alias_corrected_summary.csv', index=False)
    return fs, sm


# ============================================================ PART 3

def part3(data, feats, amap):
    print("=" * 78)
    print("PART 3 - SHAP ON THE HARMONISED SPACE")
    print("=" * 78)
    try:
        import shap
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
        import shap

    names = [f.replace('Call_', '') for f in feats]
    top = {}
    for a in ARCHS:
        df = data[a]
        X = merged_matrix(df, amap, feats)
        y = df.label.values
        tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
                      .split(X, y, groups=df.Hash.values))
        clf = rf().fit(X[tr], y[tr])
        samp = te[:1000]
        sv = shap.TreeExplainer(clf).shap_values(X[samp])
        v = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)
        imp = np.abs(v).mean(axis=0)
        order = np.argsort(-imp)[:15]
        top[a] = [(names[i], imp[i]) for i in order]
        print(f"\n{a} top 10:")
        for n, s in top[a][:10]:
            print(f"   {n:24s} {s:.4f}")

    sets = {a: {n for n, _ in top[a]} for a in ARCHS}
    shared = set.intersection(*sets.values())
    print(f"\nshared across all four top-15 ({len(shared)}): {sorted(shared)}")
    arm_only = sets['ARM'] - set().union(*[sets[a] for a in ARCHS if a != 'ARM'])
    print(f"ARM-only in top-15 ({len(arm_only)}): {sorted(arm_only)}")
    print("\n  Compare with the manuscript's current table. If clock_gettime no")
    print("  longer dominates ARM, its 0.2527 value was a naming artefact.\n")

    pd.DataFrame([{'arch': a, 'rank': i + 1, 'feature': n, 'mean_abs_shap': s}
                  for a in ARCHS for i, (n, s) in enumerate(top[a])]
                 ).to_csv(f'{BASE_DIR}/paper3_shap_harmonised.csv', index=False)


if __name__ == '__main__':
    print("Loading...\n")
    data = load_all()
    part1(data)
    feats, amap = part2(data)
    part3(data, feats, amap)
    print("Saved: paper3_alias_corrected_full.csv,")
    print("       paper3_alias_corrected_summary.csv,")
    print("       paper3_shap_harmonised.csv")

Loading...

PART 1 - ALIAS MAP, WITH OBSERVED TOTALS
A group qualifies only if each architecture populates ONE member.

--- STRICT ---
  [stat]  SPLIT
      stat                   x86=38,375  ARM=-  MIPS=40,646  MIPSEL=33,051
      stat64                 x86=-  ARM=25,817  MIPS=-  MIPSEL=-
  [fstat]  SPLIT
      fstat                  x86=13,200  ARM=74  MIPS=13,205  MIPSEL=12,758
      fstat64                x86=198  ARM=20,076  MIPS=-  MIPSEL=0
  [lstat]  SPLIT
      lstat                  x86=815  ARM=-  MIPS=-  MIPSEL=-
      lstat64                x86=-  ARM=179  MIPS=-  MIPSEL=-
  [fcntl]  SPLIT
      fcntl                  x86=22,621  ARM=18,619  MIPS=24,673  MIPSEL=24,704
      fcntl64                x86=21,842  ARM=20,842  MIPS=23,622  MIPSEL=22,800
  [clock_gettime]  SPLIT
      clock_gettime          x86=2  ARM=121,634  MIPS=0  MIPSEL=2
      clock_gettime64        x86=363,181  ARM=-  MIPS=338,248  MIPSEL=265,567
  [getdents]  SPLIT
      getdents               x86=721  ARM=

In [ ]:
"""
=============================================================================
FINAL SCRIPT
=============================================================================

Everything here was already measured once, on the OLD feature set that had
the naming problem. This re-measures it on the CORRECTED feature set, so the
whole paper uses one set of numbers.

  PART 1   Class-balance control.   Does ARM's high malware share explain
                                    what is left of the gap?
  PART 2   Binary-count control.    Does ARM having fewer binaries explain it?
  PART 3   Other classifiers.       Does the result hold with Logistic
                                    Regression and XGBoost?
  PART 4   One summary table for the paper.

Nothing new is being invented. This only brings the old experiments onto the
corrected data.

Runtime: about 60 to 75 minutes.
=============================================================================
"""

import sys, subprocess
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, recall_score

SEED = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)

# the same strict alias map that produced the 93-feature space
STRICT_GROUPS = [
    ['stat','stat64'], ['fstat','fstat64'], ['lstat','lstat64'],
    ['fcntl','fcntl64'], ['clock_gettime','clock_gettime64'],
    ['clock_getres','clock_getres_time64'], ['getdents','getdents64'],
    ['sendfile','sendfile64'], ['truncate','truncate64'],
    ['ftruncate','ftruncate64'], ['pread','pread64'], ['pwrite','pwrite64'],
    ['statfs','statfs64'], ['fstatfs','fstatfs64'],
    ['getuid','getuid32'], ['geteuid','geteuid32'],
    ['getgid','getgid32'], ['getegid','getegid32'],
    ['setuid','setuid32'], ['setgid','setgid32'],
    ['getgroups','getgroups32'], ['setgroups','setgroups32'],
    ['getresuid','getresuid32'], ['getresgid','getresgid32'],
    ['setresuid','setresuid32'], ['setresgid','setresgid32'],
    ['setreuid','setreuid32'], ['setregid','setregid32'],
    ['chown','chown32'], ['fchown','fchown32'], ['lchown','lchown32'],
    ['fadvise64','fadvise64_64'], ['futex','futex_time64'],
    ['utimensat','utimensat_time64'],
]


def load_all():
    d = {}
    for a, p in ARCH_FILES.items():
        df = pd.read_parquet(p)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        d[a] = df
    return d


def alias_map():
    m = {}
    for g in STRICT_GROUPS:
        for n in g:
            m['Call_' + n] = 'Call_' + g[0]
    return m


def harmonise(df, amap, order):
    """Add together the columns that mean the same thing."""
    out = {}
    for c in [c for c in df.columns if c.startswith('Call_')]:
        t = amap.get(c, c)
        out[t] = out[t] + df[c].values if t in out else df[c].values.astype(float)
    return np.column_stack([out.get(f, np.zeros(len(df))) for f in order])


def feature_list(data, amap):
    sets = [{amap.get(c, c) for c in df.columns if c.startswith('Call_')}
            for df in data.values()]
    return sorted(set.intersection(*sets))


def model(name, y):
    if name == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                      random_state=SEED, n_jobs=-1)
    if name == 'LogisticRegression':
        return make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight='balanced',
                                                max_iter=2000, random_state=SEED, n_jobs=-1))
    if name == 'XGBoost':
        spw = (y == 0).sum() / max((y == 1).sum(), 1)
        return xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 scale_pos_weight=spw, random_state=SEED, n_jobs=-1,
                                 tree_method='hist', eval_metric='logloss', verbosity=0)
    raise ValueError(name)


def scores(y, p):
    pred = (p >= 0.5).astype(int)
    return (f1_score(y, pred, average='macro'),
            roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
            recall_score(y, pred, pos_label=0))


def run_matrix(X, y, h, mname, tag):
    """4 same-architecture results plus all 12 transfers."""
    rows = []
    for a in ARCHS:
        tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
                      .split(X[a], y[a], groups=h[a]))
        clf = model(mname, y[a][tr]).fit(X[a][tr], y[a][tr])
        f1, auc, br = scores(y[a][te], clf.predict_proba(X[a][te])[:, 1])
        rows.append(dict(setting=tag, model=mname, kind='same', train=a, test=a,
                         macro_f1=f1, roc_auc=auc, recall_benign=br))
    for s in ARCHS:
        clf = model(mname, y[s]).fit(X[s], y[s])
        for t in ARCHS:
            if s == t:
                continue
            f1, auc, br = scores(y[t], clf.predict_proba(X[t])[:, 1])
            rows.append(dict(setting=tag, model=mname, kind='transfer', train=s, test=t,
                             macro_f1=f1, roc_auc=auc, recall_benign=br))
    return pd.DataFrame(rows)


def gaps(r):
    t = r[r.kind == 'transfer']
    arm = t[(t.train == 'ARM') | (t.test == 'ARM')]
    non = t[(t.train != 'ARM') & (t.test != 'ARM')]
    into = t[t.test == 'ARM']; out = t[t.train == 'ARM']
    return dict(arm_f1=arm.macro_f1.mean(), non_f1=non.macro_f1.mean(),
                gap_f1=non.macro_f1.mean() - arm.macro_f1.mean(),
                arm_auc=arm.roc_auc.mean(), non_auc=non.roc_auc.mean(),
                gap_auc=non.roc_auc.mean() - arm.roc_auc.mean(),
                into_f1=into.macro_f1.mean(), out_f1=out.macro_f1.mean(),
                out_auc=out.roc_auc.mean(), out_benrec=out.recall_benign.mean())


# ----------------------------------------------------------- controls

def drop_malware_binaries(df, target_share, seed=SEED):
    """Keep every benign binary; drop malware binaries until the row-level
    malware share reaches target_share."""
    rng = np.random.RandomState(seed)
    lab = df.groupby('Hash')['label'].first()
    n = df.groupby('Hash').size()
    ben = lab[lab == 0].index.tolist()
    mal = lab[lab == 1].index.tolist()
    ben_rows = n[ben].sum()
    want = target_share * ben_rows / (1 - target_share)
    rng.shuffle(mal)
    keep, acc = [], 0
    for h in mal:
        if acc >= want:
            break
        keep.append(h); acc += n[h]
    return df[df.Hash.isin(set(ben) | set(keep))].copy()


def cap_binaries(df, k, seed=SEED):
    """Reduce to k binaries, keeping the same malware/benign ratio."""
    rng = np.random.RandomState(seed)
    lab = df.groupby('Hash')['label'].first()
    ben = lab[lab == 0].index.to_numpy(); mal = lab[lab == 1].index.to_numpy()
    frac = k / len(lab)
    nb = max(1, int(round(len(ben) * frac))); nm = max(1, k - nb)
    keep = set(rng.choice(ben, min(nb, len(ben)), replace=False)) | \
           set(rng.choice(mal, min(nm, len(mal)), replace=False))
    return df[df.Hash.isin(keep)].copy()


# --------------------------------------------------------------- main

if __name__ == '__main__':
    print("Loading...\n")
    data = load_all()
    amap = alias_map()
    feats = feature_list(data, amap)
    print(f"Corrected feature set: {len(feats)} features\n")

    def pack(d):
        return ({a: harmonise(d[a], amap, feats) for a in ARCHS},
                {a: d[a].label.values for a in ARCHS},
                {a: d[a].Hash.values for a in ARCHS})

    all_rows = []

    # baseline on the corrected data
    print("=" * 74)
    print("BASELINE (corrected data, Random Forest)")
    print("=" * 74)
    X, y, h = pack(data)
    base = run_matrix(X, y, h, 'RandomForest', 'baseline')
    all_rows.append(base)
    g = gaps(base)
    print(f"  ARM gap  macroF1 {g['gap_f1']:+.4f}   AUC {g['gap_auc']:+.4f}")
    print(f"  out of ARM: AUC {g['out_auc']:.4f}  benign recall {g['out_benrec']:.4f}\n")

    # PART 1 - class balance
    print("=" * 74)
    print("PART 1 - CLASS BALANCE CONTROL")
    print("=" * 74)
    target = data['x86'].label.mean()
    d1 = dict(data)
    d1['ARM'] = drop_malware_binaries(data['ARM'], target)
    print(f"  ARM before: {len(data['ARM']):,} rows, {data['ARM'].Hash.nunique():,} binaries, "
          f"{data['ARM'].label.mean()*100:.1f}% malware rows")
    print(f"  ARM after : {len(d1['ARM']):,} rows, {d1['ARM'].Hash.nunique():,} binaries, "
          f"{d1['ARM'].label.mean()*100:.1f}% malware rows")
    X1, y1, h1 = pack(d1)
    r1 = run_matrix(X1, y1, h1, 'RandomForest', 'class_control')
    all_rows.append(r1)
    g1 = gaps(r1)
    print(f"  ARM gap  macroF1 {g1['gap_f1']:+.4f} (was {g['gap_f1']:+.4f})"
          f"   AUC {g1['gap_auc']:+.4f} (was {g['gap_auc']:+.4f})")
    print(f"  out of ARM benign recall {g1['out_benrec']:.4f} (was {g['out_benrec']:.4f})\n")

    # PART 2 - binary count
    print("=" * 74)
    print("PART 2 - BINARY COUNT CONTROL")
    print("=" * 74)
    k = data['ARM'].Hash.nunique()
    d2 = {a: (data[a] if a == 'ARM' else cap_binaries(data[a], k)) for a in ARCHS}
    for a in ARCHS:
        print(f"  {a:7s} {d2[a].Hash.nunique():,} binaries, {len(d2[a]):,} rows")
    X2, y2, h2 = pack(d2)
    r2 = run_matrix(X2, y2, h2, 'RandomForest', 'binary_control')
    all_rows.append(r2)
    g2 = gaps(r2)
    print(f"  ARM gap  macroF1 {g2['gap_f1']:+.4f} (was {g['gap_f1']:+.4f})"
          f"   AUC {g2['gap_auc']:+.4f} (was {g['gap_auc']:+.4f})\n")

    # PART 3 - other classifiers
    print("=" * 74)
    print("PART 3 - OTHER CLASSIFIERS")
    print("=" * 74)
    for m in ['LogisticRegression', 'XGBoost']:
        print(f"  running {m} ...")
        r = run_matrix(X, y, h, m, 'baseline')
        all_rows.append(r)
        gm = gaps(r)
        print(f"    ARM gap  macroF1 {gm['gap_f1']:+.4f}   AUC {gm['gap_auc']:+.4f}"
              f"   out-of-ARM AUC {gm['out_auc']:.4f}")
    print()

    # PART 4 - summary
    res = pd.concat(all_rows, ignore_index=True)
    print("=" * 74)
    print("PART 4 - SUMMARY FOR THE PAPER")
    print("=" * 74)
    out = []
    for lbl, sel in [
        ('RF baseline',        (res.setting == 'baseline') & (res.model == 'RandomForest')),
        ('RF class control',   res.setting == 'class_control'),
        ('RF binary control',  res.setting == 'binary_control'),
        ('LogisticRegression', (res.setting == 'baseline') & (res.model == 'LogisticRegression')),
        ('XGBoost',            (res.setting == 'baseline') & (res.model == 'XGBoost')),
    ]:
        gg = gaps(res[sel]); gg['row'] = lbl
        out.append(gg)
    summ = pd.DataFrame(out)[['row', 'arm_f1', 'non_f1', 'gap_f1',
                              'arm_auc', 'non_auc', 'gap_auc',
                              'into_f1', 'out_f1', 'out_auc', 'out_benrec']]
    print(summ.round(4).to_string(index=False))

    res.to_csv(f'{BASE_DIR}/paper3_final_full.csv', index=False)
    summ.to_csv(f'{BASE_DIR}/paper3_final_summary.csv', index=False)
    print("\nSaved: paper3_final_full.csv, paper3_final_summary.csv")
    print("\nPaste the summary table above. That is everything needed to")
    print("rebuild the paper.")

Loading...

Corrected feature set: 93 features

BASELINE (corrected data, Random Forest)
  ARM gap  macroF1 +0.0734   AUC +0.0037
  out of ARM: AUC 0.9892  benign recall 0.7409

PART 1 - CLASS BALANCE CONTROL
  ARM before: 199,903 rows, 1,992 binaries, 96.0% malware rows
  ARM after : 35,544 rows, 620 binaries, 77.5% malware rows
  ARM gap  macroF1 +0.0202 (was +0.0734)   AUC +0.0008 (was +0.0037)
  out of ARM benign recall 0.8729 (was 0.7409)

PART 2 - BINARY COUNT CONTROL
  x86     1,992 binaries, 101,985 rows
  ARM     1,992 binaries, 199,903 rows
  MIPS    1,992 binaries, 96,936 rows
  MIPSEL  1,992 binaries, 106,255 rows
  ARM gap  macroF1 +0.0746 (was +0.0734)   AUC +0.0015 (was +0.0037)

PART 3 - OTHER CLASSIFIERS
  running LogisticRegression ...
    ARM gap  macroF1 +0.0547   AUC +0.0338   out-of-ARM AUC 0.9602
  running XGBoost ...
    ARM gap  macroF1 +0.0616   AUC +0.0055   out-of-ARM AUC 0.9884

PART 4 - SUMMARY FOR THE PAPER
               row  arm_f1  non_f1  gap_f1  arm_

In [ ]:
"""
FINAL RERUN ON THE FIXED FEATURE SET
====================================
Plain purpose: every number in the paper's Results section was measured on
the old feature set, where ARM's syscall names did not line up with the other
three. This script measures everything again on the fixed set, in one run, so
all paper numbers come from the same place.

What it runs (all Random Forest unless stated, seed 42, same hash-grouped
splits and same full-source -> full-target transfer as the paper):

  1  exact-name space (the old 87 features)       -> should match the paper
  2  strict merged space (93 features)             -> the main fixed result
  3  strict + relative frequency
  4  strict + Logistic Regression
  5  strict + XGBoost
  6  strict without the fcntl merge                (fcntl breaks the stated rule)
  7  old space with ONLY clock_gettime removed     (is it one column?)
  8  controls on the strict space:
       a. class-mix control   (ARM cut to ~77.5% malware rows)
       b. diversity control   (other three cut to 1,992 binaries)
       c. ARM cut to ~620 binaries, original class mix
  9  leave-one-merged-column-out on the strict space

Expected time: roughly 1 to 1.5 hours on free Colab. A timer prints after
each step. Results are saved after every step, so nothing is lost if the
session stops.
"""

import sys, subprocess, time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, recall_score

SEED = 42
BASE_DIR = '/content/drive/My Drive/Colab Notebooks2'
ARCH_FILES = {
    'x86':    f'{BASE_DIR}/paper3_x86_sample.parquet',
    'ARM':    f'{BASE_DIR}/paper3_arm_sample.parquet',
    'MIPS':   f'{BASE_DIR}/paper3_mips_sample.parquet',
    'MIPSEL': f'{BASE_DIR}/paper3_mipsel_sample.parquet',
}
ARCHS = list(ARCH_FILES)

# Pairs of names for the same operation (differ only in width / ABI naming).
# fstatat64, newfstatat, pselect6, ppoll, socketcall, ipc, select are NOT merged:
# they are different operations or multiplexers.
GROUPS = {
    'stat':          ['stat', 'stat64'],
    'fstat':         ['fstat', 'fstat64'],
    'lstat':         ['lstat', 'lstat64'],
    'fcntl':         ['fcntl', 'fcntl64'],
    'clock_gettime': ['clock_gettime', 'clock_gettime64'],
    'getdents':      ['getdents', 'getdents64'],
    'getuid':        ['getuid', 'getuid32'],
    'geteuid':       ['geteuid', 'geteuid32'],
    'getgid':        ['getgid', 'getgid32'],
    'setuid':        ['setuid', 'setuid32'],
    'setgid':        ['setgid', 'setgid32'],
    'chown':         ['chown', 'chown32'],
}

T0 = time.time()
def stamp(msg):
    print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)


# ---------------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------------
def load():
    data = {}
    for a, p in ARCH_FILES.items():
        df = pd.read_parquet(p)
        df['label'] = (df['MalwareFamily'] != 'Benign').astype(int)
        data[a] = df
    return data

data = load()
schema = {a: set(c for c in df.columns if c.startswith('Call_')) for a, df in data.items()}
EXACT = sorted(set.intersection(*schema.values()))
stamp(f"loaded; exact-name common features = {len(EXACT)}  (paper says 87)")


# ---------------------------------------------------------------------------
# WHAT DOES '-' MEAN?  print, for every name, whether the column exists
# ---------------------------------------------------------------------------
print("\nSCHEMA PRESENCE  (absent = column not in the file; number = column total)")
for g, members in GROUPS.items():
    print(f"  [{g}]")
    for m in members:
        c = 'Call_' + m
        cells = []
        for a in ARCHS:
            if c in data[a].columns:
                cells.append(f"{a}={int(data[a][c].sum()):,}")
            else:
                cells.append(f"{a}=absent")
        print(f"      {m:18s} " + "  ".join(cells))


# ---------------------------------------------------------------------------
# FEATURE SPACES
# A "spec" is a list of (feature name, [columns whose counts are summed]).
# ---------------------------------------------------------------------------
def exact_spec():
    return [(c, [c]) for c in EXACT]

def merged_spec(groups):
    qualified = {}
    for g, members in groups.items():
        cols = ['Call_' + m for m in members]
        # a group is used only if every architecture has at least one member
        if all(any(c in schema[a] for c in cols) for a in ARCHS):
            qualified[g] = cols
    used = set(c for cols in qualified.values() for c in cols)
    spec = [(c, [c]) for c in EXACT if c not in used]
    spec += [('M_' + g, cols) for g, cols in qualified.items()]
    return spec, sorted(qualified)

STRICT, strict_groups = merged_spec(GROUPS)
STRICT_NO_FCNTL, _ = merged_spec({k: v for k, v in GROUPS.items() if k != 'fcntl'})
EXACT_SPEC = exact_spec()
EXACT_NO_CLOCK = [(n, c) for n, c in EXACT_SPEC if n != 'Call_clock_gettime']

stamp(f"strict space = {len(STRICT)} features (earlier run: 93); merged groups: {strict_groups}")
stamp(f"strict without fcntl merge = {len(STRICT_NO_FCNTL)}; exact minus clock_gettime = {len(EXACT_NO_CLOCK)}")


def matrix(df, spec, relfreq=False):
    cols_out = []
    for _, cols in spec:
        present = [c for c in cols if c in df.columns]
        if present:
            cols_out.append(df[present].fillna(0).sum(axis=1).values)
        else:
            cols_out.append(np.zeros(len(df)))
    M = np.column_stack(cols_out).astype(np.float32)
    if relfreq:
        t = M.sum(axis=1, keepdims=True)
        t[t == 0] = 1
        M = M / t
    return M

def prep(dfs, spec, relfreq=False):
    X = {a: matrix(d, spec, relfreq) for a, d in dfs.items()}
    y = {a: d['label'].values for a, d in dfs.items()}
    h = {a: d['Hash'].values for a, d in dfs.items()}
    return X, y, h


# ---------------------------------------------------------------------------
# MODELS AND METRICS
# ---------------------------------------------------------------------------
def make_model(name, ytr):
    if name == 'RF':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                      random_state=SEED, n_jobs=-1)
    if name == 'LR':
        # scaler is fitted on the SOURCE training data only
        return make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight='balanced', max_iter=2000,
                                                random_state=SEED, n_jobs=-1))
    if name == 'XGB':
        spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1)
        return xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 scale_pos_weight=spw, random_state=SEED, n_jobs=-1,
                                 tree_method='hist', eval_metric='logloss', verbosity=0)
    raise ValueError(name)

def metrics(clf, X, y):
    p = clf.predict_proba(X)[:, 1]
    pred = (p >= 0.5).astype(int)
    return dict(f1=f1_score(y, pred, average='macro'),
                auc=roc_auc_score(y, p),
                benrec=recall_score(y, pred, pos_label=0))

def run_config(X, y, h, model, do_same=True, cols=None):
    """cols: optional column index list, to drop features without rebuilding."""
    sel = (lambda M: M) if cols is None else (lambda M: M[:, cols])
    rows = []
    if do_same:
        for a in ARCHS:
            gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
            tr, te = next(gss.split(X[a], y[a], groups=h[a]))
            clf = make_model(model, y[a][tr])
            clf.fit(sel(X[a])[tr], y[a][tr])
            rows.append(dict(src=a, tgt=a, kind='same', **metrics(clf, sel(X[a])[te], y[a][te])))
    for s in ARCHS:
        clf = make_model(model, y[s])
        clf.fit(sel(X[s]), y[s])
        for t in ARCHS:
            if t != s:
                rows.append(dict(src=s, tgt=t, kind='transfer', **metrics(clf, sel(X[t]), y[t])))
    return pd.DataFrame(rows)

def summarise(res, label):
    t = res[res.kind == 'transfer']
    arm = t[(t.src == 'ARM') | (t.tgt == 'ARM')]
    non = t[(t.src != 'ARM') & (t.tgt != 'ARM')]
    into = t[t.tgt == 'ARM']
    out = t[t.src == 'ARM']
    same = res[res.kind == 'same'].f1.mean() if (res.kind == 'same').any() else np.nan
    return dict(config=label, same=same,
                arm_f1=arm.f1.mean(), non_f1=non.f1.mean(),
                gap_f1=non.f1.mean() - arm.f1.mean(),
                gap_auc=non.auc.mean() - arm.auc.mean(),
                into_f1=into.f1.mean(), out_f1=out.f1.mean(),
                out_auc=out.auc.mean(), out_benrec=out.benrec.mean(),
                non_benrec=non.benrec.mean())

ALL_ROWS, SUMMARY = [], []
def record(res, label):
    r = res.copy(); r['config'] = label
    ALL_ROWS.append(r)
    SUMMARY.append(summarise(res, label))
    pd.concat(ALL_ROWS).to_csv(f'{BASE_DIR}/paper3_final_all_pairs.csv', index=False)
    pd.DataFrame(SUMMARY).to_csv(f'{BASE_DIR}/paper3_final_summary.csv', index=False)
    s = SUMMARY[-1]
    stamp(f"{label:34s} gap_f1={s['gap_f1']:+.4f} gap_auc={s['gap_auc']:+.4f} "
          f"into={s['into_f1']:.4f} out={s['out_f1']:.4f} out_benrec={s['out_benrec']:.4f}")


# ---------------------------------------------------------------------------
# 1-7  MAIN CONFIGURATIONS
# ---------------------------------------------------------------------------
print("\n" + "=" * 78 + "\nMAIN CONFIGURATIONS\n" + "=" * 78)

Xe, ye, he = prep(data, EXACT_SPEC)
res = run_config(Xe, ye, he, 'RF')
record(res, '1 exact-name 87, RF')
g = summarise(res, '')['gap_f1']
if abs(g - 0.1926) > 0.005:
    print(f"  WARNING: exact-space gap {g:.4f} differs from the earlier 0.1926.")

# 7 uses the exact matrices with one column removed
names_e = [n for n, _ in EXACT_SPEC]
keep_noclock = [i for i, n in enumerate(names_e) if n != 'Call_clock_gettime']
res = run_config(Xe, ye, he, 'RF', cols=keep_noclock)
record(res, '7 exact minus clock_gettime only, RF')
del Xe

Xs, ys, hs = prep(data, STRICT)
res = run_config(Xs, ys, hs, 'RF')
record(res, '2 strict 93, RF')
strict_pairs = res.copy()
g = summarise(res, '')['gap_f1']
if abs(g - 0.0734) > 0.005:
    print(f"  WARNING: strict gap {g:.4f} differs from the earlier 0.0734.")

res = run_config(Xs, ys, hs, 'LR');  record(res, '4 strict, Logistic Regression')
res = run_config(Xs, ys, hs, 'XGB'); record(res, '5 strict, XGBoost')

Xr, yr, hr = prep(data, STRICT, relfreq=True)
res = run_config(Xr, yr, hr, 'RF'); record(res, '3 strict, relative frequency, RF')
del Xr

Xn, yn, hn = prep(data, STRICT_NO_FCNTL)
res = run_config(Xn, yn, hn, 'RF'); record(res, '6 strict without fcntl merge, RF')
del Xn


# ---------------------------------------------------------------------------
# 8  CONTROLS (on the strict space, RF)
# ---------------------------------------------------------------------------
print("\n" + "=" * 78 + "\nCONTROLS ON THE STRICT SPACE\n" + "=" * 78)

def share_report(label, d):
    g = d.groupby('Hash')['label'].max()
    print(f"    {label:10s} rows={len(d):7,d} hashes={len(g):5,d} "
          f"malware rows={d.label.mean()*100:5.1f}%  malware binaries={g.mean()*100:5.1f}%")

def arm_class_control(df, target=0.775):
    g = df.groupby('Hash').agg(label=('label', 'max'), n=('label', 'size'))
    ben_rows = g.loc[g.label == 0, 'n'].sum()
    mal = g[g.label == 1].sample(frac=1, random_state=SEED)
    allowed = target / (1 - target) * ben_rows
    keep = mal[mal.n.cumsum() <= allowed].index
    keep_h = set(g[g.label == 0].index) | set(keep)
    return df[df.Hash.isin(keep_h)]

def shrink_binaries(df, n):
    """Keep n binaries, sampling malware and benign binaries in the original binary-level proportion."""
    g = df.groupby('Hash')['label'].max()
    rng = np.random.RandomState(SEED)
    keep = []
    for lab in (0, 1):
        hs = g[g == lab].index.values
        k = int(round(n * len(hs) / len(g)))
        keep.extend(rng.choice(hs, size=min(k, len(hs)), replace=False))
    return df[df.Hash.isin(set(keep))]

# 8a  class-mix control: ARM cut to 77.5% malware rows by removing whole malware binaries
d = dict(data); d['ARM'] = arm_class_control(data['ARM'])
print("  8a class-mix control (earlier run gave 35,544 rows / 620 hashes):")
share_report('ARM', d['ARM'])
X, y, h = prep(d, STRICT)
record(run_config(X, y, h, 'RF', do_same=False), '8a strict + ARM class-mix control')

# 8b  diversity control: the other three cut to 1,992 binaries
d = dict(data)
print("  8b diversity control:")
for a in ['x86', 'MIPS', 'MIPSEL']:
    d[a] = shrink_binaries(data[a], 1992)
    share_report(a, d[a])
X, y, h = prep(d, STRICT)
record(run_config(X, y, h, 'RF', do_same=False), '8b strict + others cut to 1,992 binaries')

# 8c  ARM cut to ~620 binaries, original class mix (tests that the 620 side effect is harmless)
d = dict(data); d['ARM'] = shrink_binaries(data['ARM'], 620)
print("  8c ARM cut to 620 binaries, original binary-level mix:")
share_report('ARM', d['ARM'])
X, y, h = prep(d, STRICT)
record(run_config(X, y, h, 'RF', do_same=False), '8c strict + ARM cut to 620 binaries')


# ---------------------------------------------------------------------------
# 9  LEAVE ONE MERGED COLUMN OUT (strict space)
# ---------------------------------------------------------------------------
print("\n" + "=" * 78 + "\nLEAVE ONE MERGED COLUMN OUT (strict space)\n" + "=" * 78)
names_s = [n for n, _ in STRICT]
for g_name in ['clock_gettime', 'getdents', 'fstat', 'stat', 'fcntl']:
    drop = 'M_' + g_name
    if drop not in names_s:
        continue
    keep = [i for i, n in enumerate(names_s) if n != drop]
    res = run_config(Xs, ys, hs, 'RF', do_same=False, cols=keep)
    record(res, f'9 strict minus {g_name}')


# ---------------------------------------------------------------------------
# FINAL TABLES
# ---------------------------------------------------------------------------
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
print("\n" + "=" * 78 + "\nSUMMARY TABLE (paste this)\n" + "=" * 78)
print(pd.DataFrame(SUMMARY).round(4).to_string(index=False))

print("\n" + "=" * 78 + "\nSTRICT-SPACE TRANSFER MATRIX, RF (macro F1)\n" + "=" * 78)
t = strict_pairs[strict_pairs.kind == 'transfer']
print(t.pivot(index='src', columns='tgt', values='f1').reindex(index=ARCHS, columns=ARCHS).round(4).to_string())
print("\nSTRICT-SPACE SAME-ARCHITECTURE, RF (macro F1)")
print(strict_pairs[strict_pairs.kind == 'same'][['src', 'f1']].round(4).to_string(index=False))

print("\nSaved: paper3_final_all_pairs.csv, paper3_final_summary.csv")

[   0.1 min] loaded; exact-name common features = 87  (paper says 87)

SCHEMA PRESENCE  (absent = column not in the file; number = column total)
  [stat]
      stat               x86=38,375  ARM=absent  MIPS=40,646  MIPSEL=33,051
      stat64             x86=absent  ARM=25,817  MIPS=absent  MIPSEL=absent
  [fstat]
      fstat              x86=13,200  ARM=74  MIPS=13,205  MIPSEL=12,758
      fstat64            x86=198  ARM=20,076  MIPS=absent  MIPSEL=0
  [lstat]
      lstat              x86=815  ARM=absent  MIPS=absent  MIPSEL=absent
      lstat64            x86=absent  ARM=179  MIPS=absent  MIPSEL=absent
  [fcntl]
      fcntl              x86=22,621  ARM=18,619  MIPS=24,673  MIPSEL=24,704
      fcntl64            x86=21,842  ARM=20,842  MIPS=23,622  MIPSEL=22,800
  [clock_gettime]
      clock_gettime      x86=2  ARM=121,634  MIPS=0  MIPSEL=2
      clock_gettime64    x86=363,181  ARM=absent  MIPS=338,248  MIPSEL=265,567
  [getdents]
      getdents           x86=721  ARM=104,418  MIPS=ab

In [ ]:
import pandas as pd
d = pd.read_csv('/content/drive/My Drive/Colab Notebooks2/paper3_final_full.csv')
t = d[(d.setting=='baseline') & (d.model=='RandomForest') & (d.kind=='transfer')]
for _, r in t.iterrows():
    print(f"{r.train} -> {r.test}: {r.macro_f1:.4f}")

x86 -> ARM: 0.9258
x86 -> MIPS: 0.9931
x86 -> MIPSEL: 0.9930
ARM -> x86: 0.9037
ARM -> MIPS: 0.9088
ARM -> MIPSEL: 0.9112
MIPS -> x86: 0.9907
MIPS -> ARM: 0.9342
MIPS -> MIPSEL: 0.9931
MIPSEL -> x86: 0.9928
MIPSEL -> ARM: 0.9328
MIPSEL -> MIPS: 0.9942


In [ ]:
import pandas as pd

BASE = '/content/drive/My Drive/Colab Notebooks2'
FILES = {'x86':'paper3_x86_sample.parquet','ARM':'paper3_arm_sample.parquet',
         'MIPS':'paper3_mips_sample.parquet','MIPSEL':'paper3_mipsel_sample.parquet'}

GROUPS = [['stat','stat64'], ['fstat','fstat64'], ['lstat','lstat64'],
          ['fcntl','fcntl64'], ['clock_gettime','clock_gettime64'],
          ['clock_getres','clock_getres_time64'], ['getdents','getdents64'],
          ['sendfile','sendfile64'], ['truncate','truncate64'],
          ['ftruncate','ftruncate64'], ['pread','pread64'], ['pwrite','pwrite64'],
          ['statfs','statfs64'], ['fstatfs','fstatfs64'],
          ['getuid','getuid32'], ['geteuid','geteuid32'],
          ['getgid','getgid32'], ['getegid','getegid32'],
          ['setuid','setuid32'], ['setgid','setgid32'],
          ['getgroups','getgroups32'], ['setgroups','setgroups32'],
          ['getresuid','getresuid32'], ['getresgid','getresgid32'],
          ['setresuid','setresuid32'], ['setresgid','setresgid32'],
          ['setreuid','setreuid32'], ['setregid','setregid32'],
          ['chown','chown32'], ['fchown','fchown32'], ['lchown','lchown32'],
          ['fadvise64','fadvise64_64'], ['futex','futex_time64'],
          ['utimensat','utimensat_time64']]

amap = {'Call_'+n: 'Call_'+g[0] for g in GROUPS for n in g}

sets = []
for f in FILES.values():
    cols = [c for c in pd.read_parquet(f'{BASE}/{f}').columns if c.startswith('Call_')]
    sets.append({amap.get(c, c) for c in cols})

feats = sorted(set.intersection(*sets))
print(len(feats))
print(', '.join(f.replace('Call_','') for f in feats))

93
_llseek, _newselect, accept, access, bind, brk, chdir, chmod, chown, clock_gettime, clone, close, connect, dup2, epoll_create1, epoll_ctl, epoll_pwait, execve, exit, exit_group, fcntl, flock, fork, fstat, fsync, futex, getcwd, getdents, geteuid, getgid, getpid, getppid, getrusage, getsockname, getsockopt, gettid, getuid, inotify_add_watch, inotify_init, ioctl, kill, listen, lseek, mkdir, mmap2, mprotect, munmap, nanosleep, open, pipe, pipe2, poll, prctl, ptrace, read, readlink, readv, recv, recvfrom, recvmsg, rename, rmdir, rt_sigaction, rt_sigprocmask, rt_sigreturn, sched_yield, send, sendfile, sendmsg, sendto, set_tid_address, setgid, setitimer, setpgid, setrlimit, setsid, setsockopt, setuid, sigreturn, socket, stat, statfs, symlink, sync, sysinfo, times, umask, uname, unlink, utimensat, wait4, write, writev


In [ ]:
import pandas as pd
d = pd.read_csv('/content/drive/My Drive/Colab Notebooks2/paper3_final_all_pairs.csv')
for cfg in ['1 exact-name 87, RF', '2 strict 93, RF']:
    print(f"--- {cfg} ---")
    t = d[(d.config==cfg) & (d.kind=='transfer')]
    for _, r in t.iterrows():
        print(f"{r.src} -> {r.tgt}: {r.f1:.4f}")

--- 1 exact-name 87, RF ---
x86 -> ARM: 0.8450
x86 -> MIPS: 0.9888
x86 -> MIPSEL: 0.9887
ARM -> x86: 0.7350
ARM -> MIPS: 0.7513
ARM -> MIPSEL: 0.7818
MIPS -> x86: 0.9792
MIPS -> ARM: 0.8357
MIPS -> MIPSEL: 0.9872
MIPSEL -> x86: 0.9807
MIPSEL -> ARM: 0.8092
MIPSEL -> MIPS: 0.9892
--- 2 strict 93, RF ---
x86 -> ARM: 0.9161
x86 -> MIPS: 0.9931
x86 -> MIPSEL: 0.9929
ARM -> x86: 0.9108
ARM -> MIPS: 0.9177
ARM -> MIPSEL: 0.9198
MIPS -> x86: 0.9907
MIPS -> ARM: 0.9388
MIPS -> MIPSEL: 0.9929
MIPSEL -> x86: 0.9928
MIPSEL -> ARM: 0.9244
MIPSEL -> MIPS: 0.9942


In [ ]:
import sys, sklearn, xgboost, pandas, numpy, pyarrow, shap, psutil, os
print("ENV_PY   ", sys.version.split()[0])
print("ENV_SKL  ", sklearn.__version__)
print("ENV_XGB  ", xgboost.__version__)
print("ENV_PD   ", pandas.__version__)
print("ENV_NP   ", numpy.__version__)
print("ENV_PA   ", pyarrow.__version__)
print("ENV_SHAP ", shap.__version__)
print("ENV_CPU  ", os.cpu_count(), "cores")
print("ENV_RAM  ", round(psutil.virtual_memory().total/1e9, 1), "GB")

ENV_PY    3.13.16
ENV_SKL   1.6.1
ENV_XGB   3.4.1
ENV_PD    2.2.3
ENV_NP    2.1.3
ENV_PA    23.0.1
ENV_SHAP  0.52.0
ENV_CPU   2 cores
ENV_RAM   13.6 GB


In [ ]:
import pandas as pd
B='/content/drive/My Drive/Colab Notebooks2/'
for a,f in [('x86','paper3_x86_sample.parquet'),('ARM','paper3_arm_sample.parquet'),
            ('MIPS','paper3_mips_sample.parquet'),('MIPSEL','paper3_mipsel_sample.parquet')]:
    print(a, sorted(pd.read_parquet(B+f, columns=['MalwareFamily'])['MalwareFamily'].unique()))

x86 ['Benign', 'DarkNexus', 'Gafgyt', 'Generic', 'Mirai']
ARM ['Benign', 'DarkNexus', 'Gafgyt', 'Generic', 'Mirai', 'Tsunami']
MIPS ['Benign', 'DarkNexus', 'Gafgyt', 'Generic', 'Mirai']
MIPSEL ['Agent', 'Benign', 'DarkNexus', 'Gafgyt', 'Generic', 'Mirai', 'Rudedevil']


In [ ]:
import pyarrow.parquet as pq
B = '/content/drive/My Drive/Colab Notebooks2/'
F = {'x86':'paper3_x86_sample.parquet','ARM':'paper3_arm_sample.parquet',
     'MIPS':'paper3_mips_sample.parquet','MIPSEL':'paper3_mipsel_sample.parquet'}
schema = {a:{c for c in pq.read_schema(B+f).names if c.startswith('Call_')} for a,f in F.items()}
EXACT = sorted(set.intersection(*schema.values()))
G = {'stat':['stat','stat64'],'fstat':['fstat','fstat64'],'lstat':['lstat','lstat64'],
     'fcntl':['fcntl','fcntl64'],'clock_gettime':['clock_gettime','clock_gettime64'],
     'getdents':['getdents','getdents64'],'getuid':['getuid','getuid32'],
     'geteuid':['geteuid','geteuid32'],'getgid':['getgid','getgid32'],
     'setuid':['setuid','setuid32'],'setgid':['setgid','setgid32'],'chown':['chown','chown32']}
qual = {g:['Call_'+m for m in ms] for g,ms in G.items()
        if all(any(('Call_'+m) in schema[a] for m in ms) for a in F)}
used = {c for cols in qual.values() for c in cols}
names = [c[5:] for c in EXACT if c not in used] + list(qual)
print(len(EXACT), len(names), sorted(qual))
print([n for n in names if n.startswith(('sendfile','statfs'))])
print(', '.join(sorted(names)))

87 93 ['chown', 'clock_gettime', 'fcntl', 'fstat', 'getdents', 'geteuid', 'getgid', 'getuid', 'setgid', 'setuid', 'stat']
['sendfile64', 'statfs64']
_llseek, _newselect, accept, access, bind, brk, chdir, chmod, chown, clock_gettime, clone, close, connect, dup2, epoll_create1, epoll_ctl, epoll_pwait, execve, exit, exit_group, fcntl, flock, fork, fstat, fsync, futex, getcwd, getdents, geteuid, getgid, getpid, getppid, getrusage, getsockname, getsockopt, gettid, getuid, inotify_add_watch, inotify_init, ioctl, kill, listen, lseek, mkdir, mmap2, mprotect, munmap, nanosleep, open, pipe, pipe2, poll, prctl, ptrace, read, readlink, readv, recv, recvfrom, recvmsg, rename, rmdir, rt_sigaction, rt_sigprocmask, rt_sigreturn, sched_yield, send, sendfile64, sendmsg, sendto, set_tid_address, setgid, setitimer, setpgid, setrlimit, setsid, setsockopt, setuid, sigreturn, socket, stat, statfs64, symlink, sync, sysinfo, times, umask, uname, unlink, utimensat, wait4, write, writev


In [2]:
import pyarrow.parquet as pq, pandas as pd
B = '/content/drive/My Drive/Colab Notebooks2/'
F = {'x86':'paper3_x86_sample.parquet','ARM':'paper3_arm_sample.parquet',
     'MIPS':'paper3_mips_sample.parquet','MIPSEL':'paper3_mipsel_sample.parquet'}
S = {a: set(pq.read_schema(B+f).names) for a, f in F.items()}
for n in ['select','_newselect','pselect6','pselect6_time64',
          'ppoll','ppoll_time64','poll','futex']:
    c = 'Call_'+n
    row = {}
    for a, f in F.items():
        if c not in S[a]:
            row[a] = 'absent'
        else:
            v = pd.read_parquet(B+f, columns=[c])[c]
            row[a] = f"sum={int(v.sum()):,}, nz={int((v>0).sum()):,}"
    print(f"{n:18s}", row)

select             {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
_newselect         {'x86': 'sum=7,604, nz=7,155', 'ARM': 'sum=9,052, nz=8,272', 'MIPS': 'sum=6,729, nz=6,296', 'MIPSEL': 'sum=16,107, nz=6,540'}
pselect6           {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
pselect6_time64    {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
ppoll              {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
ppoll_time64       {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
poll               {'x86': 'sum=2,126, nz=1,248', 'ARM': 'sum=284, nz=140', 'MIPS': 'sum=2,701, nz=1,514', 'MIPSEL': 'sum=2,521, nz=1,369'}
futex              {'x86': 'sum=44, nz=25', 'ARM': 'sum=26, nz=12', 'MIPS': 'sum=67, nz=43', 'MIPSEL': 'sum=25, nz=14'}


In [4]:
import pyarrow.parquet as pq, pandas as pd
B = '/content/drive/My Drive/Colab Notebooks2/'
F = {'x86':'paper3_x86_sample.parquet','ARM':'paper3_arm_sample.parquet',
     'MIPS':'paper3_mips_sample.parquet','MIPSEL':'paper3_mipsel_sample.parquet'}
S = {a: set(pq.read_schema(B+f).names) for a, f in F.items()}
for n in ['fstatat64','newfstatat','socketcall','ipc']:
    c = 'Call_'+n
    row = {}
    for a, f in F.items():
        if c not in S[a]:
            row[a] = 'absent'
        else:
            v = pd.read_parquet(B+f, columns=[c])[c]
            row[a] = f"sum={int(v.sum()):,}, nz={int((v>0).sum()):,}"
    print(f"{n:14s}", row)

fstatat64      {'x86': 'absent', 'ARM': 'sum=0, nz=0', 'MIPS': 'absent', 'MIPSEL': 'absent'}
newfstatat     {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
socketcall     {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
ipc            {'x86': 'absent', 'ARM': 'absent', 'MIPS': 'absent', 'MIPSEL': 'absent'}
